# TACL Pipeline: Organized and Commented Notebook

This version keeps the original analysis logic but reorganizes the notebook into a clearer research pipeline. Outputs were cleared for a cleaner repository version.

## Project overview and setup

This notebook analyzes trust and distrust in Reddit discourse about Generative AI / LLMs. It is organized as a reproducible pipeline: data preparation, Task 1 classification, Task 2 dimension extraction, full-corpus batch inference, downstream analyses, and TACL revision experiments.

## Data requirements and expected files

The notebook references cached CSV/NDJSON/JSONL files as well as generated intermediate outputs. See the file report in the assistant response for the exact unique-file count.

### How to use this section

Before running the notebook, place required CSV/NDJSON/JSONL files in the working directory or update the paths in the relevant cells. Some files are generated by earlier cells and then loaded later as cached outputs. For fully reproducible execution, run the notebook sequentially, but for expensive API/batch sections you can skip inference cells and load the cached output files instead.

## 0. Imports, API configuration, and global utilities

Imports common libraries and configures API clients. API keys were removed and replaced with environment-variable placeholders.

In [ ]:
# Purpose: Configure API clients and run LLM inference. Requires the relevant API key in environment variables.
import os
import openai
import json
from openai import OpenAI
#client= openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

client = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

In [ ]:
# Purpose: Configure API clients and run LLM inference. Requires the relevant API key in environment variables.
import os
# import sys
# !{sys.executable} -m pip install together

from together import Together

client = Together(api_key=os.getenv('TOGETHER_API_KEY'))

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Capital of France"
        }
    ],
    temperature=0,
    stream=False
)
response.choices[0].message.content

In [ ]:
# Purpose: Configure API clients and run LLM inference. Requires the relevant API key in environment variables.
import os
import anthropic

# Initialize client with your API key
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

message = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=2048,
    # Update the thinking block here:
    messages=[
        {"role": "user", "content": "Hello Claude"}
    ]
)

# Extract and print the text response
print(message.content[0].text)


## 1. Load and prepare the manually annotated ground-truth data

Loads the cleaned annotation file, fixes ratio/agreement columns, filters to higher-agreement ground truth, and creates train/validation/test splits.

In [ ]:
# Purpose: Load a saved dataset or cached model-output file used by later analysis cells.
majority_labels_df= pd.read_csv('TACL_Final_Annotation_With_Ratios.csv', encoding="utf-8-sig")
# 1. Identify all columns ending with '_ratio'
ratio_cols = [col for col in majority_labels_df.columns if col.endswith('_ratio')]

# 2. Loop through each column and remove the leading '\t'
for col in ratio_cols:
    majority_labels_df[col] = majority_labels_df[col].astype(str).str.replace('\t', '', regex=False)

# Display the first few rows to verify the cleaning
majority_labels_df.head()

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
len(majority_labels_df), majority_labels_df['text'].nunique()

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
final_GT_data = majority_labels_df[majority_labels_df['agreement_level'] > 2/5]
len(majority_labels_df), len(final_GT_data)

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
final_GT_data = majority_labels_df[majority_labels_df['agreement_level'] > 2/5]
len(final_GT_data), final_GT_data['majority_Label'].value_counts(), final_GT_data['agreement_level'].value_counts()

In [ ]:
# Purpose: Split the manually annotated ground-truth data into train/validation/test sets.
from sklearn.model_selection import train_test_split

# 1. First split: Extract 20% for the Test set first
# The remaining 80% goes into a temporary dataframe
temp_df, test_df = train_test_split(
    final_GT_data,
    test_size=0.2,
    random_state=42,
    stratify=final_GT_data['majority_Label']
)

# 2. Second split: Divide the remaining 80% into Training and Validation
# To get 10% of the original from the 80% remaining, use test_size=0.125 (which is 1/8)
train_df, val_df = train_test_split(
    temp_df,
    test_size=0.125,  # 0.1 / 0.8 = 0.125
    random_state=42,
    stratify=temp_df['majority_Label']
)

# Verify the results
print(f"Training set: {len(train_df)} samples")   # ~70%
print(f"Validation set: {len(val_df)} samples") # ~10%
print(f"Test set: {len(test_df)} samples")       # ~20%

In [ ]:
# Purpose: Save the current processed dataframe or analysis output for reuse/reproducibility.
# 1. Handle all dimension ratio columns (e.g., Competence_ratio)
ratio_cols = [col for col in majority_labels_df.columns if col.endswith('_ratio')]
for col in ratio_cols:
    majority_labels_df[col] = "\t" + majority_labels_df[col].astype(str)

# 2. Specifically handle the agreement_level column
if 'agreement_level' in majority_labels_df.columns:
    majority_labels_df['agreement_level'] = "\t" + majority_labels_df['agreement_level'].astype(str)

# 3. Save the file
majority_labels_df.to_csv('TACL_Final_Annotation_With_Ratios.csv', encoding="utf-8-sig", index=False)

In [ ]:
# Purpose: Save the current processed dataframe or analysis output for reuse/reproducibility.

final_GT_data.to_csv('Final_GT_Data_Trust.csv', index=False, encoding='utf-8')

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
len(train_df), len(test_df), len(val_df), len(final_GT_data)

## 2. Task 1 classification: Trust / Distrust / Both / Neither

Defines prompts, runs zero-shot and few-shot model predictions on the manually annotated test split, cleans predictions, and evaluates model performance.

In [ ]:
# Purpose: Define the Task 1 prompt for classifying trust, distrust, both, or neither.
task_1_prompt= f"""

Task Instruction:** You are an advanced AI model trained for text classification. The purpose of this annotation task is to classify how people discuss trust and distrust toward Generative AI dimensions of trust in text. We define trust as a belief in the reliability, competence, or integrity of GenAI technologies, leading to positive expectations about their performance and ethical behavior. On the other hand, we define distrust as skepticism or concern about the reliability, competence, or ethical implications of GenAI, leading to negative expectations or cautious behavior toward these technologies or companies.

You will identify the presence or absence of trust or distrust in each text, as well as the dimensions of trust or distrust, which include aspects such as reliability, transparency, integrity and deception

Your task is to classify a given text:

Classify the text into a high-level category based on showing trust or distrust toward GenAI.


Follow the label definitions strictly.


Classify the text into one of the following categories in terms of showing trust or distrust toward Generative AI:

Trust → The text expresses belief in the reliability, competence, or integrity of GenAI technologies, leading to positive expectations about their performance and ethical behavior

Distrust  → The text expresses skepticism or concern about the reliability, competence, or ethical implications of GenAI, leading to negative expectations or cautious behavior toward these technologies or companies.

Both → The text indicates both trust or distrust in some way. Such a text shows tendencies that suggest both trust and distrust at the same time.

Neither → The text is not expressing trust or distrust towards GenAI, or is irrelevant to the context of our study.


Output Format:

[Label]

Strictly follow this format and don’t add any other text. Don't add explanation or justification, just the labels.

Read the following text and provide the best labels:

"""


In [ ]:
# Purpose: Define the Task 1 prompt for classifying trust, distrust, both, or neither.
import time
from tqdm import tqdm
from openai import OpenAI

# 1. Define your list of models
models_to_run = ['gpt-5.2', 'gpt-5.1', 'gpt-5', 'gpt-5-mini', 'gpt-5-nano', 'gpt-4o', 'gpt-4o-mini', 'gpt-4.1', 'gpt-4.1-mini', 'gpt-4.1-nano' ]

def classify_barriers_new_prompt(text, model_name):
    retries = 2
    while retries > 0:
        try:
            # Try with temperature=0.0 first
            try:
                completion = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": task_1_prompt + text}],
                    temperature=0.0
                )
            except Exception:
                # Silently retry without temperature if the first attempt fails
                completion = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": task_1_prompt + text}]
                )

            return completion.choices[0].message.content

        except Exception as e:
            print(f"Error with {model_name}: {e}. Retrying...")
            retries -= 1
            time.sleep(5)

    return "API_ERROR"

# 2. Loop through each model and create the new columns
tqdm.pandas()

for model in models_to_run:
    print(f"Starting classification for model: {model}")
    column_name = f"LLM_{model}"

    # Apply to test_df['text']
    test_df[column_name] = test_df['text'].progress_apply(
        lambda x: classify_barriers_new_prompt(x, model)
    )

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
test_df.head()

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# Identify the LLM prediction columns
llm_cols = [col for col in test_df.columns if col.startswith('LLM_')]

# Remove '[' and ']' from all these columns
for col in llm_cols:
    test_df[col] = test_df[col].astype(str).str.replace(r'[\[\]]', '', regex=True)

# Display the head to verify the cleanup
test_df.head()

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
# Identify the prediction columns
llm_cols = [col for col in test_df.columns if col.startswith('LLM_')]

print("Value Counts for LLM Predictions:")
print("-" * 30)

for col in llm_cols:
    print(f"Model: {col.replace('LLM_', '')}")
    # Using value_counts with dropna=False to see if any model failed to return a label
    print(test_df[col].astype(str).str.strip().value_counts(dropna=False))
    print("-" * 30)

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Identify the columns starting with LLM_
llm_cols = [col for col in test_df.columns if col.startswith('LLM_')]
ground_truth = test_df['majority_Label']

metrics_summary = []

for col in llm_cols:
    # Ensure both predictions and ground truth are clean strings
    y_pred = test_df[col]
    y_true = ground_truth

    # Calculate performance metrics
    # average='weighted' accounts for label imbalance in your majority_Label
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    metrics_summary.append({
        'Model Name': col.replace('LLM_', ''),
        'Accuracy': accuracy,
        'Precision (Weighted)': precision,
        'Recall (Weighted)': recall,
        'F1 Score (Weighted)': f1
    })

# Create the summary DataFrame and sort by best performance
performance_df = pd.DataFrame(metrics_summary).sort_values(by='F1 Score (Weighted)', ascending=False)

# Round all numerical columns to two decimal places
performance_df = performance_df.round(2)

performance_df

# Few Shot Task 1

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
import pandas as pd

# Sample 5 examples from each label category
# Assuming 'text_column' is the name of your text data column
shots_df = train_df.groupby('majority_Label').sample(n=5, random_state=42)

# Format the examples into a readable string for the prompt
examples_str = ""
for _, row in shots_df.iterrows():
    examples_str += f"Text: {row['text']}\n"
    # Remove the double braces to just print the label text
    examples_str += f"Label: {row['majority_Label']}\n\n"

In [ ]:
# Purpose: Define the Task 1 prompt for classifying trust, distrust, both, or neither.
task_1_prompt_few_shot = f"""

Task Instruction:** You are an advanced AI model trained for text classification. The purpose of this annotation task is to classify how people discuss trust and distrust toward Generative AI dimensions of trust in text. We define trust as a belief in the reliability, competence, or integrity of GenAI technologies, leading to positive expectations about their performance and ethical behavior. On the other hand, we define distrust as skepticism or concern about the reliability, competence, or ethical implications of GenAI, leading to negative expectations or cautious behavior toward these technologies or companies.

You will identify the presence or absence of trust or distrust in each text, as well as the dimensions of trust or distrust, which include aspects such as reliability, transparency, integrity and deception

Your task is to classify a given text:

Classify the text into a high-level category based on showing trust or distrust toward GenAI.


Follow the label definitions strictly.


Classify the text into one of the following categories in terms of showing trust or distrust toward Generative AI:

Trust → The text expresses belief in the reliability, competence, or integrity of GenAI technologies, leading to positive expectations about their performance and ethical behavior

Distrust  → The text expresses skepticism or concern about the reliability, competence, or ethical implications of GenAI, leading to negative expectations or cautious behavior toward these technologies or companies.

Both → The text indicates both trust or distrust in some way. Such a text shows tendencies that suggest both trust and distrust at the same time.

Neither → The text is not expressing trust or distrust towards GenAI, or is irrelevant to the context of our study.


### Here are some examples for the labels:

{examples_str}


Output Format:

[Label]

Strictly follow this format and don’t add any other text. Don't add explanation or justification, just the labels.

Read the following text and provide the best labels:

"""

In [ ]:
# Purpose: Define the Task 1 prompt for classifying trust, distrust, both, or neither.
print(task_1_prompt_few_shot)

In [ ]:
# Purpose: Define the Task 1 prompt for classifying trust, distrust, both, or neither.
import time
from tqdm import tqdm
from openai import OpenAI

# 1. Define your list of models
# models_to_run = ['gpt-5.2', 'gpt-5.1', 'gpt-5', 'gpt-5-mini', 'gpt-5-nano', 'gpt-4o', 'gpt-4o-mini', 'gpt-4.1', 'gpt-4.1-mini', 'gpt-4.1-nano' ]
# models_to_run= ['gpt-5.1-2025-11-13']
models_to_run = ['gpt-5.2', 'gpt-5.1', 'gpt-5', 'gpt-5-mini', 'gpt-5-nano', 'gpt-4o', 'gpt-4o-mini', 'gpt-4.1', 'gpt-4.1-mini', 'gpt-4.1-nano' ]

def classify_barriers_new_prompt(text, model_name):
    retries = 2
    while retries > 0:
        try:
            # Try with temperature=0.0 first
            try:
                completion = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": task_1_prompt_few_shot + text}],
                    temperature=0.0
                )
            except Exception:
                # Silently retry without temperature if the first attempt fails
                completion = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": task_1_prompt_few_shot + text}]
                )

            return completion.choices[0].message.content

        except Exception as e:
            print(f"Error with {model_name}: {e}. Retrying...")
            retries -= 1
            time.sleep(5)

    return "API_ERROR"

# 2. Loop through each model and create the new columns
tqdm.pandas()

for model in models_to_run:
    print(f"Starting classification for model: {model}")
    column_name = f"LLM_{model}_Few_Shot"

    # Apply to test_df['text']
    test_df[column_name] = test_df['text'].progress_apply(
        lambda x: classify_barriers_new_prompt(x, model)
    )

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# Identify the LLM prediction columns
llm_cols = [col for col in test_df.columns if col.endswith('_Few_Shot')]

# Remove '[' and ']' from all these columns
for col in llm_cols:
    test_df[col] = test_df[col].astype(str).str.replace(r'[\[\]]', '', regex=True)

# Display the head to verify the cleanup
test_df.head()

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
# 1. Identify all columns ending with '_Few_Shot'
few_shot_cols = [col for col in test_df.columns if col.endswith('_Few_Shot')]

print(f"Summary of Few-Shot Predictions ({len(few_shot_cols)} models):")
print("-" * 40)

# 2. Iterate through columns and print value counts
for col in few_shot_cols:
    # Clean up the name for the printout (e.g., 'LLM_gpt-5.2_Few_Shot' -> 'gpt-5.2')
    display_name = col.replace('LLM_', '').replace('_Few_Shot', '')

    print(f"Model: {display_name}")
    # Using value_counts with dropna=False to check for API misses or nulls
    counts = test_df[col].astype(str).str.strip().value_counts(dropna=False)
    print(counts)
    print("-" * 40)

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Identify the columns starting with LLM_
llm_cols_shots = [col for col in test_df.columns if col.endswith('_Few_Shot')]
ground_truth = test_df['majority_Label']

metrics_summary = []

for col in llm_cols_shots:
    # Ensure both predictions and ground truth are clean strings
    y_pred = test_df[col]
    y_true = ground_truth

    # Calculate performance metrics
    # average='weighted' accounts for label imbalance in your majority_Label
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    metrics_summary.append({
        'Model Name': col.replace('LLM_', ''),
        'Accuracy': accuracy,
        'Precision (Weighted)': precision,
        'Recall (Weighted)': recall,
        'F1 Score (Weighted)': f1
    })

# Create the summary DataFrame and sort by best performance
performance_df = pd.DataFrame(metrics_summary).sort_values(by='F1 Score (Weighted)', ascending=False)

# Round all numerical columns to two decimal places
performance_df = performance_df.round(2)

performance_df

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# 1. Identify both Zero-Shot and Few-Shot columns
llm_cols = [col for col in test_df.columns if col.startswith('LLM_') or col.endswith('_Few_Shot')]
ground_truth = test_df['majority_Label']
metrics_summary = []

for col in llm_cols:
    # Ensure data is clean
    y_pred = test_df[col].astype(str).str.strip()
    y_true = ground_truth.astype(str).str.strip()

    # Calculate performance metrics (weighted to account for label imbalance)
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # Determine Prompt Type for the final table
    prompt_type = 'Few-Shot' if 'Few_Shot' in col else 'Zero-Shot'
    # Clean model name (e.g., 'LLM_gpt-5_Few_Shot' -> 'gpt-5')
    clean_name = col.replace('LLM_', '').replace('_Few_Shot', '').replace('_Zero_Shot', '')

    metrics_summary.append({
        'Model Name': clean_name,
        'Prompt Type': prompt_type,
        'Accuracy': accuracy,
        'Precision (W)': precision,
        'Recall (W)': recall,
        'F1 Score (W)': f1
    })

# 2. Create the final combined DataFrame
combined_performance_df = pd.DataFrame(metrics_summary)

# 3. Sort by F1 Score (Highest to Lowest) and round
combined_performance_df = combined_performance_df.sort_values(by='F1 Score (W)', ascending=False).round(2)

combined_performance_df

# Other Models: Zero-Shot + Few-Shot

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# 1. Identify both Zero-Shot and Few-Shot columns
llm_cols = [col for col in test_data.columns if col.startswith('LLM_') or col.endswith('_Few_Shot')]
ground_truth = test_data['majority_Label']
metrics_summary = []

for col in llm_cols:
    # Ensure data is clean
    y_pred = test_data[col].astype(str).str.strip()
    y_true = ground_truth.astype(str).str.strip()

    # Calculate performance metrics (weighted to account for label imbalance)
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # Determine Prompt Type for the final table
    prompt_type = 'Few-Shot' if 'Few_Shot' in col else 'Zero-Shot'
    # Clean model name (e.g., 'LLM_gpt-5_Few_Shot' -> 'gpt-5')
    clean_name = col.replace('LLM_', '').replace('_Few_Shot', '').replace('_Zero_Shot', '')

    metrics_summary.append({
        'Model Name': clean_name,
        'Prompt Type': prompt_type,
        'Accuracy': accuracy,
        'Precision (W)': precision,
        'Recall (W)': recall,
        'F1 Score (W)': f1
    })

# 2. Create the final combined DataFrame
combined_performance_df = pd.DataFrame(metrics_summary)

# 3. Sort by F1 Score (Highest to Lowest) and round
combined_performance_df = combined_performance_df.sort_values(by='F1 Score (W)', ascending=False).round(2)

combined_performance_df

In [ ]:
# Purpose: Define the Task 1 prompt for classifying trust, distrust, both, or neither.
import time
from tqdm import tqdm

# 1. Variables defined in your cell
model_names = ["openai/gpt-oss-20b", "mistralai/Mixtral-8x7B-Instruct-v0.1", "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8", "gemma-3-27b-it"]
prompts = [task_1_prompt, task_1_prompt_few_shot]
prompt_labels = ["ZeroShot", "FewShot"]

# 2. Your original function structure
def classify_togetherAI_models(text, Model, Prompt):
    response = client.chat.completions.create(
        model=Model,
        messages=[
            {
                "role": "user",
                "content": Prompt + text
            }
        ],
        temperature=0,
        stream=False
    )
    return response.choices[0].message.content

tqdm.pandas()

# 3. Nested loops to apply for each model and both prompt settings
# Updated loop with error handling to skip problematic rows
for Model in model_names:
    for i, prompt_text in enumerate(prompts):
        setting = prompt_labels[i]
        clean_model_name = Model.split('/')[-1]
        column_name = f"{clean_model_name}_{setting}"

        print(f"Processing: {column_name}")

        def safe_classify(row):
            try:
                return classify_togetherAI_models(row['text'], Model, prompt_text)
            except Exception as e:
                # This will catch the error at row 94 and keep going
                return f"ERROR: {str(e)}"

        test_data[column_name] = test_data.progress_apply(safe_classify, axis=1)

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
# List of columns to clean
model_cols = [
    'gpt-oss-20b_ZeroShot', 'gpt-oss-20b_FewShot',
    'Mixtral-8x7B-Instruct-v0.1_ZeroShot', 'Mixtral-8x7B-Instruct-v0.1_FewShot',
    'Llama-4-Maverick-17B-128E-Instruct-FP8_ZeroShot', 'Llama-4-Maverick-17B-128E-Instruct-FP8_FewShot'
]

# Remove all brackets from the values in these columns
for col in model_cols:
    test_data[col] = test_data[col].str.replace('[', '', regex=False).str.replace(']', '', regex=False).str.replace(' ', '', regex=False)

# List of the two Mixtral columns to clean
mixtral_cols = ['Mixtral-8x7B-Instruct-v0.1_ZeroShot', 'Mixtral-8x7B-Instruct-v0.1_FewShot']

for col in mixtral_cols:
    # If "Both" is found in the string, replace the whole thing with "Both"
    test_data[col] = test_data[col].apply(lambda x: 'Both' if 'Both' in str(x) else x)
    test_data[col] = test_data[col].apply(lambda x: 'Neither' if 'Neither' in str(x) else x)
    test_data[col] = test_data[col].apply(lambda x: 'Distrust' if 'Distrust' in str(x) else x)

test_data['gpt-oss-20b_FewShot'] = test_data['gpt-oss-20b_FewShot'].replace(
    'I’m sorry, but I can’t help with that.', 'Neither'
)

# # Print distributions to verify the long explanations are gone
# for col in mixtral_cols:
#     print(f"\n--- Updated {col} Distribution ---")
#     print(test_data[col].value_counts())

# Print the distributions again to verify they are cleaned and merged
for col in model_cols:
    print(f"\n---{col} Distribution ---")
    print(test_data[col].value_counts())

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# 1. Identify all prediction columns (both old and new)
# We look for columns starting with 'LLM_' or ending with Shot variations
llm_cols = [col for col in test_data.columns if
            col.startswith('LLM_') or
            col.endswith('_ZeroShot') or col.endswith('_FewShot')]

ground_truth = test_data['majority_Label']
metrics_summary = []

for col in llm_cols:
    # Ensure data is clean (handle potential NaNs from skipped rows)
    y_pred = test_data[col].fillna("API_ERROR").astype(str).str.strip()
    y_true = ground_truth.astype(str).str.strip()

    # Calculate performance metrics (weighted to account for label imbalance)
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # Determine Prompt Type (handling both '_Few_Shot' and '_FewShot')
    if 'Few_Shot' in col or 'FewShot' in col:
        prompt_type = 'Few-Shot'
    else:
        prompt_type = 'Zero-Shot'

    # Clean model name for the table
    clean_name = col.replace('LLM_', '')\
                    .replace('_Few_Shot', '').replace('_Zero_Shot', '')\
                    .replace('_FewShot', '').replace('_ZeroShot', '')

    metrics_summary.append({
        'Model Name': clean_name,
        'Prompt Type': prompt_type,
        'Accuracy': accuracy,
        'Precision (W)': precision,
        'Recall (W)': recall,
        'F1 Score (W)': f1
    })

# 1. Define the name shortening mapping
name_replacements = {
    'Llama-4-Maverick-17B-128E-Instruct-FP8': 'Llama-4',
    'Mixtral-8x7B-Instruct-v0.1': 'Mixtral-8x7B'
}

# 2. Apply replacements to the 'Model Name' column
combined_performance_df['Model Name'] = combined_performance_df['Model Name'].replace(name_replacements)

# 4. Display the updated table
combined_performance_df

# 2. Create the final combined DataFrame
combined_performance_df = pd.DataFrame(metrics_summary)

# Define the name shortening mapping
name_replacements = {
    'Llama-4-Maverick-17B-128E-Instruct-FP8': 'Llama-4',
    'Mixtral-8x7B-Instruct-v0.1': 'Mixtral-8x7B'
}

# 2. Apply replacements to the 'Model Name' column
combined_performance_df['Model Name'] = combined_performance_df['Model Name'].replace(name_replacements)

# 3. Filter the DataFrame to remove Gemma rows (since they resulted in 0.0 metrics)
combined_performance_df = combined_performance_df[combined_performance_df['Model Name'] != 'gemma-3-27b-it']
# 3. Sort by F1 Score (Highest to Lowest) and round
combined_performance_df = combined_performance_df.sort_values(by='F1 Score (W)', ascending=False).round(2)

combined_performance_df

# Error Analysis

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# 1. Identify both Zero-Shot and Few-Shot columns
llm_cols = [col for col in test_data.columns if col.startswith('LLM_') or col.endswith('_Few_Shot')]
ground_truth = test_data['majority_Label']
metrics_summary = []

for col in llm_cols:
    # Ensure data is clean
    y_pred = test_data[col].astype(str).str.strip()
    y_true = ground_truth.astype(str).str.strip()

    # Calculate performance metrics (weighted to account for label imbalance)
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # Determine Prompt Type for the final table
    prompt_type = 'Few-Shot' if 'Few_Shot' in col else 'Zero-Shot'
    # Clean model name (e.g., 'LLM_gpt-5_Few_Shot' -> 'gpt-5')
    clean_name = col.replace('LLM_', '').replace('_Few_Shot', '').replace('_Zero_Shot', '')

    metrics_summary.append({
        'Model Name': clean_name,
        'Prompt Type': prompt_type,
        'Accuracy': accuracy,
        'Precision (W)': precision,
        'Recall (W)': recall,
        'F1 Score (W)': f1
    })

# 2. Create the final combined DataFrame
combined_performance_df = pd.DataFrame(metrics_summary)

# 3. Sort by F1 Score (Highest to Lowest) and round
combined_performance_df = combined_performance_df.sort_values(by='F1 Score (W)', ascending=False).round(2)

combined_performance_df

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
from sklearn.metrics import classification_report
import pandas as pd

# 1. Identify the best model column
best_model_col = 'LLM_gpt-5.1_Few_Shot'

# 2. Prepare the data
y_true = test_data['majority_Label'].astype(str).str.strip()
y_pred = test_data[best_model_col].astype(str).str.strip()

# 3. Create Classification Metrics Table
report = classification_report(y_true, y_pred, output_dict=True)

# Create DataFrame and remove the summary rows (accuracy, macro avg, weighted avg)
metrics_table = pd.DataFrame(report).transpose().iloc[:-3, :]

# Select and order the columns: precision, recall, and f1-score
metrics_table = metrics_table[['precision', 'recall', 'f1-score']]

# Sort by F1 Score and round
metrics_table = metrics_table.sort_values(by='f1-score', ascending=False).round(2)

print(f"--- Performance Metrics per Label ({best_model_col}) ---")
metrics_table

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
# 4. Create Mistake Distribution Table
errors = test_data[y_true != y_pred].copy()
mistake_counts = errors.groupby(['majority_Label', best_model_col]).size().reset_index(name='Count')
mistake_counts.columns = ['Actual (Ground Truth)', 'Predicted Label', 'Count']
mistake_table = mistake_counts.sort_values(by='Count', ascending=False).head(10)

print("\n--- Top 10 Most Common Mistakes ---")
mistake_table.style.hide(axis='index')

# Experiment with Stable Models

In [ ]:
# Purpose: Define the Task 1 prompt for classifying trust, distrust, both, or neither.
import time
from tqdm import tqdm
from openai import OpenAI

# 1. Define your list of models
models_to_run = ['gpt-5.2-2025-12-11', 'gpt-5.1-2025-11-13', 'gpt-5-2025-08-07', 'gpt-5-mini-2025-08-07', 'gpt-5-nano-2025-08-07', 'gpt-4o-2024-11-20', 'gpt-4o-mini-2024-07-18', 'gpt-4.1-2025-04-14', 'gpt-4.1-mini-2025-04-14', 'gpt-4.1-nano-2025-04-14']

def classify_barriers_new_prompt(text, model_name):
    retries = 2
    while retries > 0:
        try:
            # Try with temperature=0.0 first
            try:
                completion = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": task_1_prompt + text}],
                    temperature=0.0
                )
            except Exception:
                # Silently retry without temperature if the first attempt fails
                completion = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": task_1_prompt + text}]
                )

            return completion.choices[0].message.content

        except Exception as e:
            print(f"Error with {model_name}: {e}. Retrying...")
            retries -= 1
            time.sleep(5)

    return "API_ERROR"

# 2. Loop through each model and create the new columns
tqdm.pandas()

for model in models_to_run:
    print(f"Starting classification for model: {model}")
    column_name = f"LLM_{model}"

    # Apply to test_df['text']
    test_data[column_name] = test_data['text'].progress_apply(
        lambda x: classify_barriers_new_prompt(x, model)
    )

In [ ]:
# Purpose: Define the Task 1 prompt for classifying trust, distrust, both, or neither.
import time
from tqdm import tqdm
from openai import OpenAI

# 1. Define your list of models
models_to_run = ['gpt-5.2-2025-12-11', 'gpt-5.1-2025-11-13', 'gpt-5-2025-08-07', 'gpt-5-mini-2025-08-07', 'gpt-5-nano-2025-08-07', 'gpt-4o-2024-11-20', 'gpt-4o-mini-2024-07-18', 'gpt-4.1-2025-04-14', 'gpt-4.1-mini-2025-04-14', 'gpt-4.1-nano-2025-04-14']

def classify_barriers_new_prompt(text, model_name):
    retries = 2
    while retries > 0:
        try:
            # Try with temperature=0.0 first
            try:
                completion = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": task_1_prompt_few_shot + text}],
                    temperature=0.0
                )
            except Exception:
                # Silently retry without temperature if the first attempt fails
                completion = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": task_1_prompt_few_shot + text}]
                )

            return completion.choices[0].message.content

        except Exception as e:
            print(f"Error with {model_name}: {e}. Retrying...")
            retries -= 1
            time.sleep(5)

    return "API_ERROR"

# 2. Loop through each model and create the new columns
tqdm.pandas()

for model in models_to_run:
    print(f"Starting classification for model: {model}")
    column_name = f"LLM_{model}_Few_Shot"

    # Apply to test_df['text']
    test_data[column_name] = test_data['text'].progress_apply(
        lambda x: classify_barriers_new_prompt(x, model)
    )

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# 1. Identify both Zero-Shot and Few-Shot columns
llm_cols = [col for col in test_data.columns if col.startswith('LLM_') or col.endswith('_Few_Shot')]


# 2. Normalize labels by removing square brackets and whitespace
for col in llm_cols:
    test_data[col] = test_data[col].astype(str).str.replace(r'[\[\]]', '', regex=True).str.strip()

# 3. Verify the changes for one of the columns (e.g., gpt-5.2-2025-12-11)
print("Normalized values for gpt-5.2-2025-12-11:")
print(test_data['LLM_gpt-5.2-2025-12-11'].value_counts())
ground_truth = test_data['majority_Label']
metrics_summary = []

for col in llm_cols:
    # Ensure data is clean
    y_pred = test_data[col].astype(str).str.strip()
    y_true = ground_truth.astype(str).str.strip()

    # Calculate performance metrics (weighted to account for label imbalance)
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # Determine Prompt Type for the final table
    prompt_type = 'Few-Shot' if 'Few_Shot' in col else 'Zero-Shot'
    # Clean model name (e.g., 'LLM_gpt-5_Few_Shot' -> 'gpt-5')
    clean_name = col.replace('LLM_', '').replace('_Few_Shot', '').replace('_Zero_Shot', '')

    metrics_summary.append({
        'Model Name': clean_name,
        'Prompt Type': prompt_type,
        'Accuracy': accuracy,
        'Precision (W)': precision,
        'Recall (W)': recall,
        'F1 Score (W)': f1
    })

# 2. Create the final combined DataFrame
combined_performance_df = pd.DataFrame(metrics_summary)

# 3. Sort by F1 Score (Highest to Lowest) and round
combined_performance_df = combined_performance_df.sort_values(by='F1 Score (W)', ascending=False).round(2)

combined_performance_df

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
from sklearn.metrics import classification_report
import pandas as pd

# 1. Identify the best model column
best_model_col = 'LLM_gpt-5.1-2025-11-13'

# 2. Prepare the data
y_true = test_data['majority_Label'].astype(str).str.strip()
y_pred = test_data[best_model_col].astype(str).str.strip()

# 3. Create Classification Metrics Table
report = classification_report(y_true, y_pred, output_dict=True)

# Create DataFrame and remove the summary rows (accuracy, macro avg, weighted avg)
metrics_table = pd.DataFrame(report).transpose().iloc[:-3, :]

# Select and order the columns: precision, recall, and f1-score
metrics_table = metrics_table[['precision', 'recall', 'f1-score']]

# Sort by F1 Score and round
metrics_table = metrics_table.sort_values(by='f1-score', ascending=False).round(2)

print(f"--- Performance Metrics per Label ({best_model_col}) ---")
metrics_table

## 3. Task 2 dimension extraction on manually annotated data

Builds Task 2 prompts and evaluates trust/distrust dimension predictions on posts labeled Trust, Distrust, or Both.

# Task 2: Zero Shot

In [ ]:
# Purpose: Define the Task 2 prompt for extracting trust/distrust dimensions.
def build_task2_prompt(task1_label: str) -> str:
    # Common intro
    intro = f"""
Task Instruction:** You are an advanced AI model trained for text classification..
We define trust as a belief in the reliability, competence, or integrity of GenAI technologies, leading to positive expectations about their performance and ethical behavior.
On the other hand, we define distrust as skepticism or concern about the reliability, competence, or ethical implications of GenAI, leading to negative expectations or cautious behavior toward these technologies or companies.

Identify the dimensions of trust/distrust toward GenAI in text based on the definitions.
"""

    # Conditional instructions
    if task1_label == "Trust":
        section = """
Choose as many of the following dimensions that are explicitly mentioned or strongly implied in the text:
Reliability, Transparency, Familiarity, Integrity, Competence, Benevolence

Definitions:

Reliability: Consistent and accurate performance): Consistency and accuracy of the information provided by LLMs

Transparency: (Openness about how AI works): Being open and clear about how data is collected, processed, and how LLM works

Familiarity: (Previous experience with AI): Previous experience with AI or tools that makes LLMs more familiar

Integrity: (Ethical use of AI): AI tool following a set of (ethical) principles that the user finds acceptable

Competence: (Ability to perform tasks effectively): Having the capability, functionality, or features to do what users want

Benevolence:(Good intentions behind AI): Belief that LLMs want to do good to the user and have good intentions

"""
    elif task1_label == "Distrust":
        section = """
Choose as many of the following dimensions that are explicitly mentioned or strongly implied in the text:
Unreliability, Opaqueness, Unfamiliarity, Dishonesty, Incompetence, Deception, Malevolence

Definitions:

Unreliability: (Inconsistent, inaccurate results): Consistency and accuracy of the information provided by LLMs

Opaqueness: (Lack of openness or explainability): Being open and clear about how data is collected, processed, and how LLM works

Unfamiliarity: (Uncertainty due to new or unfamiliar tech): Previous experience with AI or tools that makes LLMs more familiar

Dishonesty: (Ethical concerns about lack of integrity, bias, manipulation): AI tool following a set of (ethical) principles that the user finds acceptable

Incompetence: (Fails to perform tasks effectively): Having the capability, functionality, or features to do what users want

Deception: (Misleading or fabricated responses): Belief that LLM is dishonest and potentially provides false information e.g. hallucination and make up stuff.

Malevolence: (Harmful or dangerous AI use): Belief that LLMs have the intention to harm the user, don’t want to do good to the user

"""
    elif task1_label == "Both":
        section = """
Choose as many of the following dimensions that are explicitly mentioned or strongly implied in the text:
Reliability, Transparency, Familiarity, Integrity, Competence, Benevolence,
Unreliability, Opaqueness, Unfamiliarity, Dishonesty, Incompetence, Deception, Malevolence

Definitions:

Reliability: Consistent and accurate performance): Consistency and accuracy of the information provided by LLMs

Transparency: (Openness about how AI works): Being open and clear about how data is collected, processed, and how LLM works

Familiarity: (Previous experience with AI): Previous experience with AI or tools that makes LLMs more familiar

Integrity: (Ethical use of AI): AI tool following a set of (ethical) principles that the user finds acceptable

Competence: (Ability to perform tasks effectively): Having the capability, functionality, or features to do what users want

Benevolence:(Good intentions behind AI): Belief that LLMs want to do good to the user and have good intentions

Unreliability: (Inconsistent, inaccurate results): Consistency and accuracy of the information provided by LLMs

Opaqueness: (Lack of openness or explainability): Being open and clear about how data is collected, processed, and how LLM works

Unfamiliarity: (Uncertainty due to new or unfamiliar tech): Previous experience with AI or tools that makes LLMs more familiar

Dishonesty: (Ethical concerns about lack of integrity, bias, manipulation): AI tool following a set of (ethical) principles that the user finds acceptable

Incompetence: (Fails to perform tasks effectively): Having the capability, functionality, or features to do what users want

Deception: (Misleading or fabricated responses): Belief that LLM is dishonest and potentially provides false information e.g. hallucination and make up stuff.

Malevolence: (Harmful or dangerous AI use): Belief that LLMs have the intention to harm the user, don’t want to do good to the user

"""
    else:  # Neither
        section = """
No further classification is required. Simply output "Neither".
"""

    # Add the output format instructions
    format_instructions = """

If none of the dimensions are relevant to the text, use NA.
Strictly follow this format and don’t add any other text:
[Dimension1, Dimension2, ...]
"""

    # Combine all parts with the text
    return f"""{intro}{section}{format_instructions}

Read the following text and provide the best dimensions:

"""


In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
data_dimensions= final_GT_data[final_GT_data['majority_Label']!= 'Neither']
len(final_GT_data), len(data_dimensions)

In [ ]:
# Purpose: Define the Task 2 prompt for extracting trust/distrust dimensions.
import time
from tqdm import tqdm
from openai import OpenAI

# 1. Define your list of models
models_to_run = ['gpt-5.2', 'gpt-5.1', 'gpt-5', 'gpt-5-mini', 'gpt-5-nano', 'gpt-4o', 'gpt-4o-mini', 'gpt-4.1', 'gpt-4.1-mini', 'gpt-4.1-nano' ]


# Function
def classify_task2(text, task1_label, Model):
    openai_client = client
    retries = 2

    # Build prompt using both the label and text
    task_2_prompt = build_task2_prompt(task1_label)

    while retries > 0:
        try:
            completion = openai_client.chat.completions.create(
                model=Model,
                messages=[
                    {"role": "user", "content": task_2_prompt + text}
                ]
            )
            return completion.choices[0].message.content
        except Exception as e:
            print(e)
            print('Timeout error, retrying...')
            retries -= 1
            time.sleep(5)

    # If all retries fail
    return "API error"

# 1. Ensure tqdm is set up for pandas
tqdm.pandas()

# 2. Iterate through each model in the list
for model in models_to_run:
    print(f"Starting Task 2 classification for model: {model}")

    # Define the new column name based on your required structure
    column_name = f"LLM_task2_{model}"

    # Apply the function and store results in the new column
    # Note: We pass the 'model' variable from the loop into the function
    data_dimensions[column_name] = data_dimensions.progress_apply(
        lambda row: classify_task2(row['text'], row['majority_Label'], model),
        axis=1
    )

print("Classification complete for all models.")

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# Identify all Task 2 columns
task2_cols = [col for col in data_dimensions.columns if col.startswith('LLM_task2_')]

# Remove '[' and ']' from those columns
for col in task2_cols:
    data_dimensions[col] = data_dimensions[col].str.replace(r'[\[\]]', '', regex=True)

# Display the head to verify
data_dimensions.head()

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Setup dimensions and model columns
trust_dimensions = ['Competence', 'Familiarity', 'Integrity', 'Reliability', 'Transparency', 'Benevolence']
distrust_dimensions = ['Deception', 'Dishonesty', 'Incompetence', 'Malevolence', 'Opaqueness', 'Unreliability', 'Unfamiliarity']
model_cols = [col for col in data_dimensions.columns if col.startswith('LLM_task2_')]

results = []

def get_metrics(y_true, y_pred, model_name, dim_name, gt_type):
    return {
        'Model': model_name,
        'Dimension': dim_name,
        'GT_Type': gt_type,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

def evaluate_model_on_dims(df, dimensions, model_col):
    model_name = model_col.replace('LLM_task2_', '')
    dim_results = []

    for dim in dimensions:
        # Binary Prediction: 1 if dimension name is in the LLM output string
        y_pred = df[model_col].fillna('').apply(lambda x: 1 if dim.lower() in str(x).lower() else 0)

        # Ground Truth 1: {dimension}_majority
        maj_col = f"{dim}_majority"
        if maj_col in df.columns:
            y_true_majority = df[maj_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_majority, y_pred, model_name, dim, 'majority'))

        # Ground Truth 2: {dimension}_any
        any_col = f"{dim}_any"
        if any_col in df.columns:
            y_true_any = df[any_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_any, y_pred, model_name, dim, 'any'))

    return dim_results

# 2. Filter dataframes based on instructions
trust_df = data_dimensions[data_dimensions['majority_Label'].isin(['Trust', 'Both'])].copy()
distrust_df = data_dimensions[data_dimensions['majority_Label'].isin(['Distrust', 'Both'])].copy()

# 3. Run evaluation
for model in model_cols:
    results.extend(evaluate_model_on_dims(trust_df, trust_dimensions, model))
    results.extend(evaluate_model_on_dims(distrust_df, distrust_dimensions, model))

# 4. Create DataFrame
performance_df = pd.DataFrame(results)

# 5. Calculate Averages separately for 'majority' and 'any'
# By grouping by both 'Model' and 'GT_Type', the averages for 'majority'
# and 'any' are never mixed.
avg_perf = performance_df.groupby(['Model', 'GT_Type'])[['Accuracy', 'Precision', 'Recall', 'F1']].mean().reset_index()
avg_perf['Dimension'] = 'AVERAGE_ALL'

# Combine individual and average results
final_report = pd.concat([performance_df, avg_perf], ignore_index=True)

# Sort so that Majority and Any results are grouped separately within each model
final_report = final_report.sort_values(by=['Model', 'GT_Type', 'Dimension'])

# 1. Pivot the table for F1 scores
f1_pivot = final_report.pivot_table(
    index=['GT_Type', 'Dimension'], # Swap index order to sort by GT_Type first
    columns='Model',
    values='F1'
)

# 2. Define the order: 'majority' first, then 'any'
# Within each group, keep your custom dimension order (AVERAGE_ALL at bottom)
all_dims = sorted([d for d in final_report['Dimension'].unique() if d != 'AVERAGE_ALL'])
final_dim_order = all_dims + ['AVERAGE_ALL']

# Reindex the levels
f1_pivot = f1_pivot.reindex(['majority', 'any'], level=0)
f1_pivot = f1_pivot.reindex(final_dim_order, level=1)

# 3. Flatten for display
f1_flat = f1_pivot.reset_index()

# 4. Find the index where 'any' starts to place the thick line
# Since 'majority' comes first, the line goes above the first 'any' row
split_index = len(f1_flat[f1_flat['GT_Type'] == 'majority'])

# 5. Apply styling
styled_f1_table = f1_flat.style.highlight_max(axis=1, color='lightgreen', subset=f1_pivot.columns) \
    .format("{:.2f}", subset=f1_pivot.columns) \
    .hide(axis='index') \
    .set_caption("Model Comparison: F1 Scores (Grouped by GT_Type)") \
    .set_table_styles([
        # Standard header styling
        {'selector': 'th', 'props': [('background-color', '#ffffff'), ('color', 'black'), ('font-weight', 'bold'), ('border', '1px solid #ccc')]},
        {'selector': 'td', 'props': [('border', '1px solid #eee')]},
        # The thick line: apply a top border to all cells in the first row of the 'any' group
        {'selector': f'tr:nth-child({split_index + 1}) td', 'props': [('border-top', '3px solid black')]}
    ])

# Display the table
styled_f1_table

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Setup dimensions and model columns
trust_dimensions = ['Competence', 'Familiarity', 'Integrity', 'Reliability', 'Transparency', 'Benevolence']
distrust_dimensions = ['Deception', 'Dishonesty', 'Incompetence', 'Malevolence', 'Opaqueness', 'Unreliability', 'Unfamiliarity']
model_cols = [col for col in data_dimensions.columns if col.startswith('LLM_task2_')]

results = []

def get_metrics(y_true, y_pred, model_name, dim_name, gt_type):
    return {
        'Model': model_name,
        'Dimension': dim_name,
        'GT_Type': gt_type,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

def evaluate_model_on_dims(df, dimensions, model_col):
    model_name = model_col.replace('LLM_task2_', '')
    dim_results = []

    for dim in dimensions:
        # Binary Prediction: 1 if dimension name is in the LLM output string
        y_pred = df[model_col].fillna('').apply(lambda x: 1 if dim.lower() in str(x).lower() else 0)

        # Ground Truth 1: {dimension}_majority
        maj_col = f"{dim}_majority"
        if maj_col in df.columns:
            y_true_majority = df[maj_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_majority, y_pred, model_name, dim, 'majority'))

        # Ground Truth 2: {dimension}_any
        any_col = f"{dim}_any"
        if any_col in df.columns:
            y_true_any = df[any_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_any, y_pred, model_name, dim, 'any'))

    return dim_results

# 2. Filter dataframes based on instructions
trust_df = data_dimensions[data_dimensions['majority_Label'].isin(['Trust'])].copy()
distrust_df = data_dimensions[data_dimensions['majority_Label'].isin(['Distrust'])].copy()

# 3. Run evaluation
for model in model_cols:
    results.extend(evaluate_model_on_dims(trust_df, trust_dimensions, model))
    results.extend(evaluate_model_on_dims(distrust_df, distrust_dimensions, model))

# 4. Create DataFrame
performance_df = pd.DataFrame(results)

# 5. Calculate Averages separately for 'majority' and 'any'
# By grouping by both 'Model' and 'GT_Type', the averages for 'majority'
# and 'any' are never mixed.
avg_perf = performance_df.groupby(['Model', 'GT_Type'])[['Accuracy', 'Precision', 'Recall', 'F1']].mean().reset_index()
avg_perf['Dimension'] = 'AVERAGE_ALL'

# Combine individual and average results
final_report = pd.concat([performance_df, avg_perf], ignore_index=True)

# Sort so that Majority and Any results are grouped separately within each model
final_report = final_report.sort_values(by=['Model', 'GT_Type', 'Dimension'])

# Create a pivot to compare Models and GT_Types more easily
# 1. Pivot the table for F1 scores
f1_pivot = final_report.pivot_table(
    index=['Dimension', 'GT_Type'],
    columns='Model',
    values='F1'
)

# 2. Reorder to keep 'AVERAGE_ALL' at the bottom
all_dims = sorted([d for d in final_report['Dimension'].unique() if d != 'AVERAGE_ALL'])
final_order = all_dims + ['AVERAGE_ALL']
f1_pivot = f1_pivot.reindex(final_order, level=0)

# 3. Move 'Dimension' and 'GT_Type' from the index to the header row
f1_flat = f1_pivot.reset_index()

# 4. Apply styling to the flattened dataframe
styled_f1_table = f1_flat.style.highlight_max(axis=1, color='lightgreen', subset=f1_pivot.columns) \
    .format("{:.2f}", subset=f1_pivot.columns) \
    .hide(axis='index') \
    .set_caption("Model Comparison: F1 Scores (Dimension and GT_Type in Header)") \
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#ffffff'), ('color', 'black'), ('font-weight', 'bold'), ('border', '1px solid #ccc')]},
        {'selector': 'td', 'props': [('border', '1px solid #eee')]}
    ])

# Display the table
styled_f1_table

In [ ]:
# Purpose: Define the Task 2 prompt for extracting trust/distrust dimensions.
import time
from tqdm import tqdm

# 1. Variables defined in your cell
model_names = ["openai/gpt-oss-20b", "mistralai/Mixtral-8x7B-Instruct-v0.1", "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"]


# 2. Your original function structure
def classify_togetherAI_models(text, task1_label, Model):

    task_2_prompt = build_task2_prompt(task1_label)
    response = client.chat.completions.create(
        model=Model,
        messages=[
            {
                "role": "user",
                "content": task_2_prompt + text
            }
        ],
        temperature=0,
        stream=False
    )
    return response.choices[0].message.content

tqdm.pandas()


# 2. Iterate through each model in the list
for model in model_names:
    print(f"Starting Task 2 classification for model: {model}")

    # Define the new column name based on your required structure
    column_name = f"LLM_task2_{model}"

    # Apply the function and store results in the new column
    # Note: We pass the 'model' variable from the loop into the function
    data_dimensions[column_name] = data_dimensions.progress_apply(
        lambda row: classify_togetherAI_models(row['text'], row['majority_Label'], model),
        axis=1
    )

print("Classification complete for all models.")

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
# Identify all Task 2 columns
task2_cols = [col for col in data_dimensions.columns if col.startswith('LLM_task2_')]

# Remove '[' and ']' from those columns
for col in task2_cols:
    data_dimensions[col] = data_dimensions[col].str.replace(r'[\[\]]', '', regex=True)

# 1. Define the name shortening mapping
name_replacements = {
    'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': 'Llama-4',
    'mistralai/Mixtral-8x7B-Instruct-v0.1': 'Mixtral-8x7B',
    'openai/gpt-oss-20b': 'gpt-oss-20b'
}


import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Setup dimensions and model columns
trust_dimensions = ['Competence', 'Familiarity', 'Integrity', 'Reliability', 'Transparency', 'Benevolence']
distrust_dimensions = ['Deception', 'Dishonesty', 'Incompetence', 'Malevolence', 'Opaqueness', 'Unreliability', 'Unfamiliarity']
model_cols = [col for col in data_dimensions.columns if col.startswith('LLM_task2_')]

results = []

def get_metrics(y_true, y_pred, model_name, dim_name, gt_type):
    return {
        'Model': model_name,
        'Dimension': dim_name,
        'GT_Type': gt_type,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

def evaluate_model_on_dims(df, dimensions, model_col):
    model_name = model_col.replace('LLM_task2_', '')
    dim_results = []

    for dim in dimensions:
        # Binary Prediction: 1 if dimension name is in the LLM output string
        y_pred = df[model_col].fillna('').apply(lambda x: 1 if dim.lower() in str(x).lower() else 0)

        # Ground Truth 1: {dimension}_majority
        maj_col = f"{dim}_majority"
        if maj_col in df.columns:
            y_true_majority = df[maj_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_majority, y_pred, model_name, dim, 'majority'))

        # Ground Truth 2: {dimension}_any
        any_col = f"{dim}_any"
        if any_col in df.columns:
            y_true_any = df[any_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_any, y_pred, model_name, dim, 'any'))

    return dim_results

# 2. Filter dataframes based on instructions
trust_df = data_dimensions[data_dimensions['majority_Label'].isin(['Trust', 'Both'])].copy()
distrust_df = data_dimensions[data_dimensions['majority_Label'].isin(['Distrust', 'Both'])].copy()

# 3. Run evaluation
for model in model_cols:
    results.extend(evaluate_model_on_dims(trust_df, trust_dimensions, model))
    results.extend(evaluate_model_on_dims(distrust_df, distrust_dimensions, model))

# 4. Create DataFrame
performance_df = pd.DataFrame(results)

performance_df['Model'] = performance_df['Model'].replace(name_replacements)

# 5. Calculate Averages separately for 'majority' and 'any'
# By grouping by both 'Model' and 'GT_Type', the averages for 'majority'
# and 'any' are never mixed.
avg_perf = performance_df.groupby(['Model', 'GT_Type'])[['Accuracy', 'Precision', 'Recall', 'F1']].mean().reset_index()
avg_perf['Dimension'] = 'AVERAGE_ALL'

# Combine individual and average results
final_report = pd.concat([performance_df, avg_perf], ignore_index=True)

# Sort so that Majority and Any results are grouped separately within each model
final_report = final_report.sort_values(by=['Model', 'GT_Type', 'Dimension'])

# 1. Pivot the table for F1 scores
f1_pivot = final_report.pivot_table(
    index=['GT_Type', 'Dimension'], # Swap index order to sort by GT_Type first
    columns='Model',
    values='F1'
)

# 2. Define the order: 'majority' first, then 'any'
# Within each group, keep your custom dimension order (AVERAGE_ALL at bottom)
all_dims = sorted([d for d in final_report['Dimension'].unique() if d != 'AVERAGE_ALL'])
final_dim_order = all_dims + ['AVERAGE_ALL']

# Reindex the levels
f1_pivot = f1_pivot.reindex(['majority', 'any'], level=0)
f1_pivot = f1_pivot.reindex(final_dim_order, level=1)

# 3. Flatten for display
f1_flat = f1_pivot.reset_index()

# 4. Find the index where 'any' starts to place the thick line
# Since 'majority' comes first, the line goes above the first 'any' row
split_index = len(f1_flat[f1_flat['GT_Type'] == 'majority'])

# 5. Apply styling
styled_f1_table = f1_flat.style.highlight_max(axis=1, color='lightgreen', subset=f1_pivot.columns) \
    .format("{:.2f}", subset=f1_pivot.columns) \
    .hide(axis='index') \
    .set_caption("Model Comparison: F1 Scores (Grouped by GT_Type)") \
    .set_table_styles([
        # Standard header styling
        {'selector': 'th', 'props': [('background-color', '#ffffff'), ('color', 'black'), ('font-weight', 'bold'), ('border', '1px solid #ccc')]},
        {'selector': 'td', 'props': [('border', '1px solid #eee')]},
        # The thick line: apply a top border to all cells in the first row of the 'any' group
        {'selector': f'tr:nth-child({split_index + 1}) td', 'props': [('border-top', '3px solid black')]}
    ])

# Display the table
styled_f1_table

# Task 2: Few Shot

In [ ]:
# Purpose: Define the Task 2 prompt for extracting trust/distrust dimensions.
import pandas as pd
from textwrap import shorten

TRUST_DIMS = ["Reliability", "Transparency", "Familiarity", "Integrity", "Competence", "Benevolence"]
DISTRUST_DIMS = ["Unreliability", "Opaqueness", "Unfamiliarity", "Dishonesty", "Incompetence", "Deception", "Malevolence"]
BOTH_DIMS = TRUST_DIMS + DISTRUST_DIMS

def _dims_for_task(task1_label: str):
    if task1_label == "Trust":
        return TRUST_DIMS
    if task1_label == "Distrust":
        return DISTRUST_DIMS
    if task1_label == "Both":
        return BOTH_DIMS
    return []  # Neither

def _examples_block_binary(X, text_col: str, dims: list[str], k) -> str:
    """
    For each dimension in `dims`, pick 3 short examples (<500 chars)
    from rows where X[dim] == 1.
    """
    lines = ["\nHere are Few-shot examples for each dimension:"]

    for d in dims:
        if d not in X.columns:
            continue  # skip if column missing

        # Filter rows where this dimension is present
        subset = X.loc[X[d] == 1, text_col].dropna()

        # Keep only short texts (<100 characters)
        subset = subset[subset.str.len() < 300]

        if subset.empty:
            continue

        # Take k examples (or fewer if less available)
        examples = subset.sample(k)

        lines.append(f"\n- {d}:")
        for txt in examples:
            lines.append(f'  • Text: "{txt}"\n    Label: {d}')

    return "\n".join(lines) + "\n"


def build_task2_prompt_shots(task1_label: str, X: pd.DataFrame, text_col: str = "text", k: int = 3, random_sample: bool = False) -> str:
    """
    Build prompt and auto-attach k examples per relevant dimension using X[X[Dim]==1][text_col].
    """
    intro = f"""
Task Instruction:** You are an advanced AI model trained for text classification..
We define trust as a belief in the reliability, competence, or integrity of GenAI technologies, leading to positive expectations about their performance and ethical behavior.
On the other hand, we define distrust as skepticism or concern about the reliability, competence, or ethical implications of GenAI, leading to negative expectations or cautious behavior toward these technologies or companies.

Identify the dimensions of trust/distrust toward GenAI in text based on {task1_label}.
"""

    if task1_label == "Trust":
        section = """
Choose one or more of the following dimensions:
Reliability, Transparency, Familiarity, Integrity, Competence, Benevolence

Definitions:

Reliability: Consistent and accurate performance; Consistency and accuracy of the information provided by LLMs

Transparency: Openness about how AI works; Being open and clear about how data is collected, processed, and how LLM works

Familiarity: Previous experience with AI; Previous experience with AI or tools that makes LLMs more familiar

Integrity: Ethical use of AI; AI tool following a set of (ethical) principles that the user finds acceptable

Competence: Ability to perform tasks effectively; Having the capability, functionality, or features to do what users want

Benevolence: Good intentions behind AI; Belief that LLMs want to do good to the user and have good intentions


"""
    elif task1_label == "Distrust":
        section = """
Choose one or more of the following dimensions:
Unreliability, Opaqueness, Unfamiliarity, Dishonesty, Incompetence, Deception, Malevolence

Definitions:

Unreliability: Inconsistent, inaccurate results; Belieft that LLMs are not consistent, accurate in the information provided

Opaqueness: Lack of openness or explainability; Being open and clear about how data is collected, processed, and how LLM works

Unfamiliarity: Uncertainty due to new or unfamiliar tech; Previous experience with AI or tools that makes LLMs more familiar

Dishonesty: Ethical concerns about lack of integrity, bias, manipulation; AI tool following a set of (ethical) principles that the user finds acceptable

Incompetence: Fails to perform tasks effectively; Having the capability, functionality, or features to do what users want

Deception: Misleading or fabricated responses; Belief that LLM is dishonest and potentially provides false information e.g. hallucination and make up stuff.

Malevolence: Harmful or dangerous AI use; Belief that LLMs have the intention to harm the user, don’t want to do good to the user

"""
    elif task1_label == "Both":
        section = """
Choose one or more of the following dimensions:

Reliability: Consistent and accurate performance; Consistency and accuracy of the information provided by LLMs

Transparency: Openness about how AI works; Being open and clear about how data is collected, processed, and how LLM works

Familiarity: Previous experience with AI; Previous experience with AI or tools that makes LLMs more familiar

Integrity: Ethical use of AI; AI tool following a set of (ethical) principles that the user finds acceptable

Competence: Ability to perform tasks effectively; Having the capability, functionality, or features to do what users want

Benevolence: Good intentions behind AI; Belief that LLMs want to do good to the user and have good intentions

Unreliability: Inconsistent, inaccurate results; Belieft that LLMs are not consistent, accurate in the information provided

Opaqueness: Lack of openness or explainability; Being open and clear about how data is collected, processed, and how LLM works

Unfamiliarity: Uncertainty due to new or unfamiliar tech; Previous experience with AI or tools that makes LLMs more familiar

Dishonesty: Ethical concerns about lack of integrity, bias, manipulation; AI tool following a set of (ethical) principles that the user finds acceptable

Incompetence: Fails to perform tasks effectively; Having the capability, functionality, or features to do what users want

Deception: Misleading or fabricated responses; Belief that LLM is dishonest and potentially provides false information e.g. hallucination and make up stuff.

Malevolence: Harmful or dangerous AI use; Belief that LLMs have the intention to harm the user, don’t want to do good to the user

"""
    else:
        section = """
No further classification is required. Simply output "Neither".
"""

    dims = _dims_for_task(task1_label)
    examples = _examples_block_binary(validation_df, text_col='text', dims=dims, k=k) if dims else ""

    format_instructions = """
After labeling the text with dimensions, add a brief cue that explains why you chose each of the dimensions.

Use the following format for your response:
[Dimension 1, Dimension 2, ...]


Strictly follow this format and don’t add any other text.
"""

    return f"""{intro}{section}{examples}{format_instructions}

Read the following text and provide the best dimensions:

"""


In [ ]:
# Purpose: Define the Task 2 prompt for extracting trust/distrust dimensions.
import pandas as pd
from textwrap import shorten

TRUST_DIMS = ["Reliability", "Transparency", "Familiarity", "Integrity", "Competence", "Benevolence"]
DISTRUST_DIMS = ["Unreliability", "Opaqueness", "Unfamiliarity", "Dishonesty", "Incompetence", "Deception", "Malevolence"]
BOTH_DIMS = TRUST_DIMS + DISTRUST_DIMS

def _dims_for_task(task1_label: str):
    if task1_label == "Trust":
        return TRUST_DIMS
    if task1_label == "Distrust":
        return DISTRUST_DIMS
    if task1_label == "Both":
        return BOTH_DIMS
    return []

def _examples_block_binary(X, text_col: str, dims: list[str], k, random_sample=False) -> str:
    """
    For each dimension in `dims`, pick k short examples (<300 chars)
    from rows where X['{dim}_majority'] == 1.
    """
    lines = ["\n### Few-shot examples for reference:"]

    for d in dims:
        # Construct the correct column name using the _majority suffix
        gt_col = f"{d}_majority"

        if gt_col not in X.columns:
            # Debug print to help identify missing columns if needed
            # print(f"Warning: {gt_col} not found in DataFrame columns.")
            continue

        # Filter for rows where the ground truth for this dimension is 1
        subset = X.loc[X[gt_col] == 1, text_col].dropna()
        subset = subset[subset.str.len() < 300]

        if subset.empty:
            continue

        # Sample k examples
        examples = subset.sample(min(k, len(subset)), random_state=42 if not random_sample else None)

        lines.append(f"\nDimension: {d}")
        for i, txt in enumerate(examples, 1):
            clean_text = txt.replace('\n', ' ')
            lines.append(f"  Example {i}: \"{clean_text}\"")

    return "\n".join(lines) + "\n"

def build_task2_prompt_shots(task1_label: str, X: pd.DataFrame, text_col: str = "text", k: int = 3, random_sample: bool = False) -> str:
    intro = f"""
Task Instruction:** You are an advanced AI model trained for text classification..
We define trust as a belief in the reliability, competence, or integrity of GenAI technologies, leading to positive expectations about their performance and ethical behavior.
On the other hand, we define distrust as skepticism or concern about the reliability, competence, or ethical implications of GenAI, leading to negative expectations or cautious behavior toward these technologies or companies.

Identify the dimensions of trust/distrust toward GenAI in text based on {task1_label}.
"""

    # Definitions block
    trust_defs = """
Reliability: Consistent and accurate performance; Consistency and accuracy of the information provided by LLMs
Transparency: Openness about how AI works; Being open and clear about how data is collected, processed, and how LLM works
Familiarity: Previous experience with AI; Previous experience with AI or tools that makes LLMs more familiar
Integrity: Ethical use of AI; AI tool following a set of (ethical) principles that the user finds acceptable
Competence: Ability to perform tasks effectively; Having the capability, functionality, or features to do what users want
Benevolence: Good intentions behind AI; Belief that LLMs want to do good to the user and have good intentions
"""
    distrust_defs = """
Unreliability: Inconsistent, inaccurate results; Belieft that LLMs are not consistent, accurate in the information provided
Opaqueness: Lack of openness or explainability; Being open and clear about how data is collected, processed, and how LLM works
Unfamiliarity: Uncertainty due to new or unfamiliar tech; Previous experience with AI or tools that makes LLMs more familiar
Dishonesty: Ethical concerns about lack of integrity, bias, manipulation; AI tool following a set of (ethical) principles that the user finds acceptable
Incompetence: Fails to perform tasks effectively; Having the capability, functionality, or features to do what users want
Deception: Misleading or fabricated responses; Belief that LLM is dishonest and potentially provides false information e.g. hallucination and make up stuff.
Malevolence: Harmful or dangerous AI use; Belief that LLMs have the intention to harm the user, don’t want to do good to the user
"""

    if task1_label == "Trust":
        section = f"\nChoose one or more of the following dimensions:\n{', '.join(TRUST_DIMS)}\n\nDefinitions:\n{trust_defs}"
    elif task1_label == "Distrust":
        section = f"\nChoose one or more of the following dimensions:\n{', '.join(DISTRUST_DIMS)}\n\nDefinitions:\n{distrust_defs}"
    elif task1_label == "Both":
        section = f"\nChoose one or more of the following dimensions:\n{', '.join(BOTH_DIMS)}\n\nDefinitions:\n{trust_defs}{distrust_defs}"
    else:
        section = "\nNo further classification is required. Simply output 'Neither'."

    # Get dimension list and generate shots
    dims = _dims_for_task(task1_label)
    examples = _examples_block_binary(X, text_col=text_col, dims=dims, k=k, random_sample=random_sample) if dims else ""

    format_instructions = """
\n Use the following format for your response:
[Dimension 1, Dimension 2, ...]
Strictly follow this format and don't add any other text.
"""

    return f"{intro}{section}{examples}{format_instructions}\nRead the following text and provide the best dimensions:\n "

In [ ]:
# Purpose: Define the Task 2 prompt for extracting trust/distrust dimensions.
# Test script to visualize the prompts for each Task 1 label
labels_to_test = ["Trust", "Distrust", "Both"]

for label in labels_to_test:
    print(f"{'='*30} PROMPT FOR: {label} {'='*30}")

    # Generate the prompt using your existing function
    # Passing validation_df or data_dimensions as X
    test_prompt = build_task2_prompt_shots(
        task1_label=label,
        X=data_dimensions, # or validation_df
        k=3
    )

    print(test_prompt)
    print("\n" + "="*80 + "\n")

In [ ]:
# Purpose: Define the Task 2 prompt for extracting trust/distrust dimensions.
#Filter rows used for the shots

# List of all dimensions to check
all_dimensions = [
    "Reliability", "Transparency", "Familiarity", "Integrity", "Competence", "Benevolence",
    "Unreliability", "Opaqueness", "Unfamiliarity", "Dishonesty", "Incompetence", "Deception", "Malevolence"
]

shot_indices = set()

for dim in all_dimensions:
    gt_col = f"{dim}_majority"
    if gt_col in data_dimensions.columns:
        # Replicate the exact filtering and sampling logic from build_task2_prompt_shots
        subset = data_dimensions.loc[data_dimensions[gt_col] == 1].dropna(subset=['text'])
        subset = subset[subset['text'].str.len() < 300]

        if not subset.empty:
            # Re-run sampling with the same random_state to get the same rows
            samples = subset.sample(min(3, len(subset)), random_state=42)
            shot_indices.update(samples.index.tolist())

# Create a new DataFrame containing only the rows used as shots
used_shots_df = data_dimensions.loc[list(shot_indices)]

# Optional: To remove these from your evaluation set, you can do:
evaluation_df = data_dimensions.drop(index=list(shot_indices))

len(used_shots_df), len(evaluation_df),  len(data_dimensions)

In [ ]:
# Purpose: Save the current processed dataframe or analysis output for reuse/reproducibility.
import time
from tqdm import tqdm
from openai import OpenAI

# 1. Define your list of models
models_to_run = ['gpt-5.2', 'gpt-5.1', 'gpt-5', 'gpt-5-mini', 'gpt-5-nano', 'gpt-4o', 'gpt-4o-mini', 'gpt-4.1', 'gpt-4.1-mini', 'gpt-4.1-nano' ]


# Pre-generate the three possible prompt versions
prompts = {
    label: build_task2_prompt_shots(label, data_dimensions)
    for label in ["Trust", "Distrust", "Both"]
}


def classify_gpt_models_task2_few_shots(text, task1_label, Model):
    retries = 2

    # Get the correct few-shot prompt block for this label
    # Ensure build_task2_prompt_shots is defined and accepts data_dimensions if needed
    task_2_prompt_few_shots = prompts.get(task1_label, "")

    while retries > 0:
        try:
            # Prepare arguments for the API call
            api_args = {
                "model": Model,
                "messages": [{"role": "user", "content": task_2_prompt_few_shots + text}],
                "stream": False
            }

            # Condition: Set temperature=0 for models NOT in the gpt-5/mini/nano list
            # For 'gpt-5', 'gpt-5-mini', and 'gpt-5-nano', the parameter is omitted
            if Model not in ['gpt-5', 'gpt-5-mini', 'gpt-5-nano']:
                api_args["temperature"] = 0

            completion = client.chat.completions.create(**api_args)
            return completion.choices[0].message.content

        except Exception as e:
            print(e)
            print('Timeout error, retrying...')
            retries -= 1
            time.sleep(5)

    return "API error"

# ---- Run over your evaluation_df ----
tqdm.pandas()

# 2. Iterate through each model in the list
for model in models_to_run:
    print(f"Starting Task 2 classification for model: {model}")

    column_name = f"LLM_task2_{model}_few_shot"

    # Check if this model has already been run (to allow resuming if interrupted)
    if column_name in evaluation_df.columns:
        print(f"Column {column_name} already exists. Skipping...")
        continue

    # Apply the classification function
    evaluation_df[column_name] = evaluation_df.progress_apply(
        lambda row: classify_gpt_models_task2_few_shots(
            text=row['text'],
            task1_label=row['majority_Label'],
            Model=model
        ),
        axis=1
    )

    # NEW: Immediately save to CSV after each model finishes
    # This ensures that even if the next model fails, you have these results stored.
    checkpoint_filename = "evaluation_task2_checkpoints.csv"
    evaluation_df.to_csv(checkpoint_filename, index=False)
    print(f"Results for {model} saved to {checkpoint_filename}.")

print("Classification complete for all models.")

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# 1. Identify all columns that start with 'LLM_task2'
llm_task2_cols = [col for col in evaluation_df.columns if col.startswith('LLM_task2')]

# 2. Loop through each column and remove the brackets
for col in llm_task2_cols:
    # We use .astype(str) to ensure the values are strings before replacing
    evaluation_df[col] = evaluation_df[col].astype(str).str.replace('[', '', regex=False).str.replace(']', '', regex=False)

evaluation_df.head()

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Setup dimensions and model columns using evaluation_df
trust_dimensions = ['Competence', 'Familiarity', 'Integrity', 'Reliability', 'Transparency', 'Benevolence']
distrust_dimensions = ['Deception', 'Dishonesty', 'Incompetence', 'Malevolence', 'Opaqueness', 'Unreliability', 'Unfamiliarity']


# 1. Define the name shortening mapping
name_replacements = {
    'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': 'Llama-4',
    'mistralai/Mixtral-8x7B-Instruct-v0.1': 'Mixtral-8x7B',
    'openai/gpt-oss-20b': 'gpt-oss-20b'
}


# Separate standard columns from few_shot columns
all_model_cols = [col for col in evaluation_df.columns if col.startswith('LLM_task2_')]
few_shot_cols = [col for col in all_model_cols if 'few_shot' in col]
standard_cols = [col for col in all_model_cols if 'few_shot' not in col]

results = []

def get_metrics(y_true, y_pred, model_name, dim_name, gt_type):
    return {
        'Model': model_name,
        'Dimension': dim_name,
        'GT_Type': gt_type,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

def evaluate_model_on_dims(df, dimensions, model_col):
    model_name = model_col.replace('LLM_task2_', '')
    dim_results = []
    for dim in dimensions:
        y_pred = df[model_col].fillna('').apply(lambda x: 1 if dim.lower() in str(x).lower() else 0)

        maj_col = f"{dim}_majority"
        if maj_col in df.columns:
            y_true_majority = df[maj_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_majority, y_pred, model_name, dim, 'majority'))

        any_col = f"{dim}_any"
        if any_col in df.columns:
            y_true_any = df[any_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_any, y_pred, model_name, dim, 'any'))
    return dim_results

# 2. Filter evaluation_df based on instructions
trust_df = evaluation_df[evaluation_df['majority_Label'].isin(['Trust', 'Both'])].copy()
distrust_df = evaluation_df[evaluation_df['majority_Label'].isin(['Distrust', 'Both'])].copy()

# 3. Run evaluation for all models
for model in all_model_cols:
    results.extend(evaluate_model_on_dims(trust_df, trust_dimensions, model))
    results.extend(evaluate_model_on_dims(distrust_df, distrust_dimensions, model))

# 4. Create DataFrame and Calculate Averages
performance_df = pd.DataFrame(results)


avg_perf = performance_df.groupby(['Model', 'GT_Type'])[['Accuracy', 'Precision', 'Recall', 'F1']].mean().reset_index()
avg_perf['Dimension'] = 'AVERAGE_ALL'
final_report = pd.concat([performance_df, avg_perf], ignore_index=True)

# 5. Pivot for F1 and handle ordering
f1_pivot = final_report.pivot_table(index=['GT_Type', 'Dimension'], columns='Model', values='F1')

# Determine column order: Standard models first, then Few-Shot models
ordered_models = sorted([m.replace('LLM_task2_', '') for m in standard_cols]) + \
                 sorted([m.replace('LLM_task2_', '') for m in few_shot_cols])
f1_pivot = f1_pivot.reindex(columns=ordered_models)

# Reindex GT_Type and Dimensions
all_dims = sorted([d for d in final_report['Dimension'].unique() if d != 'AVERAGE_ALL'])
final_dim_order = all_dims + ['AVERAGE_ALL']
f1_pivot = f1_pivot.reindex(['majority', 'any'], level=0).reindex(final_dim_order, level=1)

# 6. Styling with thick lines
f1_flat = f1_pivot.reset_index()
split_any_index = len(f1_flat[f1_flat['GT_Type'] == 'majority'])
# The column index where few-shot models start (GT_Type + Dimension = 2 columns offset)
few_shot_start_col_idx = len(standard_cols) + 2

styled_f1_table = f1_flat.style.highlight_max(axis=1, color='lightgreen', subset=f1_pivot.columns) \
    .format("{:.2f}", subset=f1_pivot.columns) \
    .hide(axis='index') \
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#fff'), ('font-weight', 'bold'), ('border', '1px solid #ccc')]},
        # Horizontal line to separate 'majority' from 'any'
        {'selector': f'tr:nth-child({split_any_index + 1}) td', 'props': [('border-top', '3px solid black')]},
        # Vertical line to separate standard from few-shot models
        {'selector': f'td:nth-child({few_shot_start_col_idx + 1})', 'props': [('border-left', '3px solid black')]},
        {'selector': f'th:nth-child({few_shot_start_col_idx + 1})', 'props': [('border-left', '3px solid black')]}
    ])

styled_f1_table

# 6. Styling with thick lines
f1_flat = f1_pivot.reset_index()
split_any_index = len(f1_flat[f1_flat['GT_Type'] == 'majority'])

# Recalculate few_shot_start_col_idx based on the new shortened standard column count
few_shot_start_col_idx = len(short_standard) + 2

styled_f1_table = f1_flat.style.highlight_max(axis=1, color='lightgreen', subset=f1_pivot.columns) \
    .format("{:.2f}", subset=f1_pivot.columns) \
    .hide(axis='index') \
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#fff'), ('font-weight', 'bold'), ('border', '1px solid #ccc'), ('text-align', 'center')]},
        # Horizontal line to separate 'majority' from 'any'
        {'selector': f'tr:nth-child({split_any_index + 1}) td', 'props': [('border-top', '3px solid black')]},
        # Vertical line to separate standard from few-shot models
        {'selector': f'td:nth-child({few_shot_start_col_idx + 1})', 'props': [('border-left', '3px solid black')]},
        {'selector': f'th:nth-child({few_shot_start_col_idx + 1})', 'props': [('border-left', '3px solid black')]}
    ])

styled_f1_table

In [ ]:
# Purpose: Define the Task 2 prompt for extracting trust/distrust dimensions.
import time
from tqdm import tqdm

# 1. Variables defined in your cell
model_names = ["openai/gpt-oss-20b", "mistralai/Mixtral-8x7B-Instruct-v0.1", "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"]

# Pre-generate the three possible prompt versions
prompts = {
    label: build_task2_prompt_shots(label, data_dimensions)
    for label in ["Trust", "Distrust", "Both"]
}


# 2. Your original function structure
def classify_togetherAI_models(text, task1_label, Model):

    task_2_prompt_few_shots = prompts.get(task1_label, "")
    response = client.chat.completions.create(
        model=Model,
        messages=[
            {
                "role": "user",
                "content": task_2_prompt_few_shots + text
            }
        ],
        temperature=0,
        stream=False
    )
    return response.choices[0].message.content



# ---- Run over your evaluation_df ----
tqdm.pandas()

# 2. Iterate through each model in the list
for model in model_names:
    print(f"Starting Task 2 classification for model: {model}")

    column_name = f"LLM_task2_{model}_few_shot"

    # Apply the classification function
    evaluation_df[column_name] = evaluation_df.progress_apply(
        lambda row: classify_togetherAI_models(
            text=row['text'],
            task1_label=row['majority_Label'],
            Model=model
        ),
        axis=1
    )


print("Classification complete for all models.")

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Setup dimensions and model columns using evaluation_df
trust_dimensions = ['Competence', 'Familiarity', 'Integrity', 'Reliability', 'Transparency', 'Benevolence']
distrust_dimensions = ['Deception', 'Dishonesty', 'Incompetence', 'Malevolence', 'Opaqueness', 'Unreliability', 'Unfamiliarity']


# 1. Define the name shortening mapping
name_replacements = {
    'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': 'Llama-4',
    'mistralai/Mixtral-8x7B-Instruct-v0.1': 'Mixtral-8x7B',
    'openai/gpt-oss-20b': 'gpt-oss-20b'
}


# Separate standard columns from few_shot columns
all_model_cols = [col for col in evaluation_df.columns if col.startswith('LLM_task2_')]
few_shot_cols = [col for col in all_model_cols if 'few_shot' in col]
standard_cols = [col for col in all_model_cols if 'few_shot' not in col]

results = []

def get_metrics(y_true, y_pred, model_name, dim_name, gt_type):
    return {
        'Model': model_name,
        'Dimension': dim_name,
        'GT_Type': gt_type,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

def evaluate_model_on_dims(df, dimensions, model_col):
    model_name = model_col.replace('LLM_task2_', '')
    dim_results = []
    for dim in dimensions:
        y_pred = df[model_col].fillna('').apply(lambda x: 1 if dim.lower() in str(x).lower() else 0)

        maj_col = f"{dim}_majority"
        if maj_col in df.columns:
            y_true_majority = df[maj_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_majority, y_pred, model_name, dim, 'majority'))

        any_col = f"{dim}_any"
        if any_col in df.columns:
            y_true_any = df[any_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_any, y_pred, model_name, dim, 'any'))
    return dim_results

# 2. Filter evaluation_df based on instructions
trust_df = evaluation_df[evaluation_df['majority_Label'].isin(['Trust', 'Both'])].copy()
distrust_df = evaluation_df[evaluation_df['majority_Label'].isin(['Distrust', 'Both'])].copy()

# 3. Run evaluation for all models
for model in all_model_cols:
    results.extend(evaluate_model_on_dims(trust_df, trust_dimensions, model))
    results.extend(evaluate_model_on_dims(distrust_df, distrust_dimensions, model))

# 4. Create DataFrame and Calculate Averages
performance_df = pd.DataFrame(results)


avg_perf = performance_df.groupby(['Model', 'GT_Type'])[['Accuracy', 'Precision', 'Recall', 'F1']].mean().reset_index()
avg_perf['Dimension'] = 'AVERAGE_ALL'
final_report = pd.concat([performance_df, avg_perf], ignore_index=True)

# 5. Pivot for F1 and handle ordering
# 5. Restructure for a narrower, longer table
# Apply name replacements to the Model column first
final_report['Model_Clean'] = final_report['Model'].replace(name_replacements)

# Create the Shot_Type column based on the model name suffix
final_report['Shot_Type'] = final_report['Model_Clean'].apply(
    lambda x: 'few_shot' if '_few_shot' in x else 'zero_shot'
)

# Clean the model names to remove the suffix so they align in the same column
final_report['Model_Final'] = final_report['Model_Clean'].str.replace('_few_shot', '')

# Pivot: Model names become columns, Shot_Type becomes part of the row index
f1_pivot = final_report.pivot_table(
    index=['GT_Type', 'Dimension', 'Shot_Type'],
    columns='Model_Final',
    values='F1'
)

# Handle Ordering
# Reindex GT_Type and Dimensions (same as before)
all_dims = sorted([d for d in final_report['Dimension'].unique() if d != 'AVERAGE_ALL'])
final_dim_order = all_dims + ['AVERAGE_ALL']

# Reindex levels: GT_Type, then Dimension, then Shot_Type (zero first, then few)
f1_pivot = f1_pivot.reindex(['majority', 'any'], level=0) \
                   .reindex(final_dim_order, level=1) \
                   .reindex(['zero_shot', 'few_shot'], level=2)

# 6. Styling with thick lines
# 6. Styling the narrow table
f1_flat = f1_pivot.reset_index()

# Find indices for horizontal dividers (every 2 rows for each Dimension)
# and the big divider between 'majority' and 'any'
split_any_index = len(f1_flat[f1_flat['GT_Type'] == 'majority'])

styled_f1_table = f1_flat.style.highlight_max(axis=1, color='lightgreen', subset=f1_pivot.columns) \
    .format("{:.2f}", subset=f1_pivot.columns) \
    .hide(axis='index') \
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#fff'), ('font-weight', 'bold'), ('border', '1px solid #ccc'), ('text-align', 'center')]},
        # Thick line between 'majority' and 'any'
        {'selector': f'tr:nth-child({split_any_index + 1}) td', 'props': [('border-top', '3px solid black')]},
        # Subtle lines between each Dimension pair for readability
        {'selector': 'tr:nth-child(2n) td', 'props': [('border-bottom', '1px solid #eee')]}
    ])

styled_f1_table

# Task 2: Stable Models

In [ ]:
# Purpose: Define the Task 2 prompt for extracting trust/distrust dimensions.
# 1. Define models that should NOT have a temperature parameter

models_to_run = ['gpt-5.2-2025-12-11', 'gpt-5.1-2025-11-13', 'gpt-5-2025-08-07', 'gpt-5-mini-2025-08-07', 'gpt-5-nano-2025-08-07', 'gpt-4o-2024-11-20', 'gpt-4o-mini-2024-07-18', 'gpt-4.1-2025-04-14', 'gpt-4.1-mini-2025-04-14', 'gpt-4.1-nano-2025-04-14']

no_temp_models = ['gpt-5-2025-08-07', 'gpt-5-mini-2025-08-07', 'gpt-5-nano-2025-08-07']

def classify_task2(text, task1_label, Model, use_temperature=True):
    openai_client = client
    retries = 2
    task_2_prompt = build_task2_prompt(task1_label)

    while retries > 0:
        try:
            # Explicit calls to avoid **kwargs
            if use_temperature:
                completion = openai_client.chat.completions.create(
                    model=Model,
                    messages=[{"role": "user", "content": task_2_prompt + text}],
                    temperature=0,
                    stream=False
                )
            else:
                completion = openai_client.chat.completions.create(
                    model=Model,
                    messages=[{"role": "user", "content": task_2_prompt + text}],
                    stream=False
                )
            return completion.choices[0].message.content

        except Exception as e:
            print(f"Error with {Model}: {e}. Retrying...")
            retries -= 1
            time.sleep(5)
    return "API error"

tqdm.pandas()

for model in models_to_run:
    print(f"Starting Task 2 classification for model: {model}")
    column_name = f"LLM_task2_{model}"

    # DECISION MADE ONCE PER MODEL
    is_no_temp_model = model in no_temp_models
    use_temp = not is_no_temp_model

    # Apply to dataframe using a direct lambda with the pre-determined 'use_temp'
    data_dimensions[column_name] = data_dimensions.progress_apply(
        lambda row: classify_task2(
            text=row['text'],
            task1_label=row['majority_Label'],
            Model=model,
            use_temperature=use_temp
        ),
        axis=1
    )

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# Identify all Task 2 columns
task2_cols = [col for col in data_dimensions.columns if col.startswith('LLM_task2_')]

# Remove '[' and ']' from those columns
for col in task2_cols:
    data_dimensions[col] = data_dimensions[col].str.replace(r'[\[\]]', '', regex=True)

# Display the head to verify
data_dimensions.head()

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
# Identify all Task 2 columns
task2_cols = [col for col in data_dimensions.columns if col.startswith('LLM_task2_')]

# Remove '[' and ']' from those columns
for col in task2_cols:
    data_dimensions[col] = data_dimensions[col].str.replace(r'[\[\]]', '', regex=True)

# 1. Define the name shortening mapping
name_replacements = {
    'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': 'Llama-4',
    'mistralai/Mixtral-8x7B-Instruct-v0.1': 'Mixtral-8x7B',
    'openai/gpt-oss-20b': 'gpt-oss-20b'
}


import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Setup dimensions and model columns
trust_dimensions = ['Competence', 'Familiarity', 'Integrity', 'Reliability', 'Transparency', 'Benevolence']
distrust_dimensions = ['Deception', 'Dishonesty', 'Incompetence', 'Malevolence', 'Opaqueness', 'Unreliability', 'Unfamiliarity']
model_cols = [col for col in data_dimensions.columns if col.startswith('LLM_task2_')]

results = []

def get_metrics(y_true, y_pred, model_name, dim_name, gt_type):
    return {
        'Model': model_name,
        'Dimension': dim_name,
        'GT_Type': gt_type,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

def evaluate_model_on_dims(df, dimensions, model_col):
    model_name = model_col.replace('LLM_task2_', '')
    dim_results = []

    for dim in dimensions:
        # Binary Prediction: 1 if dimension name is in the LLM output string
        y_pred = df[model_col].fillna('').apply(lambda x: 1 if dim.lower() in str(x).lower() else 0)

        # Ground Truth 1: {dimension}_majority
        maj_col = f"{dim}_majority"
        if maj_col in df.columns:
            y_true_majority = df[maj_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_majority, y_pred, model_name, dim, 'majority'))

        # Ground Truth 2: {dimension}_any
        any_col = f"{dim}_any"
        if any_col in df.columns:
            y_true_any = df[any_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_any, y_pred, model_name, dim, 'any'))

    return dim_results

# 2. Filter dataframes based on instructions
trust_df = data_dimensions[data_dimensions['majority_Label'].isin(['Trust', 'Both'])].copy()
distrust_df = data_dimensions[data_dimensions['majority_Label'].isin(['Distrust', 'Both'])].copy()

# 3. Run evaluation
for model in model_cols:
    results.extend(evaluate_model_on_dims(trust_df, trust_dimensions, model))
    results.extend(evaluate_model_on_dims(distrust_df, distrust_dimensions, model))

# 4. Create DataFrame
performance_df = pd.DataFrame(results)

performance_df['Model'] = performance_df['Model'].replace(name_replacements)

# 5. Calculate Averages separately for 'majority' and 'any'
# By grouping by both 'Model' and 'GT_Type', the averages for 'majority'
# and 'any' are never mixed.
avg_perf = performance_df.groupby(['Model', 'GT_Type'])[['Accuracy', 'Precision', 'Recall', 'F1']].mean().reset_index()
avg_perf['Dimension'] = 'AVERAGE_ALL'

# Combine individual and average results
final_report = pd.concat([performance_df, avg_perf], ignore_index=True)

# Sort so that Majority and Any results are grouped separately within each model
final_report = final_report.sort_values(by=['Model', 'GT_Type', 'Dimension'])

# 1. Pivot the table for F1 scores
f1_pivot = final_report.pivot_table(
    index=['GT_Type', 'Dimension'], # Swap index order to sort by GT_Type first
    columns='Model',
    values='F1'
)

# 2. Define the order: 'majority' first, then 'any'
# Within each group, keep your custom dimension order (AVERAGE_ALL at bottom)
all_dims = sorted([d for d in final_report['Dimension'].unique() if d != 'AVERAGE_ALL'])
final_dim_order = all_dims + ['AVERAGE_ALL']

# Reindex the levels
f1_pivot = f1_pivot.reindex(['majority', 'any'], level=0)
f1_pivot = f1_pivot.reindex(final_dim_order, level=1)

# 3. Flatten for display
f1_flat = f1_pivot.reset_index()

# 4. Find the index where 'any' starts to place the thick line
# Since 'majority' comes first, the line goes above the first 'any' row
split_index = len(f1_flat[f1_flat['GT_Type'] == 'majority'])

# 5. Apply styling
styled_f1_table = f1_flat.style.highlight_max(axis=1, color='lightgreen', subset=f1_pivot.columns) \
    .format("{:.2f}", subset=f1_pivot.columns) \
    .hide(axis='index') \
    .set_caption("Model Comparison: F1 Scores (Grouped by GT_Type)") \
    .set_table_styles([
        # Standard header styling
        {'selector': 'th', 'props': [('background-color', '#ffffff'), ('color', 'black'), ('font-weight', 'bold'), ('border', '1px solid #ccc')]},
        {'selector': 'td', 'props': [('border', '1px solid #eee')]},
        # The thick line: apply a top border to all cells in the first row of the 'any' group
        {'selector': f'tr:nth-child({split_index + 1}) td', 'props': [('border-top', '3px solid black')]}
    ])

# Display the table
styled_f1_table

# Task 2: Few Shot

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
data_dimensions.head()

In [ ]:
# Purpose: Save the current processed dataframe or analysis output for reuse/reproducibility.
import time
from tqdm import tqdm
from openai import OpenAI

# 1. Define your list of models
models_to_run = ['gpt-5-mini-2025-08-07', 'gpt-5-nano-2025-08-07', 'gpt-4o-2024-11-20', 'gpt-4o-mini-2024-07-18', 'gpt-4.1-2025-04-14', 'gpt-4.1-mini-2025-04-14', 'gpt-4.1-nano-2025-04-14']

# Pre-generate the three possible prompt versions
prompts = {
    label: build_task2_prompt_shots(label, data_dimensions)
    for label in ["Trust", "Distrust", "Both"]
}


def classify_gpt_models_task2_few_shots(text, task1_label, Model):
    retries = 2

    # Get the correct few-shot prompt block for this label
    # Ensure build_task2_prompt_shots is defined and accepts data_dimensions if needed
    task_2_prompt_few_shots = prompts.get(task1_label, "")

    while retries > 0:
        try:
            # Prepare arguments for the API call
            api_args = {
                "model": Model,
                "messages": [{"role": "user", "content": task_2_prompt_few_shots + text}],
                "stream": False
            }

            # Condition: Set temperature=0 for models NOT in the gpt-5/mini/nano list
            # For 'gpt-5', 'gpt-5-mini', and 'gpt-5-nano', the parameter is omitted
            if Model not in ['gpt-5-mini-2025-08-07', 'gpt-5-nano-2025-08-07']:
                api_args["temperature"] = 0

            completion = client.chat.completions.create(**api_args)
            return completion.choices[0].message.content

        except Exception as e:
            print(e)
            print('Timeout error, retrying...')
            retries -= 1
            time.sleep(5)

    return "API error"

# ---- Run over your evaluation_df ----
tqdm.pandas()

# 2. Iterate through each model in the list
for model in models_to_run:
    print(f"Starting Task 2 classification for model: {model}")

    column_name = f"LLM_task2_{model}_few_shot"

    # Check if this model has already been run (to allow resuming if interrupted)
    if column_name in evaluation_df.columns:
        print(f"Column {column_name} already exists. Skipping...")
        continue

    # Apply the classification function
    evaluation_df[column_name] = evaluation_df.progress_apply(
        lambda row: classify_gpt_models_task2_few_shots(
            text=row['text'],
            task1_label=row['majority_Label'],
            Model=model
        ),
        axis=1
    )

    # NEW: Immediately save to CSV after each model finishes
    # This ensures that even if the next model fails, you have these results stored.
    checkpoint_filename = "evaluation_task2_checkpoints_1.csv"
    evaluation_df.to_csv(checkpoint_filename, index=False)
    print(f"Results for {model} saved to {checkpoint_filename}.")

print("Classification complete for all models.")

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Setup dimensions and model columns using evaluation_df
trust_dimensions = ['Competence', 'Familiarity', 'Integrity', 'Reliability', 'Transparency', 'Benevolence']
distrust_dimensions = ['Deception', 'Dishonesty', 'Incompetence', 'Malevolence', 'Opaqueness', 'Unreliability', 'Unfamiliarity']


# 1. Define the name shortening mapping
name_replacements = {
    'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': 'Llama-4',
    'mistralai/Mixtral-8x7B-Instruct-v0.1': 'Mixtral-8x7B',
    'openai/gpt-oss-20b': 'gpt-oss-20b'
}


# Separate standard columns from few_shot columns
all_model_cols = [col for col in evaluation_df.columns if col.startswith('LLM_task2_')]
few_shot_cols = [col for col in all_model_cols if 'few_shot' in col]
standard_cols = [col for col in all_model_cols if 'few_shot' not in col]

results = []

def get_metrics(y_true, y_pred, model_name, dim_name, gt_type):
    return {
        'Model': model_name,
        'Dimension': dim_name,
        'GT_Type': gt_type,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

def evaluate_model_on_dims(df, dimensions, model_col):
    model_name = model_col.replace('LLM_task2_', '')
    dim_results = []
    for dim in dimensions:
        y_pred = df[model_col].fillna('').apply(lambda x: 1 if dim.lower() in str(x).lower() else 0)

        maj_col = f"{dim}_majority"
        if maj_col in df.columns:
            y_true_majority = df[maj_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_majority, y_pred, model_name, dim, 'majority'))

        any_col = f"{dim}_any"
        if any_col in df.columns:
            y_true_any = df[any_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_any, y_pred, model_name, dim, 'any'))
    return dim_results

# 2. Filter evaluation_df based on instructions
trust_df = evaluation_df[evaluation_df['majority_Label'].isin(['Trust', 'Both'])].copy()
distrust_df = evaluation_df[evaluation_df['majority_Label'].isin(['Distrust', 'Both'])].copy()

# 3. Run evaluation for all models
for model in all_model_cols:
    results.extend(evaluate_model_on_dims(trust_df, trust_dimensions, model))
    results.extend(evaluate_model_on_dims(distrust_df, distrust_dimensions, model))

# 4. Create DataFrame and Calculate Averages
performance_df = pd.DataFrame(results)


avg_perf = performance_df.groupby(['Model', 'GT_Type'])[['Accuracy', 'Precision', 'Recall', 'F1']].mean().reset_index()
avg_perf['Dimension'] = 'AVERAGE_ALL'
final_report = pd.concat([performance_df, avg_perf], ignore_index=True)

# 5. Pivot for F1 and handle ordering
# 5. Restructure for a narrower, longer table
# Apply name replacements to the Model column first
final_report['Model_Clean'] = final_report['Model'].replace(name_replacements)

# Create the Shot_Type column based on the model name suffix
final_report['Shot_Type'] = final_report['Model_Clean'].apply(
    lambda x: 'few_shot' if '_few_shot' in x else 'zero_shot'
)

# Clean the model names to remove the suffix so they align in the same column
final_report['Model_Final'] = final_report['Model_Clean'].str.replace('_few_shot', '')

# Pivot: Model names become columns, Shot_Type becomes part of the row index
f1_pivot = final_report.pivot_table(
    index=['GT_Type', 'Dimension', 'Shot_Type'],
    columns='Model_Final',
    values='F1'
)

# Handle Ordering
# Reindex GT_Type and Dimensions (same as before)
all_dims = sorted([d for d in final_report['Dimension'].unique() if d != 'AVERAGE_ALL'])
final_dim_order = all_dims + ['AVERAGE_ALL']

# Reindex levels: GT_Type, then Dimension, then Shot_Type (zero first, then few)
f1_pivot = f1_pivot.reindex(['majority', 'any'], level=0) \
                   .reindex(final_dim_order, level=1) \
                   .reindex(['zero_shot', 'few_shot'], level=2)

# 6. Styling with thick lines
# 6. Styling the narrow table
f1_flat = f1_pivot.reset_index()

# Find indices for horizontal dividers (every 2 rows for each Dimension)
# and the big divider between 'majority' and 'any'
split_any_index = len(f1_flat[f1_flat['GT_Type'] == 'majority'])

styled_f1_table = f1_flat.style.highlight_max(axis=1, color='lightgreen', subset=f1_pivot.columns) \
    .format("{:.2f}", subset=f1_pivot.columns) \
    .hide(axis='index') \
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#fff'), ('font-weight', 'bold'), ('border', '1px solid #ccc'), ('text-align', 'center')]},
        # Thick line between 'majority' and 'any'
        {'selector': f'tr:nth-child({split_any_index + 1}) td', 'props': [('border-top', '3px solid black')]},
        # Subtle lines between each Dimension pair for readability
        {'selector': 'tr:nth-child(2n) td', 'props': [('border-bottom', '1px solid #eee')]}
    ])

styled_f1_table

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Setup dimensions and model columns using evaluation_df
trust_dimensions = ['Competence', 'Familiarity', 'Integrity', 'Reliability', 'Transparency', 'Benevolence']
distrust_dimensions = ['Deception', 'Dishonesty', 'Incompetence', 'Malevolence', 'Opaqueness', 'Unreliability', 'Unfamiliarity']


# 1. Define the name shortening mapping
name_replacements = {
    'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': 'Llama-4',
    'mistralai/Mixtral-8x7B-Instruct-v0.1': 'Mixtral-8x7B',
    'openai/gpt-oss-20b': 'gpt-oss-20b'
}


# Separate standard columns from few_shot columns
all_model_cols = [col for col in evaluation_df.columns if col.startswith('LLM_task2_')]
few_shot_cols = [col for col in all_model_cols if 'few_shot' in col]
standard_cols = [col for col in all_model_cols if 'few_shot' not in col]

results = []

def get_metrics(y_true, y_pred, model_name, dim_name, gt_type):
    return {
        'Model': model_name,
        'Dimension': dim_name,
        'GT_Type': gt_type,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

def evaluate_model_on_dims(df, dimensions, model_col):
    model_name = model_col.replace('LLM_task2_', '')
    dim_results = []
    for dim in dimensions:
        y_pred = df[model_col].fillna('').apply(lambda x: 1 if dim.lower() in str(x).lower() else 0)

        maj_col = f"{dim}_majority"
        if maj_col in df.columns:
            y_true_majority = df[maj_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_majority, y_pred, model_name, dim, 'majority'))

        any_col = f"{dim}_any"
        if any_col in df.columns:
            y_true_any = df[any_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_any, y_pred, model_name, dim, 'any'))
    return dim_results

# 2. Filter evaluation_df based on instructions
trust_df = evaluation_df[evaluation_df['majority_Label'].isin(['Trust', 'Both'])].copy()
distrust_df = evaluation_df[evaluation_df['majority_Label'].isin(['Distrust', 'Both'])].copy()

# 3. Run evaluation for all models
for model in all_model_cols:
    results.extend(evaluate_model_on_dims(trust_df, trust_dimensions, model))
    results.extend(evaluate_model_on_dims(distrust_df, distrust_dimensions, model))

# 4. Create DataFrame and Calculate Averages
performance_df = pd.DataFrame(results)


avg_perf = performance_df.groupby(['Model', 'GT_Type'])[['Accuracy', 'Precision', 'Recall', 'F1']].mean().reset_index()
avg_perf['Dimension'] = 'AVERAGE_ALL'
final_report = pd.concat([performance_df, avg_perf], ignore_index=True)

# 5. Pivot for F1 and handle ordering
# 5. Restructure for a narrower, longer table
# Apply name replacements to the Model column first
final_report['Model_Clean'] = final_report['Model'].replace(name_replacements)

# Create the Shot_Type column based on the model name suffix
final_report['Shot_Type'] = final_report['Model_Clean'].apply(
    lambda x: 'few_shot' if '_few_shot' in x else 'zero_shot'
)

# Clean the model names to remove the suffix so they align in the same column
final_report['Model_Final'] = final_report['Model_Clean'].str.replace('_few_shot', '')

# Pivot: Model names become columns, Shot_Type becomes part of the row index
# Pivot: Shot_Type becomes a column level along with Model names
f1_pivot = final_report.pivot_table(
    index=['GT_Type', 'Dimension'],
    columns=['Model_Final', 'Shot_Type'],
    values='F1'
)

# Sort the columns so all zero_shot are together on the left
# and few_shot are together on the right
f1_pivot = f1_pivot.reindex(['zero_shot', 'few_shot'], axis=1, level=1)

# Reindex GT_Type and Dimensions (same as before)
all_dims = sorted([d for d in final_report['Dimension'].unique() if d != 'AVERAGE_ALL'])
final_dim_order = all_dims + ['AVERAGE_ALL']

f1_pivot = f1_pivot.reindex(['majority', 'any'], level=0) \
                   .reindex(final_dim_order, level=1)

# Flatten for styling but keep the column multiindex structure
# (or use the pivot directly if preferred)
styled_f1_table = f1_pivot.style.highlight_max(axis=1, color='lightgreen') \
    .format("{:.2f}")

# Calculate where the vertical line should go
# Since we sorted by shot_type, we find the first 'few_shot' column
col_names = f1_pivot.columns.get_level_values(1)
first_few_shot_idx = list(col_names).index('few_shot')

# Find the row index for the big horizontal divider
split_any_index = len(f1_pivot.loc['majority'])

styled_f1_table.set_table_styles([
    # Standard header styling
    {'selector': 'th', 'props': [('background-color', '#fff'), ('font-weight', 'bold'),
                                 ('border', '1px solid #ccc'), ('text-align', 'center')]},

    # THICK VERTICAL LINE: Between zero_shot and few_shot
    # We apply it to the left border of the first few_shot column
    {'selector': f'td:nth-child({first_few_shot_idx + 3})', # +3 to account for GT_Type/Dim index columns
     'props': [('border-left', '3px solid black')]},
    {'selector': f'th.col_heading.level0:nth-child({first_few_shot_idx + 1})',
     'props': [('border-left', '3px solid black')]},

    # THICK HORIZONTAL LINE: Between 'majority' and 'any'
    {'selector': f'tr:nth-child({split_any_index + 1}) td',
     'props': [('border-top', '3px solid black')]},

    # Subtle lines for readability
    {'selector': 'tr:nth-child(n) td', 'props': [('border-bottom', '1px solid #eee')]}
])

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Setup dimensions and model columns using evaluation_df
trust_dimensions = ['Competence', 'Familiarity', 'Integrity', 'Reliability', 'Transparency', 'Benevolence']
distrust_dimensions = ['Deception', 'Dishonesty', 'Incompetence', 'Malevolence', 'Opaqueness', 'Unreliability', 'Unfamiliarity']


# 1. Define the name shortening mapping
name_replacements = {
    'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': 'Llama-4',
    'mistralai/Mixtral-8x7B-Instruct-v0.1': 'Mixtral-8x7B',
    'openai/gpt-oss-20b': 'gpt-oss-20b'
}


# Separate standard columns from few_shot columns
all_model_cols = [col for col in evaluation_df.columns if col.startswith('LLM_task2_')]
few_shot_cols = [col for col in all_model_cols if 'few_shot' in col]
standard_cols = [col for col in all_model_cols if 'few_shot' not in col]

results = []

def get_metrics(y_true, y_pred, model_name, dim_name, gt_type):
    return {
        'Model': model_name,
        'Dimension': dim_name,
        'GT_Type': gt_type,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

def evaluate_model_on_dims(df, dimensions, model_col):
    model_name = model_col.replace('LLM_task2_', '')
    dim_results = []
    for dim in dimensions:
        y_pred = df[model_col].fillna('').apply(lambda x: 1 if dim.lower() in str(x).lower() else 0)

        maj_col = f"{dim}_majority"
        if maj_col in df.columns:
            y_true_majority = df[maj_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_majority, y_pred, model_name, dim, 'majority'))

        any_col = f"{dim}_any"
        if any_col in df.columns:
            y_true_any = df[any_col].fillna(0).apply(lambda x: 1 if (str(x).strip() != '0' and str(x).strip() != '') else 0)
            dim_results.append(get_metrics(y_true_any, y_pred, model_name, dim, 'any'))
    return dim_results

# 2. Filter evaluation_df based on instructions
trust_df = evaluation_df[evaluation_df['majority_Label'].isin(['Trust', 'Both'])].copy()
distrust_df = evaluation_df[evaluation_df['majority_Label'].isin(['Distrust', 'Both'])].copy()

# 3. Run evaluation for all models
for model in all_model_cols:
    results.extend(evaluate_model_on_dims(trust_df, trust_dimensions, model))
    results.extend(evaluate_model_on_dims(distrust_df, distrust_dimensions, model))

# 4. Create DataFrame and Calculate Averages
performance_df = pd.DataFrame(results)


avg_perf = performance_df.groupby(['Model', 'GT_Type'])[['Accuracy', 'Precision', 'Recall', 'F1']].mean().reset_index()
avg_perf['Dimension'] = 'AVERAGE_ALL'
final_report = pd.concat([performance_df, avg_perf], ignore_index=True)

# 5. Pivot for F1 and handle ordering
# 5. Restructure for a narrower, longer table
# Apply name replacements to the Model column first
final_report['Model_Clean'] = final_report['Model'].replace(name_replacements)

# Create the Shot_Type column based on the model name suffix
final_report['Shot_Type'] = final_report['Model_Clean'].apply(
    lambda x: 'few_shot' if '_few_shot' in x else 'zero_shot'
)

# Clean the model names to remove the suffix so they align in the same column
final_report['Model_Final'] = final_report['Model_Clean'].str.replace('_few_shot', '')

# Pivot: Model names become columns, Shot_Type becomes part of the row index
# --- STEP 5: PIVOTING & STRUCTURING ---

# 1. Standard Pivot (F1 is the value, Models and Shot_Type are columns)
f1_pivot = final_report.pivot_table(
    index=['GT_Type', 'Dimension'],
    columns=['Model_Final', 'Shot_Type'],
    values='F1'
)

# 2. Force every model to have both 'zero_shot' and 'few_shot' columns
unique_models = final_report['Model_Final'].unique()
shot_types = ['zero_shot', 'few_shot']

# Create the full combination of Model + Shot Type
full_col_index = pd.MultiIndex.from_product(
    [unique_models, shot_types],
    names=['Model_Final', 'Shot_Type']
)

# Reindex the columns to insert missing 'few_shot' or 'zero_shot' as NaN
f1_pivot = f1_pivot.reindex(columns=full_col_index)

# 3. Reindex Rows (GT_Type and Dimensions)
all_dims = sorted([d for d in final_report['Dimension'].unique() if d != 'AVERAGE_ALL'])
final_dim_order = all_dims + ['AVERAGE_ALL']

f1_pivot = f1_pivot.reindex(['majority', 'any'], level=0) \
                   .reindex(final_dim_order, level=1)


# --- STEP 6: STYLING ---

# Identify the row index where 'majority' ends and 'any' begins for the horizontal line
split_any_index = len(f1_pivot.loc['majority'])

# Apply styles
styled_f1_table = f1_pivot.style \
    .highlight_max(axis=1, color='lightgreen') \
    .format("{:.2f}", na_rep="-") \
    .set_table_styles([
        # Header Styling
        {'selector': 'th', 'props': [
            ('background-color', '#f8f9fa'),
            ('color', '#333'),
            ('font-weight', 'bold'),
            ('border', '1px solid #ccc'),
            ('text-align', 'center')
        ]},

        # THICK VERTICAL LINES: After every few_shot column (every 2nd data column)
        # We target the right border of the 'few_shot' column in each pair
        {'selector': 'td:nth-child(2n+2)',
         'props': [('border-right', '3px solid black')]},
        {'selector': 'th.col_heading.level1:nth-child(2n)',
         'props': [('border-right', '3px solid black')]},
        {'selector': 'th.col_heading.level0:nth-child(n)',
         'props': [('border-right', '3px solid black')]},

        # THICK HORIZONTAL LINE: Between 'majority' and 'any'
        {'selector': f'tr:nth-child({split_any_index + 1}) td',
         'props': [('border-top', '3px solid black')]},

        # Subtle row borders for readability
        {'selector': 'td', 'props': [('border', '1px solid #eee')]}
    ])

# Display the table
styled_f1_table

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
import pandas as pd

# Define the model to inspect
target_model = 'gpt-4.1-nano-2025-04-14'

# 1. Filter the report for the specific model
model_subset = final_report[final_report['Model_Final'] == target_model].copy()

# 2. Reshape the data to include Precision, Recall, and F1
metrics_melted = model_subset.melt(
    id_vars=['GT_Type', 'Dimension', 'Shot_Type'],
    value_vars=['Precision', 'Recall', 'F1'], # Added F1 here
    var_name='Metric',
    value_name='Value'
)

# 3. Create the Pivot Table
metric_pivot = metrics_melted.pivot_table(
    index=['GT_Type', 'Dimension'],
    columns=['Shot_Type', 'Metric'],
    values='Value'
)

# 4. Reorder columns to be [Precision, Recall, F1] for each shot type
metric_pivot = metric_pivot.reindex(['Precision', 'Recall', 'F1'], axis=1, level=1)

# 5. Apply consistent row ordering (Majority first, then Any)
all_dims = sorted([d for d in model_subset['Dimension'].unique() if d != 'AVERAGE_ALL'])
final_dim_order = all_dims + ['AVERAGE_ALL']

metric_pivot = metric_pivot.reindex(['majority', 'any'], level=0).reindex(final_dim_order, level=1)

# 6. Styling
styled_metric_table = metric_pivot.style \
    .format("{:.2f}") \
    .set_caption(f"Detailed Metrics (P/R/F1): {target_model}") \
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#f8f9fa'), ('font-weight', 'bold'), ('border', '1px solid #ccc')]},
        {'selector': 'tr:nth-child(14) td', 'props': [('border-bottom', '3px solid black')]}
    ])

# Display the table
styled_metric_table

## 4. Annotation agreement, dimension prevalence, and performance relationships

Computes agreement/ratio summaries for annotation dimensions and examines whether dimension prevalence is associated with model F1 scores.

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# 1. Setup dimensions and model columns using evaluation_df
trust_dimensions = ['Competence', 'Familiarity', 'Integrity', 'Reliability', 'Transparency', 'Benevolence']
distrust_dimensions = ['Deception', 'Dishonesty', 'Incompetence', 'Malevolence', 'Opaqueness', 'Unreliability', 'Unfamiliarity']

trust_df = final_GT_data[final_GT_data['majority_Label'].isin(['Trust', 'Both'])]
distrust_df = final_GT_data[final_GT_data['majority_Label'].isin(['Distrust', 'Both'])]

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
def report_dimension_stats(df, dimensions, title):
    print(f"--- {title} ---")
    results = []
    for dim in dimensions:
        col = f"{dim}_ratio"
        if col in df.columns:
            # Convert "x/y" strings to actual float values
            numeric_vals = df[col].apply(lambda x: eval(str(x)) if '/' in str(x) else pd.to_numeric(x, errors='coerce'))
            mean_val = numeric_vals.mean()
            std_val = numeric_vals.std()
            results.append({
                'Dimension': dim,
                'Mean': mean_val,
                'Std': std_val,
                'Range': (round(mean_val - std_val, 4), round(mean_val + std_val, 4))
            })

    # Create the DataFrame
    results_df = pd.DataFrame(results).set_index('Dimension')
    print(results_df)
    print("\n")

    # THIS IS THE MISSING PART:
    return results_df

# Now run them and assign to variables
trust_results = report_dimension_stats(trust_df, trust_dimensions, "Trust Group Stats")
distrust_results = report_dimension_stats(distrust_df, distrust_dimensions, "Distrust Group Stats")

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
dims= ['Benevolence','Competence','Deception','Dishonesty','Familiarity','Incompetence','Integrity',
       'Malevolence','Opaqueness','Reliability','Transparency','Unfamiliarity','Unreliability']

F1_scores_Any= [57,97,59,52,77,87,26,60,57,74,46,35, 93]
F1_scores_Majority= [45,89,49,51,34,76,29,59,39,69,24,19,80]
# Create the dictionary using zip
f1_dict_Any = dict(zip(dims, F1_scores_Any))
f1_dict_Majority = dict(zip(dims, F1_scores_Majority))

# View the result
print(f1_dict_Any), print(f1_dict_Majority)

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
# 1. Combine the Mean columns from both result sets
comparison_df = pd.concat([
    trust_results[['Mean']],
    distrust_results[['Mean']]
])

# 2. Add the F1 scores using your dictionary
comparison_df['F1_Score_Any'] = comparison_df.index.map(f1_dict_Any)
comparison_df['F1_Score_Majority'] = comparison_df.index.map(f1_dict_Majority)

# View the result
comparison_df

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
correlation = comparison_df['Mean'].corr(comparison_df['F1_Score_Any'])
print(f"Correlation: {correlation:.4f}")

from scipy import stats

# Calculate Pearson correlation and p-value
r_coeff, p_value = stats.pearsonr(comparison_df['Mean'], comparison_df['F1_Score_Any'])

print(f"Pearson Correlation Coefficient (r): {r_coeff:.4f}")
print(f"P-value: {p_value:.6f}")

# Interpretation
alpha = 0.05
if p_value < alpha:
    print(f"Result: Statistically Significant (p < {alpha})")
else:
    print(f"Result: Not Statistically Significant (p >= {alpha})")

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
correlation = comparison_df['Mean'].corr(comparison_df['F1_Score_Majority'])
print(f"Correlation: {correlation:.4f}")

from scipy import stats

# Calculate Pearson correlation and p-value
r_coeff, p_value = stats.pearsonr(comparison_df['Mean'], comparison_df['F1_Score_Majority'])

print(f"Pearson Correlation Coefficient (r): {r_coeff:.4f}")
print(f"P-value: {p_value:.6f}")

# Interpretation
alpha = 0.05
if p_value < alpha:
    print(f"Result: Statistically Significant (p < {alpha})")
else:
    print(f"Result: Not Statistically Significant (p >= {alpha})")

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
from scipy import stats

# Perform linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(comparison_df['Mean'], comparison_df['F1_Score_Majority'])

print(f"Slope: {slope:.4f}")
print(f"Intercept: {intercept:.4f}")
print(f"R-squared: {r_value**2:.4f}")
print(f"P-value: {p_value:.6f}")

# Predict a value
# example_prediction = intercept + slope * some_mean_value

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
from scipy import stats

# Perform linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(comparison_df['Mean'], comparison_df['F1_Score_Any'])

print(f"Slope: {slope:.4f}")
print(f"Intercept: {intercept:.4f}")
print(f"R-squared: {r_value**2:.4f}")
print(f"P-value: {p_value:.6f}")

# Predict a value
# example_prediction = intercept + slope * some_mean_value

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
r_spearman, p_spearman = stats.spearmanr(comparison_df['Mean'], comparison_df['F1_Score_Majority'])
print(f"Spearman Rank Correlation: {r_spearman:.4f}")

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
r_spearman, p_spearman = stats.spearmanr(comparison_df['Mean'], comparison_df['F1_Score_Any'])
print(f"Spearman Rank Correlation: {r_spearman:.4f}")

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
import matplotlib.pyplot as plt
import numpy as np

def create_individual_plot(x, y, title, color, filename):
    # Dimensions bigger: 12x9 for high-impact PDFs
    fig, ax = plt.subplots(figsize=(22, 15))

    m, b = np.polyfit(x, y, 1)

    # Points even larger for clarity
    ax.scatter(x, y, color=color, alpha=0.7, s=200, edgecolors='white')

    # Regression line
    ax.plot(x, m * x + b, color='black', linestyle='--', linewidth=2.5)

    # INCREASED: Annotation font size
    for i, txt in enumerate(comparison_df.index):
        ax.annotate(txt, (x.iloc[i], y.iloc[i]), xytext=(8, 8),
                    textcoords='offset points', fontsize=32)

    # INCREASED: Title and Axis Labels
    ax.set_title(title, fontsize=30, fontweight='bold', pad=20)
    #ax.set_xlabel('Aggregate Ratio (Mean)', fontsize=16, labelpad=10)
    #ax.set_ylabel('F1 Score', fontsize=16, labelpad=10)

    # INCREASED: Axis Tick Labels
    ax.tick_params(axis='both', which='major', labelsize=36)

    ax.grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout()
    plt.savefig(filename, format='pdf', bbox_inches='tight')

    # UPDATED: Show the plot in the notebook
    plt.show()
    print(f"Saved: {filename} and displayed plot.")

# Execute for Majority
create_individual_plot(
    comparison_df['Mean'],
    comparison_df['F1_Score_Majority'],
    'Mean vs F1 Majority',
    'royalblue',
    'f1_majority_large.pdf'
)

# Execute for Any
create_individual_plot(
    comparison_df['Mean'],
    comparison_df['F1_Score_Any'],
    'Mean vs F1 Any',
    'seagreen',
    'f1_any_large.pdf'
)

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import matplotlib.pyplot as plt

# 1. Separate 'majority' and 'any' data from your summary_table
majority_data = summary_table[summary_table.index.str.endswith('_majority')].copy()
any_data = summary_table[summary_table.index.str.endswith('_any')].copy()

# 2. Clean index names
majority_data.index = majority_data.index.str.replace('_majority', '')
any_data.index = any_data.index.str.replace('_any', '')

# 3. Sort values
majority_data = majority_data.sort_values(by='Total Count (Sum)', ascending=True)
any_data = any_data.sort_values(by='Total Count (Sum)', ascending=True)

# --- NEW: Calculate global max for equal X-axis range ---
max_x_value = max(majority_data['Total Count (Sum)'].max(), any_data['Total Count (Sum)'].max()) * 1.05

# 4. Plot Majority Approach
plt.figure(figsize=(12, 8))
plt.barh(majority_data.index, majority_data['Total Count (Sum)'], color='#2c3e50') # Dark bar
plt.xlim(0, max_x_value) # Set equal range

# Significantly increase axis font sizes and bold them
plt.xticks(fontsize=20, fontweight='bold')
plt.yticks(fontsize=24, fontweight='bold')

plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('Majority_Approach_Plot.pdf', format='pdf', bbox_inches='tight')
plt.show()

# 5. Plot Any Approach
plt.figure(figsize=(12, 8))
plt.barh(any_data.index, any_data['Total Count (Sum)'], color='#2c3e50') # Dark bar
plt.xlim(0, max_x_value) # Set equal range

# Significantly increase axis font sizes and bold them
plt.xticks(fontsize=20, fontweight='bold')
plt.yticks(fontsize=24, fontweight='bold')

plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('Any_Approach_Plot.pdf', format='pdf', bbox_inches='tight')
plt.show()

## 5. Full-corpus data preparation and Task 1 batch labeling

Combines full Reddit corpus files, preprocesses text, creates OpenAI Batch files, retrieves/stitches outputs, normalizes labels, and visualizes full-corpus Task 1 trends.

# Task 1 : Batch

In [ ]:
# Purpose: Load a saved dataset or cached model-output file used by later analysis cells.
import os
import pandas as pd

path1 = r'/Users/ap3943/Desktop/subreddits/Filtered Data'
path2 = r'/Users/ap3943/Desktop/subreddits/GenAI_filtered_posts'

all_data = pd.DataFrame()

# Iterate through both paths
for path in [path1, path2]:
    for files in os.listdir(path):
        if files != ".DS_Store":
            path_sub = os.path.join(path, files)
            data_sub = pd.read_csv(path_sub)
            all_data = pd.concat([all_data, data_sub])

all_data= all_data.reset_index(drop= True)

len(all_data)

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
all_data= all_data[~all_data['selftext'].isin([['removed'],['deleted']])]
all_data = all_data.dropna(subset=['selftext'])
# Remove duplicate rows
all_data = all_data.drop_duplicates()
len(all_data)

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
all_data['all_text']= all_data['title']+ ' '+ '**' + ' '+ all_data['selftext']
all_data['token_count'] = all_data['all_text'].str.split().str.len()
all_data= all_data[all_data['token_count']>5]
len(all_data)

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
batch_size = 10000
num_batches = len(all_data) // batch_size + int(len(all_data) % batch_size != 0)

batches = [all_data[i * batch_size:(i + 1) * batch_size] for i in range(num_batches)]
num_batches, len(batches)

In [ ]:
# Purpose: Define the Task 1 prompt for classifying trust, distrust, both, or neither.
import json
import math
import re


# ---- 1) Constants you define once ----
MODEL = "gpt-5.1"                 # or any chat model you want
SYSTEM_PROMPT = task_1_prompt_few_shot  # your constant prompt
TEXT_COL = "all_text"

# (optional) basic cleaner so inputs won't break batching
RESERVED = re.compile(r"<\|[^>]+?\|>")  # catches tokens like <|endoftext|>
def sanitize(s):
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = str(s).replace("\r\n", "\n").strip()
    s = RESERVED.sub("[RESERVED_TOKEN]", s)
    return s[:8000]  # keep inputs reasonable

# ---- 2) Writer: one JSON object per line ----
def write_ndjson_from_df(df, custom_id_col=None, max_tokens=1000, temperature=0):
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        for i, row in df.reset_index(drop=True).iterrows():
            custom_id = (
                str(row[custom_id_col])
                if custom_id_col and custom_id_col in df.columns
                else f"row-{i+1:06d}"
            )
            user_text = sanitize(row[TEXT_COL])

            obj = {
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": MODEL,
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": user_text}
                    ],
                    #"max_completion_tokens": max_tokens,
                    "temperature": 0
                }
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
### Batch 1

                     # dataframe column that holds the input text
OUT_PATH = "batch_1_input.ndjson"        # output file


write_ndjson_from_df(batches[0], custom_id_col=None)

# ---- 4) Quick validity check (no API call) ----
with open(OUT_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        json.loads(line)  # should not raise
print("NDJSON looks valid.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json

path = "batch_1_input.ndjson"

with open(path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        try:
            obj = json.loads(line)
        except Exception as e:
            raise ValueError(f"Line {i} is not valid JSON: {e}")

print("✅ All lines are valid JSON")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
batch_input_file = client.files.create(
    file=open("batch_1_input.ndjson", "rb"),
    purpose="batch"
)

print(batch_input_file)

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
batch_input_file_id = batch_input_file.id
client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "description": "classification of trust and distrust towards GenAI"
    }
)

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
batch = client.batches.retrieve("batch_698c10bc7bcc819085578c6872e57732")
print(batch.status)
print(batch)

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
out_file = client.files.content(batch.output_file_id)
with open("batch_1_output.ndjson", "wb") as f:
    f.write(out_file.content)

# Optional: check errors
if batch.error_file_id:
    err_file = client.files.content(batch.error_file_id)
    with open("batch_errors.ndjson", "wb") as f:
        f.write(err_file.content)



In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
output_values_batch_1= []

with open("batch_1_output.ndjson","r",encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        output_values_batch_1.append(json.loads(line)['response']['body']['choices'][0]['message']['content'])

len(output_values_batch_1)

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
batches[0]['output_values_batch']= output_values_batch_1
batches[0].head()

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
### Batch 1

                     # dataframe column that holds the input text
OUT_PATH = "batch_1_input.ndjson"        # output file


write_ndjson_from_df(batches[0], custom_id_col=None)

# ---- 4) Quick validity check (no API call) ----
with open(OUT_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        json.loads(line)  # should not raise
print("NDJSON looks valid.")

import json

path = "batch_1_input.ndjson"

with open(path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        try:
            obj = json.loads(line)
        except Exception as e:
            raise ValueError(f"Line {i} is not valid JSON: {e}")

print("✅ All lines are valid JSON")

batch_input_file = client.files.create(
    file=open("batch_1_input.ndjson", "rb"),
    purpose="batch"
)

print(batch_input_file)


batch_input_file_id = batch_input_file.id
client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "description": "classification of trust and distrust towards GenAI"
    }
)

batch = client.batches.retrieve("batch_69897484f21881908b1dd02689c48397")
print(batch.status)
print(batch)


output_values_batch_1= []

with open("batch_1_output.ndjson","r",encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        output_values_batch_1.append(json.loads(line)['response']['body']['choices'][0]['message']['content'])

len(output_values_batch_1)


batches[0]['output_values_batch']= output_values_batch_1
batches[0].head()

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import os
import json

submitted_batches = []

for i, batch_df in enumerate(batches):
    batch_num = i + 1
    input_filename = f"batch_{batch_num}_input.ndjson"

    print(f"--- Processing Batch {batch_num}/{len(batches)} ---")

    # Reset index and use it as the custom_id column
    temp_df = batch_df.copy()
    temp_df['row_id'] = temp_df.index.astype(str)


    # 1. Create the local file
    write_ndjson_from_df(temp_df, custom_id_col='row_id')

    # Check if the default file exists before renaming
    if os.path.exists("batch_1_input.ndjson"):
        os.rename("batch_1_input.ndjson", input_filename)
    else:
        raise FileNotFoundError(f"write_ndjson_from_df failed to create batch_1_input.ndjson for batch {batch_num}")

    # 2. Upload and 3. Create Batch Job (Your existing logic is correct)
    batch_input_file = client.files.create(
        file=open(input_filename, "rb"),
        purpose="batch"
    )

    batch_job = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={"description": f"classification - Batch {batch_num}"}
    )

    submitted_batches.append(batch_job)
    print(f"Submitted Batch ID: {batch_job.id}")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json
import pandas as pd

for i, batch_job in enumerate(submitted_batches):
    batch_num = i + 1
    current_id = batch_job.id

    print(f"Checking Batch {batch_num} ({current_id})...")

    # Refresh the batch status
    status_check = client.batches.retrieve(current_id)

    if status_check.status == 'completed':
        output_filename = f"batch_{batch_num}_output.ndjson"

        # 1. Download the output file
        file_id = status_check.output_file_id
        content = client.files.content(file_id).content
        with open(output_filename, "wb") as f:[]
            f.write(content)

        # 2. Map results using custom_id to handle missing/failed rows
        results_map = {}
        with open(output_filename, "r", encoding="utf-8") as f:
            for line in f:
                data = json.loads(line)
                custom_id = data['custom_id']
                # Extract the label from the response
                label = data['response']['body']['choices'][0]['message']['content']
                results_map[custom_id] = label

        # 3. Assign values using mapping to ensure alignment
        # This assumes your custom_id corresponds to the index of your dataframe
        batches[i] = batches[i].copy() # Prevents SettingWithCopyWarning
        batches[i].loc[:, 'output_values_batch'] = batches[i].index.astype(str).map(results_map)

        num_found = len(results_map)
        num_total = len(batches[i])
        print(f"✅ Batch {batch_num} results saved ({num_found}/{num_total} rows successful).")

    else:
        print(f"⏳ Batch {batch_num} is still {status_check.status}.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json
import pandas as pd

# Target Batch 4 (index 3)
i = 3
batch_job = submitted_batches[i]
batch_num = i + 1
current_id = batch_job.id

# 1. Retrieve the latest status and errors
status_check = client.batches.retrieve(current_id)

if status_check.status == 'completed':
    output_filename = f"batch_{batch_num}_output.ndjson"
    content = client.files.content(status_check.output_file_id).content
    with open(output_filename, "wb") as f:
        f.write(content)

    # 2. Build the results map from the output file
    results_map = {}
    with open(output_filename, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            custom_id = data['custom_id']
            # Get content if successful, otherwise you can check 'error' fields
            label = data['response']['body']['choices'][0]['message']['content']
            results_map[custom_id] = label

    # 3. Handle Errors/Missing Rows
    # If a row failed entirely, it won't be in the output_file, but might be in errors_file
    if status_check.error_file_id:
        error_content = client.files.content(status_check.error_file_id).content
        with open(f"batch_{batch_num}_errors.ndjson", "wb") as f:
            f.write(error_content)

        with open(f"batch_{batch_num}_errors.ndjson", "r") as f:
            for line in f:
                err_data = json.loads(line)
                # Mark the specific row with its error code/message
                results_map[err_data['custom_id']] = f"Error: {err_data['error']['code']}"

    # 4. Map back to the dataframe using the index to guarantee order
    batches[i] = batches[i].copy()
    # We use the index as the string 'custom_id' to pull from our map
    batches[i]['output_values_batch'] = batches[i].index.astype(str).map(results_map)

    # Fill any remaining NaNs (rows that didn't even make it to the error file)
    batches[i]['output_values_batch'] = batches[i]['output_values_batch'].fillna("Unknown Batch Error")

    print(f"✅ Batch {batch_num} processed. {len(results_map)} results mapped by ID.")
else:
    print(f"Status is {status_check.status}. Errors (if any): {status_check.errors}")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json

# List of the files you already downloaded
# Make sure these filenames match what is actually on your disk
for i in range(len(batches)):
    batch_num = i + 1
    output_filename = f"batch_{batch_num}_output.ndjson"

    try:
        output_labels = []
        with open(output_filename, "r", encoding="utf-8") as f:
            for line in f:
                data = json.loads(line)
                # Just grab the content in order
                label = data['response']['body']['choices'][0]['message']['content']
                output_labels.append(label)

        # Check if the length matches (it might be off by 1 for Batch 4)
        if len(output_labels) == len(batches[i]):
            batches[i]['output_values_batch'] = output_labels
            print(f"✅ Batch {batch_num} recovered perfectly.")
        else:
            # For Batch 4, we'll pad the missing value with None to make it fit
            print(f"⚠️ Batch {batch_num} has a mismatch ({len(output_labels)} vs {len(batches[i])}). Padding...")
            while len(output_labels) < len(batches[i]):
                output_labels.append(None)
            batches[i]['output_values_batch'] = output_labels

    except FileNotFoundError:
        print(f"❌ Could not find {output_filename}. You'll need the new batch for this one.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json

# 1. Create a clean list of just the outputs in the order they appear
batch_4_outputs_list = []
output_filename = "batch_4_output.ndjson"

with open(output_filename, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        # Extract the label text only
        label = data['response']['body']['choices'][0]['message']['content']
        batch_4_outputs_list.append(label)

# 2. Verify the list
print(f"Total outputs extracted: {len(batch_4_outputs_list)}")
print("First 10 values in order:", batch_4_outputs_list[:10])

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json

# 1. Get the set of all IDs that successfully processed
successful_ids = set()
with open("batch_4_output.ndjson", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        # Assuming your custom_id was the row index (e.g., "7540")
        successful_ids.add(int(data['custom_id'].split('-')[-1]))

# 2. Find the index in batches[3] that is NOT in the successful set
legit_error_index = None
for idx in batches[3].index:
    if idx not in successful_ids:
        legit_error_index = idx
        break

# 3. Display the result
if legit_error_index is not None:
    print(f"The legitimate error is at Index: {legit_error_index}")
    print("\n--- Text Content ---")
    print(batches[3].loc[legit_error_index, 'all_text'])
else:
    print("No missing index found. Check if custom_id format matches.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json

# 1. Load the 9999 successful labels into an iterator
with open("batch_4_output.ndjson", "r", encoding="utf-8") as f:
    labels_iter = iter([json.loads(line)['response']['body']['choices'][0]['message']['content'] for line in f])

# 2. Build the final 10,000 item list
final_batch_4_list = []

for idx in batches[3].index:
    if idx == 8705:
        # This is the legit error index the previous code found
        final_batch_4_list.append("error")
    else:
        # Pull the next label from the file
        try:
            final_batch_4_list.append(next(labels_iter))
        except StopIteration:
            final_batch_4_list.append("error")

# 3. Assign to your dataframe
batches[3] = batches[3].copy()
batches[3]['output_values_batch'] = final_batch_4_list

print(f"Done! Row 8705 is now labeled: {batches[3].loc[8705, 'output_values_batch']}")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
# This will return the actual DataFrame index of the missing row
error_index = batches[3][batches[3]['output_values_batch'].isna()].index.tolist()

print(f"The error occurs at index: {error_index}")

# To see the specific content of that row:
batches[3].loc[error_index]

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
batches[3].output_values_batch.value_counts()

In [ ]:
# Purpose: Save the current processed dataframe or analysis output for reuse/reproducibility.
import pandas as pd

# 1. Combine all 10 batches (assuming they are stored in the list 'batches')
# Using ignore_index=False to keep your global original indices (0-99999)
final_df = pd.concat(batches, axis=0, ignore_index=False)

# 2. Cleanup: Ensure the output column name is consistent
# If you used different names during testing, you can rename them here
if 'output_values_batch' in final_df.columns:
    final_df = final_df.rename(columns={'output_values_batch': 'LLM_Predictions'})


# # 4. Quick check of the first few rows
# final_df.head()
len(final_df)

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
# 1. Define the normalization mapping
normalization_map = {
    # Normalize 'Neither' variations
    '[Neither]': 'Neither',
    'NEITHER': 'Neither',
    'Not Harmful': 'Neither',

    # Normalize 'Trust' variations
    '[Trust]': 'Trust',

    # Normalize 'Distrust' variations
    '[Distrust]': 'Distrust',

    # Normalize 'Both' variations
    '[Both]': 'Both'
}

# 2. Apply the mapping
# We use regex=False for speed since these are direct string matches
final_df['LLM_Predictions'] = final_df['LLM_Predictions'].replace(normalization_map)

# 4. Verify the clean counts
print(final_df['LLM_Predictions'].value_counts())

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
final_df.head()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt

# 1. Cleaning and conversion
# Exclude error rows and convert timestamps
df_filtered = final_df[final_df['LLM_Predictions'] != 'error'].copy()
df_filtered['date'] = pd.to_datetime(df_filtered['created_utc'], unit='s')

# 2. Group by month and calculate the total count of rows
df_filtered['month'] = df_filtered['date'].dt.to_period('M')
total_counts = df_filtered.groupby('month').size()

# 3. Convert months to strings in 'YYYY-MM' format for the x-axis
month_strings = [m.strftime('%Y-%m') for m in total_counts.index]

# 4. Plotting
fig, ax = plt.subplots(figsize=(14, 7))

# Plot total counts
ax.plot(month_strings, total_counts.values, marker='o', color='purple', label='Total Rows')

plt.title('Total Number of Rows Over Time')
plt.ylabel('Frequency (Count)')
plt.grid(True, linestyle='--', alpha=0.6)

# 5. Formatting: Every 3 months on x-axis
plt.xticks(ticks=range(0, len(month_strings), 3), labels=month_strings[::3], rotation=45)

# Note: We omit plt.ylim(-0.5, 0.5) here because these are absolute counts
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

plt.show()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt


# 2. Exclude the row(s) with 'error' in LLM_Predictions
df_filtered = final_df[final_df['LLM_Predictions'] != 'error'].copy()

# 3. Convert created_utc to datetime objects
# The unit='s' converts the Unix timestamp (seconds) to a datetime
df_filtered['date'] = pd.to_datetime(df_filtered['created_utc'], unit='s')

# 4. Group by month and the LLM_Predictions label
# You can also use 'W' for weekly or 'D' for daily if you want more granularity
df_filtered['month'] = df_filtered['date'].dt.to_period('M')
distribution_over_time = df_filtered.groupby(['month', 'LLM_Predictions']).size().unstack(fill_value=0)

# 5. Plot the frequency distribution
distribution_over_time.plot(kind='line', marker='o', figsize=(12, 6))

plt.title('Frequency Distribution of LLM_Predictions Over Time')
plt.xlabel('Time (Monthly)')
plt.ylabel('Frequency (Count)')
plt.legend(title='Labels')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt

# 1. Cleaning and conversion
# Exclude error rows and convert timestamps
df_filtered = final_df[final_df['LLM_Predictions'] != 'error'].copy()
df_filtered['date'] = pd.to_datetime(df_filtered['created_utc'], unit='s')

# 2. Group by month and normalize frequencies
df_filtered['month'] = df_filtered['date'].dt.to_period('M')
distribution = df_filtered.groupby(['month', 'LLM_Predictions']).size().unstack(fill_value=0)
normalized_dist = distribution.div(distribution.sum(axis=1), axis=0)

# 3. Convert months to strings in 'YYYY-MM' format to prevent auto-formatting
month_strings = [m.strftime('%Y-%m') for m in normalized_dist.index]

# 4. Plotting
fig, ax = plt.subplots(figsize=(14, 7))

for column in normalized_dist.columns:
    ax.plot(month_strings, normalized_dist[column], marker='o', label=column)

plt.title('Normalized Frequency Distribution of LLM Predictions')
plt.ylabel('Proportion')
plt.grid(True, linestyle='--', alpha=0.6)

# 5. Set X-Axis to show every 3 months with the requested format
plt.xticks(ticks=range(0, len(month_strings), 3), labels=month_strings[::3], rotation=45)

# 6. Set Y-Axis range from 0 to 0.8 as requested
plt.ylim(0, 0.6)

plt.legend(title='Labels', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

plt.show()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt

# 1. Data Cleaning
df_filtered = final_df[final_df['LLM_Predictions'] != 'error'].copy()
df_filtered['date'] = pd.to_datetime(df_filtered['created_utc'], unit='s')

# 2. Monthly Grouping and Normalization
df_filtered['month'] = df_filtered['date'].dt.to_period('M')
distribution = df_filtered.groupby(['month', 'LLM_Predictions']).size().unstack(fill_value=0)
normalized_dist = distribution.div(distribution.sum(axis=1), axis=0)

# 3. Calculate the Difference (Trust - Distrust)
trust_vals = normalized_dist['Trust'] if 'Trust' in normalized_dist.columns else 0
distrust_vals = normalized_dist['Distrust'] if 'Distrust' in normalized_dist.columns else 0
normalized_dist['Diff_Trust_Distrust'] = trust_vals - distrust_vals

# 4. Format X-axis labels to 'YYYY-MM'
month_strings = [m.strftime('%Y-%m') for m in normalized_dist.index]

# 5. Plotting
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(month_strings, normalized_dist['Diff_Trust_Distrust'],
        marker='o', color='tab:blue', label='Trust - Distrust')

plt.title('Monthly Difference: Normalized Trust vs Distrust')
plt.ylabel('Difference (Proportion)')
plt.grid(True, linestyle='--', alpha=0.6)

# 6. Formatting: Every 3 months and Y-range -0.5 to 0.5
plt.xticks(ticks=range(0, len(month_strings), 3), labels=month_strings[::3], rotation=45)

# SET Y-AXIS RANGE
plt.ylim(-0.2, 0.2)

# Keeps the zero line visible to easily distinguish Trust-heavy vs Distrust-heavy months
plt.axhline(0, color='black', linewidth=0.8, linestyle='-')

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

plt.show()

# Task 1 Batch

In [ ]:
# Purpose: Define the Task 1 prompt for classifying trust, distrust, both, or neither.
import json
import math
import re


# ---- 1) Constants you define once ----
MODEL = "gpt-5.1-2025-11-13"                 # or any chat model you want
SYSTEM_PROMPT = task_1_prompt  # your constant prompt
TEXT_COL = "all_text"

# (optional) basic cleaner so inputs won't break batching
RESERVED = re.compile(r"<\|[^>]+?\|>")  # catches tokens like <|endoftext|>
def sanitize(s):
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = str(s).replace("\r\n", "\n").strip()
    s = RESERVED.sub("[RESERVED_TOKEN]", s)
    return s[:8000]  # keep inputs reasonable

# ---- 2) Writer: one JSON object per line ----
def write_ndjson_from_df(df, custom_id_col=None, max_tokens=1000, temperature=0):
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        for i, row in df.reset_index(drop=True).iterrows():
            custom_id = (
                str(row[custom_id_col])
                if custom_id_col and custom_id_col in df.columns
                else f"row-{i+1:06d}"
            )
            user_text = sanitize(row[TEXT_COL])

            obj = {
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": MODEL,
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": user_text}
                    ],
                    #"max_completion_tokens": max_tokens,
                    "temperature": 0
                }
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import os
import json

submitted_batches = []

for i, batch_df in enumerate(batches):
    batch_num = i + 1
    input_filename = f"batch_{batch_num}_input.ndjson"

    print(f"--- Processing Batch {batch_num}/{len(batches)} ---")

    # Reset index and use it as the custom_id column
    temp_df = batch_df.copy()
    temp_df['row_id'] = temp_df.index.astype(str)


    # 1. Create the local file
    write_ndjson_from_df(temp_df, custom_id_col='row_id')

    # Check if the default file exists before renaming
    if os.path.exists("batch_1_input.ndjson"):
        os.rename("batch_1_input.ndjson", input_filename)
    else:
        raise FileNotFoundError(f"write_ndjson_from_df failed to create batch_1_input.ndjson for batch {batch_num}")

    # 2. Upload and 3. Create Batch Job (Your existing logic is correct)
    batch_input_file = client.files.create(
        file=open(input_filename, "rb"),
        purpose="batch"
    )

    batch_job = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={"description": f"classification - Batch {batch_num}"}
    )

    submitted_batches.append(batch_job)
    print(f"Submitted Batch ID: {batch_job.id}")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json
import pandas as pd

for i, batch_job in enumerate(submitted_batches):
    batch_num = i + 1
    current_id = batch_job.id

    print(f"Checking Batch {batch_num} ({current_id})...")

    # Refresh the batch status
    status_check = client.batches.retrieve(current_id)

    if status_check.status == 'completed':
        output_filename = f"batch_{batch_num}_output.ndjson"

        # 1. Download the output file
        file_id = status_check.output_file_id
        content = client.files.content(file_id).content
        with open(output_filename, "wb") as f:
            f.write(content)

        # 2. Map results using custom_id to handle missing/failed rows
        results_map = {}
        with open(output_filename, "r", encoding="utf-8") as f:
            for line in f:
                data = json.loads(line)
                custom_id = data['custom_id']
                # Extract the label from the response
                label = data['response']['body']['choices'][0]['message']['content']
                results_map[custom_id] = label

        # 3. Assign values using mapping to ensure alignment
        # This assumes your custom_id corresponds to the index of your dataframe
        batches[i] = batches[i].copy() # Prevents SettingWithCopyWarning
        batches[i]['output_values_batch'] = None  # Or use .astype(object)
        batches[i]['output_values_batch'] = batches[i]['output_values_batch'].astype(object)
        batches[i].loc[:, 'output_values_batch'] = batches[i].index.astype(str).map(results_map)

        num_found = len(results_map)
        num_total = len(batches[i])
        print(f"✅ Batch {batch_num} results saved ({num_found}/{num_total} rows successful).")

    else:
        print(f"⏳ Batch {batch_num} is still {status_check.status}.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json

# 1. Get the set of all IDs that successfully processed
successful_ids = set()
with open("batch_4_output.ndjson", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        # Assuming your custom_id was the row index (e.g., "7540")
        successful_ids.add(int(data['custom_id'].split('-')[-1]))

# 2. Find the index in batches[3] that is NOT in the successful set
legit_error_index = None
for idx in batches[3].index:
    if idx not in successful_ids:
        legit_error_index = idx
        break

# 3. Display the result
if legit_error_index is not None:
    print(f"The legitimate error is at Index: {legit_error_index}")
    print("\n--- Text Content ---")
    print(batches[3].loc[legit_error_index, 'all_text'])
else:
    print("No missing index found. Check if custom_id format matches.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json

# 1. Load the 9999 successful labels into an iterator
with open("batch_4_output.ndjson", "r", encoding="utf-8") as f:
    labels_iter = iter([json.loads(line)['response']['body']['choices'][0]['message']['content'] for line in f])

# 2. Build the final 10,000 item list
final_batch_4_list = []

for idx in batches[3].index:
    if idx == 38756:
        # This is the legit error index the previous code found
        final_batch_4_list.append("error")
    else:
        # Pull the next label from the file
        try:
            final_batch_4_list.append(next(labels_iter))
        except StopIteration:
            final_batch_4_list.append("error")

# 3. Assign to your dataframe
batches[3] = batches[3].copy()
batches[3]['output_values_batch'] = final_batch_4_list

print(f"Done! Row 38756 is now labeled: {batches[3].loc[38756, 'output_values_batch']}")

In [ ]:
# Purpose: Save the current processed dataframe or analysis output for reuse/reproducibility.
import pandas as pd

# 1. Combine all 10 batches (assuming they are stored in the list 'batches')
# Using ignore_index=False to keep your global original indices (0-99999)
final_df = pd.concat(batches, axis=0, ignore_index=False)

# 2. Cleanup: Ensure the output column name is consistent
# If you used different names during testing, you can rename them here
if 'output_values_batch' in final_df.columns:
    final_df = final_df.rename(columns={'output_values_batch': 'LLM_Predictions'})

# 1. Define the normalization mapping
normalization_map = {
    # Normalize 'Neither' variations
    '[Neither]': 'Neither',
    'NEITHER': 'Neither',
    'Not Harmful': 'Neither',

    # Normalize 'Trust' variations
    '[Trust]': 'Trust',
    'Desconfiança': 'Distrust',

    # Normalize 'Distrust' variations
    '[Distrust]': 'Distrust',

    # Normalize 'Both' variations
    '[Both]': 'Both'
}

# 2. Apply the mapping
# We use regex=False for speed since these are direct string matches
final_df['LLM_Predictions'] = final_df['LLM_Predictions'].replace(normalization_map)

# 4. Verify the clean counts
print(len(final_df))
print(final_df['LLM_Predictions'].value_counts())

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt

# 1. Cleaning and conversion
df_filtered = final_df[final_df['LLM_Predictions'] != 'error'].copy()
df_filtered['date'] = pd.to_datetime(df_filtered['created_utc'], unit='s')

# 2. Group by month and calculate counts
df_filtered['month_dt'] = df_filtered['date'].dt.to_period('M')
total_counts = df_filtered.groupby('month_dt').size().sort_index()

# --- Filter from 2022-11 onwards (as per your snippet) ---
total_counts = total_counts[total_counts.index >= "2022-11"]

# 3. Convert months to strings for the x-axis
month_strings = [m.strftime('%Y-%m') for m in total_counts.index]

# 4. Plotting with NEW LAYOUT
fig, ax = plt.subplots(figsize=(18, 12))  # Increased size

# Plot with thicker line and steelblue color
ax.plot(month_strings, total_counts.values, marker='o',
        color='steelblue', linewidth=5, label='Total Rows')

plt.title('Total Number of Rows Over Time', fontsize=20, fontweight='bold')
plt.ylabel('Frequency (Count)', fontsize=16, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)

# 5. Custom X-Ticks and Bold Formatting
ax.set_xticks(range(len(month_strings)))
ax.set_xticklabels(month_strings, rotation=45, fontsize=16)

# Make ticks bold
for label in ax.get_xticklabels():
    label.set_fontweight('bold')
for label in ax.get_yticklabels():
    label.set_fontweight('bold')
    label.set_fontsize(16)

plt.tight_layout()
plt.show()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt

# 1. Cleaning and conversion
df_filtered = final_df[final_df['LLM_Predictions'] != 'error'].copy()
df_filtered['date'] = pd.to_datetime(df_filtered['created_utc'], unit='s')

# 2. Group by month and calculate counts
df_filtered['month_dt'] = df_filtered['date'].dt.to_period('M')
total_counts = df_filtered.groupby('month_dt').size().sort_index()

# --- Filter from 2022-11 onwards ---
total_counts = total_counts[total_counts.index >= "2022-11"]

# 3. Convert months to strings for the x-axis
month_strings = [m.strftime('%Y-%m') for m in total_counts.index]

# 4. Plotting with NEW LAYOUT
fig, ax = plt.subplots(figsize=(18, 15))

# Plot with thicker line and steelblue color
ax.plot(month_strings, total_counts.values, marker='o', color='steelblue', linewidth=5, label='Total Rows')

# --- Add Event Markers ---
events = {
    "2022-11": "ChatGPT launch",
    "2023-03": "GPT-4 release",
    "2023-07": "LLaMA 2",
    "2023-11": "OpenAI Dev Day",
    "2024-03": "Claude 3",
    "2024-04": "LLaMA 3",
    "2024-06": "Claude 3.5",
    "2025-04": "LLaMA 4",
    "2025-05": "Claude 4"
}

y_max = total_counts.max()
# ... (previous code)

for date_str, label in events.items():
    if date_str in month_strings:
        idx = month_strings.index(date_str)
        # Add vertical line
        ax.axvline(x=idx, color='black', linestyle='--', linewidth=4, alpha=0.6)

        # Updated text label with 45-degree rotation
        ax.text(idx, ax.get_ylim()[1], f" {label}",
                color='black',
                rotation=60,          # Changed from 90 to 45
                va='bottom',
                ha='left',            # Changed to 'left' for better 45-degree alignment
                fontweight='bold',
                fontsize=30)

# ... (rest of your plotting and savefig code)

#plt.title('Total Number of Rows Over Time with AI Milestones', fontsize=20, fontweight='bold')
# plt.ylabel('Frequency (Count)', fontsize=16, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.8)

# 5. Custom X-Ticks and Bold Formatting
ax.set_xticks(range(len(month_strings)))
ax.set_xticklabels(month_strings, rotation=45, fontsize=18)

# Make ticks bold
for label in ax.get_xticklabels():
    label.set_fontweight('bold')
    label.set_fontsize(24)
for label in ax.get_yticklabels():
    label.set_fontweight('bold')
    label.set_fontsize(36)

plt.tight_layout()

# Save the plot as a PDF
plt.savefig('TACL_RR_Fig1.pdf', format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt

# 1. Cleaning and conversion
df_filtered = final_df[final_df['LLM_Predictions'] != 'error'].copy()
df_filtered['date'] = pd.to_datetime(df_filtered['created_utc'], unit='s')

# 2. Group by month and calculate counts
df_filtered['month_dt'] = df_filtered['date'].dt.to_period('M')
total_counts = df_filtered.groupby('month_dt').size().sort_index()

# --- Filter from 2022-11 onwards ---
total_counts = total_counts[total_counts.index >= "2022-11"]

# 3. Convert months to strings for the x-axis
month_strings = [m.strftime('%Y-%m') for m in total_counts.index]

# ... (Keep cleaning and total_counts logic the same)

# 4. Plotting with NEW LAYOUT
fig, ax = plt.subplots(figsize=(18, 15))

# Plot with thicker line and steelblue color
ax.plot(month_strings, total_counts.values, marker='o', color='steelblue', linewidth=5, label='Total Rows')

# --- Add Event Markers ---
# (events dictionary remains the same)

# --- Updated Event Markers (Using Relative Coordinates) ---
for date_str, label in events.items():
    if date_str in month_strings:
        idx = month_strings.index(date_str)
        ax.axvline(x=idx, color='black', linestyle='--', linewidth=4, alpha=0.6)

        # Use transform=ax.transAxes to keep text at the top regardless of Y scale
        ax.text(idx, 0.95, f" {label}",
                transform=ax.get_xaxis_transform(), # Keeps X in data units, Y in relative (0-1)
                color='black',
                rotation=45,
                va='top',
                ha='left',
                fontweight='bold',
                fontsize=26)

#         # 5. Event Markers with 45-Degree Rotation
# for date_str, label in events.items():
#     if date_str in month_strings:
#         idx = month_strings.index(date_str)
#         ax.axvline(x=idx, color='black', linestyle='--', linewidth=4, alpha=0.6)

#         # Text label with requested 45-degree rotation
#         ax.text(idx, ax.get_ylim()[1], f" {label}",
#                 color='black',
#                 rotation=45,          # Updated to 45
#                 va='bottom',
#                 ha='left',            # Left alignment works best for 45-degree text
#                 fontweight='bold',
#                 fontsize=26)

# 5. Custom X-Ticks and Bold Formatting (Updated to 3-month interval)
plt.grid(True, linestyle='--', alpha=0.8)

# Set ticks every 3 months
ax.set_xticks(range(0, len(month_strings), 3))

# Apply labels with increased font size
ax.set_xticklabels(month_strings[::3], rotation=45, fontsize=32, fontweight='bold')

# Ensure Y-axis ticks are also large and bold
for label in ax.get_yticklabels():
    label.set_fontweight('bold')
    label.set_fontsize(36)

plt.tight_layout()

# Save the plot as a PDF
plt.savefig('TACL_RR_Fig1_Updated.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt


# 2. Exclude the row(s) with 'error' in LLM_Predictions
df_filtered = final_df[final_df['LLM_Predictions'] != 'error'].copy()

# 3. Convert created_utc to datetime objects
# The unit='s' converts the Unix timestamp (seconds) to a datetime
df_filtered['date'] = pd.to_datetime(df_filtered['created_utc'], unit='s')

# 4. Group by month and the LLM_Predictions label
# You can also use 'W' for weekly or 'D' for daily if you want more granularity
df_filtered['month'] = df_filtered['date'].dt.to_period('M')
distribution_over_time = df_filtered.groupby(['month', 'LLM_Predictions']).size().unstack(fill_value=0)

# 5. Plot the frequency distribution
distribution_over_time.plot(kind='line', marker='o', figsize=(12, 6))

plt.title('Frequency Distribution of LLM_Predictions Over Time')
plt.xlabel('Time (Monthly)')
plt.ylabel('Frequency (Count)')
plt.legend(title='Labels')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt

# 1. Cleaning and conversion
df_filtered = final_df[final_df['LLM_Predictions'] != 'error'].copy()
df_filtered['date'] = pd.to_datetime(df_filtered['created_utc'], unit='s')

# 2. Group by month and normalize frequencies
df_filtered['month'] = df_filtered['date'].dt.to_period('M')
distribution = df_filtered.groupby(['month', 'LLM_Predictions']).size().unstack(fill_value=0)
normalized_dist = distribution.div(distribution.sum(axis=1), axis=0)

# Filter from 2022-11 onwards
normalized_dist = normalized_dist[normalized_dist.index >= "2022-11"]

# 3. Convert months to strings
month_strings = [m.strftime('%Y-%m') for m in normalized_dist.index]

# 4. Plotting with UPDATED LAYOUT
# 4. Plotting
fig, ax = plt.subplots(figsize=(18, 10))

for column in normalized_dist.columns:
    # --- ADDED CONDITION: Calculate proportions for all, but skip drawing 'Neither' ---
    if column == 'Neither':
        continue

    ax.plot(month_strings, normalized_dist[column], marker='o', linewidth=5, label=column)

events = {
    "2022-11": "ChatGPT launch",
    "2023-03": "GPT-4 release",
    "2023-07": "LLaMA 2",
    "2023-11": "OpenAI Dev Day",
    "2024-03": "Claude 3",
    "2024-04": "LLaMA 3",
    "2024-06": "Claude 3.5",
    "2025-04": "LLaMA 4",
    "2025-05": "Claude 4"
}

# (The rest of your event loop remains the same)
for date_str, label in events.items():
    if date_str in month_strings:
        idx = month_strings.index(date_str)
        # Add vertical line
        ax.axvline(x=idx, color='black', linestyle='--', linewidth=4, alpha=0.6)

        # Updated text label with 45-degree rotation
        ax.text(idx, ax.get_ylim()[1], f" {label}",
                color='black',
                rotation=90,          # Changed from 90 to 45
                va='bottom',
                ha='left',            # Changed to 'left' for better 45-degree alignment
                fontweight='bold',
                fontsize=30)

# 5. Formatting Ticks and Labels
plt.grid(True, linestyle='--', alpha=0.6)
ax.set_xticks(range(len(month_strings)))
ax.set_xticklabels(month_strings, rotation=60, fontsize=24, fontweight='bold')

for label in ax.get_yticklabels():
    label.set_fontweight('bold')
    label.set_fontsize(36)

# 6. Legend and Save
plt.ylim(0, 0.7)
plt.legend(title='Labels', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=16)
plt.tight_layout()

# Save the plot as PDF
plt.savefig('TACL_RR_Figure2.pdf', format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt

# 1. Cleaning and conversion
df_filtered = final_df[final_df['LLM_Predictions'] != 'error'].copy()
df_filtered['date'] = pd.to_datetime(df_filtered['created_utc'], unit='s')

# 2. Group by month and normalize frequencies
df_filtered['month'] = df_filtered['date'].dt.to_period('M')
distribution = df_filtered.groupby(['month', 'LLM_Predictions']).size().unstack(fill_value=0)
normalized_dist = distribution.div(distribution.sum(axis=1), axis=0)

# Filter from 2022-11 onwards
normalized_dist = normalized_dist[normalized_dist.index >= "2022-11"]

# 3. Convert months to strings
month_strings = [m.strftime('%Y-%m') for m in normalized_dist.index]

# 4. Plotting with UPDATED LAYOUT
# ... (Keep cleaning and normalization logic the same)

# 4. Plotting
fig, ax = plt.subplots(figsize=(18, 10))

for column in normalized_dist.columns:
    if column == 'Neither':
        continue
    ax.plot(month_strings, normalized_dist[column], marker='o', linewidth=5, label=column)

# 5. Event Markers with 45-Degree Rotation
for date_str, label in events.items():
    if date_str in month_strings:
        idx = month_strings.index(date_str)
        ax.axvline(x=idx, color='black', linestyle='--', linewidth=4, alpha=0.6)

        # Text label with requested 45-degree rotation
        ax.text(idx, ax.get_ylim()[1], f" {label}",
                color='black',
                rotation=45,          # Updated to 45
                va='bottom',
                ha='left',            # Left alignment works best for 45-degree text
                fontweight='bold',
                fontsize=26)

# 6. Formatting Ticks and Labels (3-month interval & Large Font)
plt.grid(True, linestyle='--', alpha=0.6)

# Set ticks every 3 months
ax.set_xticks(range(0, len(month_strings), 3))
# Increase x-axis font size (Set to 32 for better visibility)
ax.set_xticklabels(month_strings[::3], rotation=45, fontsize=32, fontweight='bold')

for label in ax.get_yticklabels():
    label.set_fontweight('bold')
    label.set_fontsize(36)

# 7. Legend and Save
plt.ylim(0, 0.5) # Set Y-axis to (0, 1)
plt.legend(title='Labels', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=30)
plt.tight_layout()

plt.savefig('TACL_RR_Figure2_Updated.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt

# 1. Cleaning and conversion
df_filtered = final_df[final_df['LLM_Predictions'] != 'error'].copy()
df_filtered['date'] = pd.to_datetime(df_filtered['created_utc'], unit='s')

# 2. Group by month and normalize frequencies
df_filtered['month'] = df_filtered['date'].dt.to_period('M')
distribution = df_filtered.groupby(['month', 'LLM_Predictions']).size().unstack(fill_value=0)
normalized_dist = distribution.div(distribution.sum(axis=1), axis=0)

# Filter from 2022-11 onwards to match the new style's logic
normalized_dist = normalized_dist[normalized_dist.index >= "2022-11"]

# 3. Convert months to strings
month_strings = [m.strftime('%Y-%m') for m in normalized_dist.index]

# 4. Plotting with UPDATED LAYOUT
fig, ax = plt.subplots(figsize=(18, 12))  # Updated figure size

for column in normalized_dist.columns:
    # Increased linewidth to 5 to match the requested style
    ax.plot(month_strings, normalized_dist[column], marker='o', linewidth=5, label=column)

plt.title('Normalized Frequency Distribution of LLM Predictions', fontsize=20, fontweight='bold')
plt.ylabel('Proportion', fontsize=16, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)

# 5. Set custom x-ticks and bold formatting
ax.set_xticks(range(len(month_strings)))
ax.set_xticklabels(month_strings, rotation=45, fontsize=16)

# Make ticks bold
for label in ax.get_xticklabels():
    label.set_fontweight('bold')
for label in ax.get_yticklabels():
    label.set_fontweight('bold')
    label.set_fontsize(16)

# 6. Set Y-Axis range
plt.ylim(0, 0.6)

plt.legend(title='Labels', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt

# 1. Data Cleaning
df_filtered = final_df[final_df['LLM_Predictions'] != 'error'].copy()
df_filtered['date'] = pd.to_datetime(df_filtered['created_utc'], unit='s')

# 2. Monthly Grouping and Normalization
df_filtered['month'] = df_filtered['date'].dt.to_period('M')
distribution = df_filtered.groupby(['month', 'LLM_Predictions']).size().unstack(fill_value=0)
normalized_dist = distribution.div(distribution.sum(axis=1), axis=0)

# 3. Calculate the Difference (Trust - Distrust)
trust_vals = normalized_dist['Trust'] if 'Trust' in normalized_dist.columns else 0
distrust_vals = normalized_dist['Distrust'] if 'Distrust' in normalized_dist.columns else 0
normalized_dist['Diff_Trust_Distrust'] = trust_vals - distrust_vals

# --- Filter from 2022-11 onwards for consistency ---
normalized_dist = normalized_dist[normalized_dist.index >= "2022-11"]

# 4. Format X-axis labels to 'YYYY-MM'
month_strings = [m.strftime('%Y-%m') for m in normalized_dist.index]

# 5. Plotting with NEW LAYOUT
fig, ax = plt.subplots(figsize=(18, 12)) # Updated size

# Plot with thicker line and marker
ax.plot(month_strings, normalized_dist['Diff_Trust_Distrust'],
        marker='o', color='steelblue', linewidth=5, label='Trust - Distrust')

plt.title('Monthly Difference: Normalized Trust vs Distrust', fontsize=20, fontweight='bold')
plt.ylabel('Difference (Proportion)', fontsize=16, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)

# 6. Set custom x-ticks (showing all months) and bold formatting
ax.set_xticks(range(len(month_strings)))
ax.set_xticklabels(month_strings, rotation=45, fontsize=16)

# Make ticks bold
for label in ax.get_xticklabels():
    label.set_fontweight('bold')
for label in ax.get_yticklabels():
    label.set_fontweight('bold')
    label.set_fontsize(16)

# SET Y-AXIS RANGE
plt.ylim(-0.2, 0.2)
plt.axhline(0, color='black', linewidth=2, linestyle='-') # Thicker zero line

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt

# 1. Data Cleaning
df_filtered = final_df[final_df['LLM_Predictions'] != 'error'].copy()
df_filtered['date'] = pd.to_datetime(df_filtered['created_utc'], unit='s')

# 2. Monthly Grouping and Normalization
df_filtered['month'] = df_filtered['date'].dt.to_period('M')
distribution = df_filtered.groupby(['month', 'LLM_Predictions']).size().unstack(fill_value=0)
normalized_dist = distribution.div(distribution.sum(axis=1), axis=0)

# 3. Calculate the Difference (Trust - Distrust)
trust_vals = normalized_dist['Trust'] if 'Trust' in normalized_dist.columns else 0
distrust_vals = normalized_dist['Distrust'] if 'Distrust' in normalized_dist.columns else 0
normalized_dist['Diff_Trust_Distrust'] = trust_vals - distrust_vals

# --- Filter from 2022-11 onwards for consistency ---
normalized_dist = normalized_dist[normalized_dist.index >= "2022-11"]

# 4. Format X-axis labels to 'YYYY-MM'
month_strings = [m.strftime('%Y-%m') for m in normalized_dist.index]

# 5. Plotting with UPDATED LAYOUT
fig, ax = plt.subplots(figsize=(18, 10)) # Increased height for rotated text

# Plot the Difference line
ax.plot(month_strings, normalized_dist['Diff_Trust_Distrust'],
        marker='o', color='steelblue', linewidth=5, label='Trust - Distrust')

# --- Add Event Markers (45-degree rotation) ---
events = {
    "2022-11": "ChatGPT launch",
    "2023-03": "GPT-4 release",
    "2023-07": "LLaMA 2",
    "2023-11": "OpenAI Dev Day",
    "2024-03": "Claude 3",
    "2024-04": "LLaMA 3",
    "2024-06": "Claude 3.5",
    "2025-04": "LLaMA 4",
    "2025-05": "Claude 4"
}

for date_str, label in events.items():
    if date_str in month_strings:
        idx = month_strings.index(date_str)
        # Vertical dashed line
        ax.axvline(x=idx, color='black', linestyle='--', linewidth=3, alpha=0.5)

        # Event Text at 45 degrees
        # We place it at the top of the current Y-limit
        ax.text(idx, ax.get_ylim()[1], f" {label}",
                color='black',
                rotation=45,
                va='bottom',
                ha='left',
                fontweight='bold',
                fontsize=24)

# 6. Formatting Ticks and Labels
#plt.ylabel('Difference (Proportion)', fontsize=20, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)

ax.set_xticks(range(len(month_strings)))
ax.set_xticklabels(month_strings, rotation=45, fontsize=18, fontweight='bold')

# Bold Y-axis ticks
for label in ax.get_yticklabels():
    label.set_fontweight('bold')
    label.set_fontsize(18)

# 7. Final Touches and Save
plt.ylim(-0.1, 0.2) # Adjusted range to fit labels at the top
plt.axhline(0, color='black', linewidth=2, linestyle='-') # Zero line

# Make ticks bold
for label in ax.get_xticklabels():
    label.set_fontweight('bold')
    label.set_fontsize(24)
for label in ax.get_yticklabels():
    label.set_fontweight('bold')
    label.set_fontsize(36)

plt.tight_layout()

# Save the plot as a PDF
plt.savefig('TACL_RR_Fig3.pdf', format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt

# 1. Data Cleaning
df_filtered = final_df[final_df['LLM_Predictions'] != 'error'].copy()
df_filtered['date'] = pd.to_datetime(df_filtered['created_utc'], unit='s')

# 2. Monthly Grouping and Normalization
df_filtered['month'] = df_filtered['date'].dt.to_period('M')
distribution = df_filtered.groupby(['month', 'LLM_Predictions']).size().unstack(fill_value=0)
normalized_dist = distribution.div(distribution.sum(axis=1), axis=0)

# 3. Calculate the Difference (Trust - Distrust)
trust_vals = normalized_dist['Trust'] if 'Trust' in normalized_dist.columns else 0
distrust_vals = normalized_dist['Distrust'] if 'Distrust' in normalized_dist.columns else 0
normalized_dist['Diff_Trust_Distrust'] = trust_vals - distrust_vals

# --- Filter from 2022-11 onwards for consistency ---
normalized_dist = normalized_dist[normalized_dist.index >= "2022-11"]

# 4. Format X-axis labels to 'YYYY-MM'
month_strings = [m.strftime('%Y-%m') for m in normalized_dist.index]

# 5. Plotting with UPDATED LAYOUT
fig, ax = plt.subplots(figsize=(18, 10))

# Plot the Difference line
ax.plot(month_strings, normalized_dist['Diff_Trust_Distrust'],
        marker='o', color='steelblue', linewidth=5, label='Trust - Distrust')

# --- Add Event Markers (Existing logic) ---
events = {
    "2022-11": "ChatGPT launch", "2023-03": "GPT-4 release",
    "2023-07": "LLaMA 2", "2023-11": "OpenAI Dev Day",
    "2024-03": "Claude 3", "2024-04": "LLaMA 3",
    "2024-06": "Claude 3.5", "2025-04": "LLaMA 4", "2025-05": "Claude 4"
}
for date_str, label in events.items():
    if date_str in month_strings:
        idx = month_strings.index(date_str)
        ax.axvline(x=idx, color='black', linestyle='--', linewidth=3, alpha=0.5)
        ax.text(idx, ax.get_ylim()[1], f" {label}", color='black',
                rotation=45, va='bottom', ha='left', fontweight='bold', fontsize=24)

# 6. Formatting Ticks and Labels (Updated for 3-month interval)
# ... (Previous logic for data and markers remains the same)

# 6. Formatting Ticks and Labels (Updated Font Sizes and Interval)
plt.grid(True, linestyle='--', alpha=0.6)

# Select every 3rd index for the ticks and labels
ax.set_xticks(range(0, len(month_strings), 3))

# INCREASED font size to 32 (previously 24)
ax.set_xticklabels(month_strings[::3], rotation=45, fontsize=32, fontweight='bold')

# 7. Final Touches and Save
plt.ylim(-0.1, 0.2)  # Fixed range (0, 1)
plt.axhline(0, color='black', linewidth=2, linestyle='-')

# Ensure Y-axis ticks are also large and bold to match
for label in ax.get_yticklabels():
    label.set_fontweight('bold')
    label.set_fontsize(36)

plt.tight_layout()
plt.savefig('TACL_RR_Fig3.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
final_df.columns, len(final_df), final_df['LLM_Predictions'].value_counts()

## 6. Full-corpus Task 2 batch labeling

Filters the full corpus to Trust/Distrust/Both posts, creates Task 2 batch files across models, retrieves outputs, and saves finalized dimension labels.

# Task 2: Batch

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
data_task2= final_df[final_df['LLM_Predictions'].isin(['Trust','Distrust','Both'])]
len(data_task2)

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
batch_size = 10000
num_batches = len(data_task2) // batch_size + int(len(data_task2) % batch_size != 0)

batches = [data_task2[i * batch_size:(i + 1) * batch_size] for i in range(num_batches)]
num_batches, len(batches)

In [ ]:
# Purpose: Define the Task 2 prompt for extracting trust/distrust dimensions.
# ---- 1) Updated Constants & Setup ----
MODEL = "gpt-5-mini-2025-08-07"
TEXT_COL = "all_text"             # The column containing the raw text
CONDITION_COL = "LLM_Predictions" # The column the prompt depends on

# Assuming build_task2_prompt(label) is already defined in your notebook

# ---- 2) Modified Writer for Conditional Prompting ----
def write_ndjson_from_df_conditional(df, out_path, model_name, custom_id_col=None):
    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in df.reset_index(drop=True).iterrows():
            custom_id = str(row[custom_id_col]) if custom_id_col in df.columns else f"row-{i+1:06d}"
            user_text = sanitize(row["all_text"])
            dynamic_system_prompt = build_task2_prompt(row["LLM_Predictions"])

            obj = {
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": model_name, # Now uses the passed argument
                    "messages": [
                        {"role": "system", "content": dynamic_system_prompt},
                        {"role": "user", "content": user_text}
                    ],
                    "temperature": 0
                }
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import os

# Create a directory to store the batch files
os.makedirs("batch_files", exist_ok=True)

generated_files = []

# Using the MODEL variable defined in your previous cell
for i, batch_df in enumerate(batches):
    # Clean the model name for the filename (removing dots or dashes if preferred,
    # but standard OpenAI model strings are usually safe for filenames)
    model_safe_name = MODEL.replace("/", "_")

    file_path = f"batch_files/batch_{model_safe_name}_part_{i+1}.jsonl"

    # Call your conditional writer function
    write_ndjson_from_df_conditional(batch_df, file_path)

    generated_files.append(file_path)
    print(f"Generated: {file_path}")

In [ ]:
# Purpose: Save the current processed dataframe or analysis output for reuse/reproducibility.
import os
import pandas as pd

models_to_run = ['gpt-5-mini-2025-08-07', 'gpt-4o-2024-11-20', 'gpt-4.1-2025-04-14', 'gpt-4.1-nano-2025-04-14']

# 1. Initialize tracking
submitted_batches = []
batch_job_details = []
os.makedirs("batch_files", exist_ok=True)

for model_name in models_to_run:
    print(f"\n🚀 Starting submissions for: {model_name}")
    model_safe = model_name.replace("/", "_").replace(".", "_")

    for i, batch_df in enumerate(batches):
        file_path = f"batch_files/batch_{model_safe}_part_{i+1}.jsonl"

        # Pass the model_name here
        write_ndjson_from_df_conditional(batch_df, file_path, model_name=model_name)

        # Upload
        with open(file_path, "rb") as f:
            batch_input_file = client.files.create(file=f, purpose="batch")

        # Submit
        batch_job = client.batches.create(
            input_file_id=batch_input_file.id,
            endpoint="/v1/chat/completions",
            completion_window="24h",
            metadata={
                "model": model_name,
                "description": f"Exp: {model_name} - Part {i+1}"
            }
        )

        # Track
        submitted_batches.append(batch_job)
        batch_job_details.append({
            "model": model_name,
            "batch_id": batch_job.id,
            "part": i+1
        })

        # THE UPDATE: This will now print 56 times (once per part)
        print(f"  ✅ {model_name} | Part {i+1}/14 submitted successfully.")

        # Optional: Save CSV incrementally so you don't lose data on a crash
        pd.DataFrame(batch_job_details).to_csv("all_models_tracking.csv", index=False)

print("\n🎉 All 56 batches have been submitted!")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import time

print(f"Checking status for {len(submitted_batches)} batches...\n")

status_summary = []

for i, batch_job in enumerate(submitted_batches):
    # 1. Refresh the status from OpenAI
    status_check = client.batches.retrieve(batch_job.id)

    # 2. Extract metadata and progress
    model_name = status_check.metadata.get("model", "Unknown")
    description = status_check.metadata.get("description", f"Part {i+1}")
    status = status_check.status.upper()

    # 3. Get request counts (completed vs total)
    prog = status_check.request_counts
    progress_str = f"({prog.completed}/{prog.total})" if prog else ""

    print(f"[{i+1}/56] {description} | Status: {status} {progress_str}")

    status_summary.append(status)

# Final Summary
completed_count = status_summary.count('COMPLETED')
print(f"\n--- SUMMARY ---")
print(f"✅ Completed: {completed_count}")
print(f"⏳ In Progress/Pending: {len(status_summary) - completed_count}")

In [ ]:
# Purpose: Define the Task 2 prompt for extracting trust/distrust dimensions.
# ---- 1) Updated Constants & Setup ----
MODEL = "gpt-5-mini-2025-08-07"
TEXT_COL = "all_text"             # The column containing the raw text
CONDITION_COL = "LLM_Predictions" # The column the prompt depends on

# Assuming build_task2_prompt(label) is already defined in your notebook

# ---- 2) Modified Writer for Conditional Prompting ----
def write_ndjson_from_df_conditional_without_temp(df, out_path, model_name, custom_id_col=None):
    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in df.reset_index(drop=True).iterrows():
            custom_id = str(row[custom_id_col]) if custom_id_col in df.columns else f"row-{i+1:06d}"
            user_text = sanitize(row["all_text"])
            dynamic_system_prompt = build_task2_prompt(row["LLM_Predictions"])

            obj = {
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": model_name, # Now uses the passed argument
                    "messages": [
                        {"role": "system", "content": dynamic_system_prompt},
                        {"role": "user", "content": user_text}
                    ],
                    # "temperature": 0
                }
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [ ]:
# Purpose: Save the current processed dataframe or analysis output for reuse/reproducibility.
import os
import pandas as pd

# 1. Update list to ONLY include the model you need to rerun
models_to_rerun = ['gpt-5-mini-2025-08-07']

# 2. Ensure the output directory exists
os.makedirs("batch_files", exist_ok=True)

# Note: Make sure 'submitted_batches' and 'batch_job_details' are initialized
# if you want to keep a fresh log, or leave them as is to append to existing logs.
# submitted_batches = []
# batch_job_details = []

for model_name in models_to_rerun:
    print(f"\n🚀 Starting targeted rerun for: {model_name}")
    model_safe = model_name.replace("/", "_").replace(".", "_")

    # Assuming 'batches' is your list of 24 DataFrames defined earlier in the notebook
    for i, batch_df in enumerate(batches):
        file_path = f"batch_files/batch_{model_safe}_part_{i+1}.jsonl"

        # 3. Call your updated writer that excludes the temperature parameter
        write_ndjson_from_df_conditional_without_temp(
            batch_df,
            file_path,
            model_name=model_name,
            custom_id_col=None # Or your specific ID column
        )

        # 4. Upload the new JSONL file
        with open(file_path, "rb") as f:
            batch_input_file = client.files.create(file=f, purpose="batch")

        # 5. Submit the Batch Job
        batch_job = client.batches.create(
            input_file_id=batch_input_file.id,
            endpoint="/v1/chat/completions",
            completion_window="24h",
            metadata={
                "model": model_name,
                "description": f"Rerun: {model_name} - Part {i+1} (No Temp)"
            }
        )

        # 6. Track the new batches
        submitted_batches.append(batch_job)
        batch_job_details.append({
            "model": model_name,
            "batch_id": batch_job.id,
            "part": i+1,
            "status": "Submitted"
        })

        print(f"  ✅ {model_name} | Part {i+1}/{len(batches)} submitted successfully.")

# 7. Save the updated tracking info
pd.DataFrame(batch_job_details).to_csv("rerun_gpt5_mini_tracking.csv", index=False)
print("\n🎉 Targeted GPT-5-mini batches have been submitted!")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import time

# You submitted 14 parts for the gpt-5-mini rerun
num_new_batches = 14
new_mini_batches = submitted_batches[-num_new_batches:]

print(f"Checking status for the {num_new_batches} NEW gpt-5-mini batches... \n")

for i, batch_job in enumerate(new_mini_batches):
    # Refresh status from OpenAI
    status_check = client.batches.retrieve(batch_job.id)

    status = status_check.status.upper()
    prog = status_check.request_counts

    # Format progress (e.g., 800/1000)
    progress_str = f"({prog.completed}/{prog.total})" if prog else ""

    print(f"New Batch {i + 1} | ID: {batch_job.id} | Status: {status} {progress_str}")

print(f"\n✅ Monitoring only the latest gpt-5-mini run.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
print(f"Total batches to check: {len(submitted_batches)}\n")

for i, batch_job in enumerate(submitted_batches):
    # Retrieve the latest status from the OpenAI client
    status_check = client.batches.retrieve(batch_job.id)

    # Extract metadata and counts
    status = status_check.status.upper()
    metadata = status_check.metadata if status_check.metadata else "No Metadata"
    prog = status_check.request_counts

    # Format progress string
    progress_str = f"({prog.completed}/{prog.total})" if prog else "(N/A)"

    print(f"Batch {i+1:02d} | ID: {batch_job.id} | Status: {status:<12} | Progress: {progress_str:<15} | Metadata: {metadata}")

print("\n✅ All batch statuses retrieved.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
print(f"Total batches to check: {len(submitted_batches)}\n")

for i, batch_job in enumerate(submitted_batches[14:]):
    # Retrieve the latest status from the OpenAI client
    status_check = client.batches.retrieve(batch_job.id)

    # Extract metadata and counts
    status = status_check.status.upper()
    metadata = status_check.metadata if status_check.metadata else "No Metadata"
    prog = status_check.request_counts

    # Format progress string
    progress_str = f"({prog.completed}/{prog.total})" if prog else "(N/A)"

    print(f"Batch {i+1:02d} | ID: {batch_job.id} | Status: {status:<12} | Progress: {progress_str:<15} | Metadata: {metadata}")

print("\n✅ All batch statuses retrieved.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json
import pandas as pd

# 1. Define the range starting from the successful gpt-4o batches
target_batches = submitted_batches[14:]

# Dictionary to store results: { 'model_name': { 'custom_id_string': 'response_text' } }
model_data_store = {}

print(f"Processing {len(target_batches)} batches...")

for batch_job in target_batches:
    status_check = client.batches.retrieve(batch_job.id)
    model_name = status_check.metadata.get('model', 'unknown_model')

    if model_name not in model_data_store:
        model_data_store[model_name] = {}

    if status_check.status == 'completed':
        print(f"Downloading {model_name} | ID: {batch_job.id}...")
        file_id = status_check.output_file_id
        content = client.files.content(file_id).content.decode("utf-8")

        for line in content.strip().split('\n'):
            data = json.loads(line)
            # Store results using the custom_id (string) as the key
            model_data_store[model_name][data['custom_id']] = data['response']['body']['choices'][0]['message']['content']
    else:
        print(f"⚠️ Skipping {batch_job.id} - Status: {status_check.status}")

# 2. Create lists and assign to new columns
print("\nGenerating lists and assigning to data_task2...")

for model_name, results_dict in model_data_store.items():
    # We build a list by iterating through the ACTUAL dataframe index
    # We convert the index to a string just for the dictionary lookup
    concatenated_list = [results_dict.get(str(idx), None) for idx in data_task2.index]

    # Assign the list to the new column
    column_name = f"Output_{model_name}"
    data_task2[column_name] = concatenated_list

    # Verify success
    count = data_task2[column_name].notna().sum()
    print(f"✅ Created {column_name}: {count}/{len(data_task2)} rows filled.")

print("\nDone! All model outputs added as columns.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
# Define the models exactly as they appear in your metadata
models = ['gpt-4o-2024-11-20', 'gpt-4.1-2025-04-14', 'gpt-4.1-nano-2025-04-14', 'gpt-5-mini-2025-08-07']

for model_name in models:
    print(f"--- Stitching Model: {model_name} ---")

    # 1. Get all batches for this model from your successful run (index 14 onwards)
    # We retrieve them to make sure we have the right order
    model_batches = []
    for b in submitted_batches[14:]:
        info = client.batches.retrieve(b.id)
        if info.metadata.get('model') == model_name:
            model_batches.append(info)

    # 2. Extract every single result from these batches in order
    all_responses_for_model = []

    for batch_info in model_batches:
        content = client.files.content(batch_info.output_file_id).content.decode("utf-8")

        # We sort lines by custom_id WITHIN the batch to ensure row 1 comes before row 2
        batch_results = []
        for line in content.strip().split('\n'):
            if not line.strip(): continue
            data = json.loads(line)
            batch_results.append(data)

        # Sort by the numeric part of 'row-000001'
        batch_results.sort(key=lambda x: int(x['custom_id'].split('-')[-1]))

        # Add the sorted responses from THIS batch to our big master list
        for item in batch_results:
            all_responses_for_model.append(item['response']['body']['choices'][0]['message']['content'])

    # 3. Assign the big list (should be 135,061 items) to the column
    col_name = f"Output_{model_name}"

    # Safety check: if the list is longer/shorter than the DF, we align it
    if len(all_responses_for_model) >= len(data_task2):
        data_task2.loc[:, col_name] = all_responses_for_model[:len(data_task2)]
    else:
        # Pad with None if somehow we have fewer results than rows
        padding = [None] * (len(data_task2) - len(all_responses_for_model))
        data_task2.loc[:, col_name] = all_responses_for_model + padding

    print(f"✅ {col_name} mapped: {len(all_responses_for_model)} rows total.")

# FINAL VERIFICATION
print("\nFinal Check - Row 0 (Should be Trust dimensions only):")
print(data_task2[['LLM_Predictions', 'Output_gpt-4o-2024-11-20']].head(1))

# Task 2 Batch with Few Shot Models

In [ ]:
# Purpose: Define the Task 2 prompt for extracting trust/distrust dimensions.
# ---- 1) Updated Constants & Setup ----
# MODEL = "gpt-5-mini-2025-08-07"
# TEXT_COL = "all_text"             # The column containing the raw text
# CONDITION_COL = "LLM_Predictions" # The column the prompt depends on

# Create the lookup dictionary once
prompt_cache = {}
labels_to_cache = ["Trust", "Distrust", "Both"]

for label in labels_to_cache:
    prompt_cache[label] = build_task2_prompt_shots(
        task1_label=label,
        X=data_dimensions,
        k=3
    )


# Assuming build_task2_prompt(label) is already defined in your notebook

def write_ndjson_from_df_conditional_few(df, out_path, model_name, custom_id_col=None):
    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in df.reset_index(drop=True).iterrows():
            custom_id = str(row[custom_id_col]) if custom_id_col in df.columns else f"row-{i+1:06d}"
            user_text = sanitize(row["all_text"])

            # LOOKUP: Get the pre-generated prompt based on the row's label
            # If label isn't in cache, default to 'Neither' logic
            label = row["LLM_Predictions"]
            dynamic_system_prompt = prompt_cache.get(label)

            obj = {
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": model_name,
                    "messages": [
                        {"role": "system", "content": dynamic_system_prompt},
                        {"role": "user", "content": user_text}
                    ],
                    "temperature": 0
                }
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [ ]:
# Purpose: Save the current processed dataframe or analysis output for reuse/reproducibility.
import os
import pandas as pd

# 1. Update your model list and naming logic
models_to_run = ['gpt-4.1-2025-04-14', 'gpt-4.1-nano-2025-04-14']

# Initialize tracking for the few-shot run
submitted_batches_fewshot = []
batch_job_details_fewshot = []
os.makedirs("batch_files_fewshot", exist_ok=True)

for model_name in models_to_run:
    print(f"\n🚀 Starting FEW-SHOT submissions for: {model_name}")

    # ADJUSTMENT: Safe name includes 'fewshot' to prevent overwriting zero-shot jsonl files
    model_safe = model_name.replace("/", "_").replace(".", "_") + "_fewshot"

    for i, batch_df in enumerate(batches):
        file_path = f"batch_files_fewshot_batch_{model_safe}_part_{i+1}.jsonl"

        # ADJUSTMENT: Use the few-shot specific writer from Cell 120
        # This function internally calls build_task2_prompt_shots
        write_ndjson_from_df_conditional_few(
            batch_df,
            file_path,
            model_name=model_name,
            custom_id_col="custom_id"
        )

        # Upload the few-shot jsonl
        with open(file_path, "rb") as f:
            batch_input_file = client.files.create(file=f, purpose="batch")

        # Submit with updated metadata to distinguish in the OpenAI Dashboard
        batch_job = client.batches.create(
            input_file_id=batch_input_file.id,
            endpoint="/v1/chat/completions",
            completion_window="24h",
            metadata={
                "model": model_name,
                "setting": "few-shot",
                "description": f"Exp: {model_name} - FewShot - Part {i+1}"
            }
        )

        # Track separately from zero-shot results
        submitted_batches_fewshot.append(batch_job)
        batch_job_details_fewshot.append({
            "model": model_name,
            "batch_id": batch_job.id,
            "part": i+1,
            "setting": "few-shot"
        })

        print(f"  ✅ {model_name} (Few-Shot) | Part {i+1}/{len(batches)} submitted.")

    # Save tracking CSV incrementally with a unique name
    pd.DataFrame(batch_job_details_fewshot).to_csv("fewshot_models_tracking.csv", index=False)

print("\n🎉 All few-shot batches have been submitted successfully!")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import pandas as pd

# Iterate through the batches submitted in cell 128
results = []
for job in submitted_batches_fewshot:
    status_check = client.batches.retrieve(job.id)
    results.append({
        "Model": status_check.metadata.get("model"),
        "Batch ID": status_check.id,
        "Status": status_check.status,
        "Completed": f"{status_check.request_counts.completed}/{status_check.request_counts.total}" if status_check.request_counts else "N/A",
        "Created At": pd.to_datetime(status_check.created_at, unit='s')
    })

# Display as a clean table
status_df = pd.DataFrame(results)
print(status_df)

In [ ]:
# Purpose: Define the Task 2 prompt for extracting trust/distrust dimensions.
# ---- 1) Updated Constants & Setup ----
# MODEL = "gpt-5-mini-2025-08-07"
# TEXT_COL = "all_text"             # The column containing the raw text
# CONDITION_COL = "LLM_Predictions" # The column the prompt depends on

# Create the lookup dictionary once
prompt_cache = {}
labels_to_cache = ["Trust", "Distrust", "Both"]

for label in labels_to_cache:
    prompt_cache[label] = build_task2_prompt_shots(
        task1_label=label,
        X=data_dimensions,
        k=3
    )


# Assuming build_task2_prompt(label) is already defined in your notebook

def write_ndjson_from_df_conditional_few_no_temp(df, out_path, model_name, custom_id_col=None):
    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in df.reset_index(drop=True).iterrows():
            custom_id = str(row[custom_id_col]) if custom_id_col in df.columns else f"row-{i+1:06d}"
            user_text = sanitize(row["all_text"])

            # LOOKUP: Get the pre-generated prompt based on the row's label
            # If label isn't in cache, default to 'Neither' logic
            label = row["LLM_Predictions"]
            dynamic_system_prompt = prompt_cache.get(label)

            obj = {
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": model_name,
                    "messages": [
                        {"role": "system", "content": dynamic_system_prompt},
                        {"role": "user", "content": user_text}
                    ],
                    #"temperature": 0
                }
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [ ]:
# Purpose: Save the current processed dataframe or analysis output for reuse/reproducibility.
import os
import pandas as pd

# 1. Update your model list and naming logic
models_to_run = ['gpt-5-mini-2025-08-07']

# Initialize tracking for the few-shot run
submitted_batches_fewshot = []
batch_job_details_fewshot = []
os.makedirs("batch_files_fewshot", exist_ok=True)

for model_name in models_to_run:
    print(f"\n🚀 Starting FEW-SHOT submissions for: {model_name}")

    # ADJUSTMENT: Safe name includes 'fewshot' to prevent overwriting zero-shot jsonl files
    model_safe = model_name.replace("/", "_").replace(".", "_") + "_fewshot"

    for i, batch_df in enumerate(batches):
        file_path = f"batch_files_fewshot_batch_{model_safe}_part_{i+1}.jsonl"

        # ADJUSTMENT: Use the few-shot specific writer from Cell 120
        # This function internally calls build_task2_prompt_shots
        write_ndjson_from_df_conditional_few_no_temp(
            batch_df,
            file_path,
            model_name=model_name,
            custom_id_col="custom_id"
        )

        # Upload the few-shot jsonl
        with open(file_path, "rb") as f:
            batch_input_file = client.files.create(file=f, purpose="batch")

        # Submit with updated metadata to distinguish in the OpenAI Dashboard
        batch_job = client.batches.create(
            input_file_id=batch_input_file.id,
            endpoint="/v1/chat/completions",
            completion_window="24h",
            metadata={
                "model": model_name,
                "setting": "few-shot",
                "description": f"Exp: {model_name} - FewShot - Part {i+1}"
            }
        )

        # Track separately from zero-shot results
        submitted_batches_fewshot.append(batch_job)
        batch_job_details_fewshot.append({
            "model": model_name,
            "batch_id": batch_job.id,
            "part": i+1,
            "setting": "few-shot"
        })

        print(f"  ✅ {model_name} (Few-Shot) | Part {i+1}/{len(batches)} submitted.")

    # Save tracking CSV incrementally with a unique name
    pd.DataFrame(batch_job_details_fewshot).to_csv("fewshot_models_tracking.csv", index=False)

print("\n🎉 All few-shot batches have been submitted successfully!")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json
import pandas as pd

# Iterate through the gpt-5-mini batches currently in memory
for i, batch_job in enumerate(submitted_batches_fewshot):
    batch_num = i + 1
    current_id = batch_job.id

    print(f"Checking gpt-5-mini Batch {batch_num} ({current_id})...")

    # Refresh status
    status_check = client.batches.retrieve(current_id)

    if status_check.status == 'completed':
        output_filename = f"gpt5_mini_fewshot_batch_{batch_num}_output.ndjson"

        # 1. Download the output
        file_id = status_check.output_file_id
        content = client.files.content(file_id).content
        with open(output_filename, "wb") as f:
            f.write(content)

        # 2. Map results
        results_map = {}
        with open(output_filename, "r", encoding="utf-8") as f:
            for line in f:
                data = json.loads(line)
                c_id = data['custom_id']
                label = data['response']['body']['choices'][0]['message']['content']
                results_map[c_id] = label

        # 3. Add to a specific gpt-5-mini column to avoid confusion
        # We target the corresponding dataframe in your 'batches' list
        target_df = batches[i]
        column_name = 'gpt5_mini_fewshot_results'

        target_df[column_name] = target_df.index.astype(str).map(results_map)

        num_found = len(results_map)
        num_total = len(target_df)
        print(f"✅ Success: {num_found}/{num_total} rows added to '{column_name}'.")

    else:
        print(f"⏳ Batch {batch_num} is still {status_check.status}. (Progress: {status_check.request_counts.completed}/{status_check.request_counts.total})")

print("\nProcessing complete.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
# Exact Cell 72 logic for gpt-5-mini
model_name = 'gpt-5-mini-2025-08-07'
column_name = f"Output_{model_name}_few"

all_responses = []

# Process each batch object in your list
for i, batch_job in enumerate(submitted_batches_fewshot):
    print(f"Processing Batch {i+1}...")
    status_check = client.batches.retrieve(batch_job.id)

    if status_check.status == 'completed':
        # Download and decode
        content = client.files.content(status_check.output_file_id).content.decode("utf-8")

        # Parse lines into a list
        batch_results = [json.loads(line) for line in content.strip().split('\n') if line.strip()]

        # THE CELL 72 SORT: Sort by the numeric suffix of the custom_id
        batch_results.sort(key=lambda x: int(x['custom_id'].split('-')[-1]))

        # Append only the content to our master list
        for item in batch_results:
            all_responses.append(item['response']['body']['choices'][0]['message']['content'])
    else:
        print(f"Skipping Batch {i+1}: Status is {status_check.status}")

# Direct list assignment (ensure length matches data_task2)
if len(all_responses) == len(data_task2):
    data_task2[column_name] = all_responses
    print(f"✅ Success: Added {len(all_responses)} rows to {column_name}")
else:
    print(f"❌ Length Mismatch: Model gave {len(all_responses)} but data_task2 has {len(data_task2)}")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json

models_to_stitch = ['gpt-4.1-2025-04-14', 'gpt-4.1-nano-2025-04-14']

# 1. Retrieve all batches from your account
all_recent_batches = client.batches.list(limit=50)

for model_name in models_to_stitch:
    print(f"--- Stitching Model: {model_name} ---")

    # 2. Filter batches for this specific model and 'few-shot' setting
    # We sort them by description/metadata to ensure we process Part 1 through 14 in order
    model_batches = [
        b for b in all_recent_batches.data
        if b.metadata.get('model') == model_name and b.metadata.get('setting') == 'few-shot'
    ]

    # Sort the batches themselves by their 'Part' number to maintain global order
    model_batches.sort(key=lambda x: int(x.metadata.get('description').split('Part ')[-1]))

    all_responses_for_model = []
    all_parts_ready = True

    for batch_info in model_batches:
        if batch_info.status == 'completed':
            # Download and decode
            content = client.files.content(batch_info.output_file_id).content.decode("utf-8")

            # THE CELL 72 SORT: Parse and sort WITHIN the batch
            batch_results = [json.loads(line) for line in content.strip().split('\n') if line.strip()]
            batch_results.sort(key=lambda x: int(x['custom_id'].split('-')[-1]))

            for item in batch_results:
                all_responses_for_model.append(item['response']['body']['choices'][0]['message']['content'])
        else:
            print(f"  ❌ {batch_info.metadata.get('description')} is {batch_info.status}. Cannot stitch yet.")
            all_parts_ready = False
            break

    # 3. Final Assignment
    column_name = f"Output_{model_name}_few"
    if all_parts_ready and len(all_responses_for_model) == len(data_task2):
        data_task2[column_name] = all_responses_for_model
        print(f"✅ Success: Added {len(all_responses_for_model)} rows to {column_name}\n")
    elif all_parts_ready:
        print(f"❌ Length Mismatch: Found {len(all_responses_for_model)} results, expected {len(data_task2)}\n")

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# 1. Identify all columns starting with 'Output'
output_cols = [col for col in data_task2.columns if col.startswith('Output')]

# 2. Clean each column by removing the '[' and ']' characters
for col in output_cols:
    # Convert to string first to ensure .str methods work, then replace brackets
    data_task2[col] = data_task2[col].astype(str).str.replace(r'[\[\]]', '', regex=True)

# 3. Verify the cleaning
print(f"Cleaned {len(output_cols)} columns: {output_cols}")
data_task2.head(25)

## 7. Analyze full-corpus Task 2 dimension results

Loads finalized Task 2 results and analyzes trust/distrust dimension frequencies, temporal patterns, and model-specific dimension mappings.

# Analyze the Dimensions with Batches Results

In [ ]:
# Purpose: Load a saved dataset or cached model-output file used by later analysis cells.
import pandas as pd
import numpy as np
data_task2= pd.read_csv("Batch_Task2_Finalized.csv", encoding='utf-8')
len(data_task2)

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
data_task2.head()

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
trust_dims = data_task2[data_task2['LLM_Predictions'].isin(['Trust', 'Both'])].copy()
distrust_dims =data_task2[data_task2['LLM_Predictions'].isin(['Distrust', 'Both'])].copy()

trust_dimensions = ['Competence', 'Familiarity', 'Integrity', 'Reliability', 'Transparency', 'Benevolence']
distrust_dimensions = ['Deception', 'Dishonesty', 'Incompetence', 'Malevolence', 'Opaqueness', 'Unreliability', 'Unfamiliarity']

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
def count_dimension_model_data(df, dim, model):
    # Ensure we are looking at the correct column and handle missing values
    # We convert to string and lower case for a robust search
    data_dimension = df[model].fillna('').str.contains(dim, case=False, na=False).sum()

    return data_dimension

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_dimension_frequencies_over_time(df, dimension_model_map, title="Dimension Frequency Over Time"):
    # 1. Convert UTC to datetime
    temp_df = df.copy()
    temp_df['date'] = pd.to_datetime(temp_df['created_utc'], unit='s')

    # Use a monthly frequency for the period
    temp_df['month_year'] = temp_df['date'].dt.to_period('M')

    fig, ax = plt.subplots(figsize=(14, 7))

    # 2. Iterate through each dimension and its assigned model
    for dim, model_col in dimension_model_map.items():
        if model_col not in temp_df.columns:
            print(f"Warning: {model_col} not found in DataFrame. Skipping {dim}.")
            continue

        # Create boolean mask
        temp_df[f'has_{dim}'] = temp_df[model_col].fillna('').apply(
            lambda x: 1 if dim.lower() in str(x).lower() else 0
        )

        # Aggregate
        time_series = temp_df.groupby('month_year')[f'has_{dim}'].sum()

        # Plotting - converting index to timestamp for better axis control
        time_series.index = time_series.index.to_timestamp()
        ax.plot(time_series.index, time_series.values, label=dim, marker='o')

    # 3. Format X-Axis to show ALL months and rotate
    ax.xaxis.set_major_locator(mdates.MonthLocator())  # Sets a tick for every month
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y')) # 'Jan 2023'

    plt.xticks(rotation=45) # Rotate labels by 45 degrees

    plt.title(title)
    plt.xlabel("Time (Month-Year)")
    plt.ylabel("Frequency Count")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# --- Example Usage ---
my_mapping = {
    'Competence': 'Output_gpt-5-mini-2025-08-07',
    'Reliability': 'Output_gpt-4.1-2025-04-14_few',
    'Familiarity': 'Output_gpt-4.1-2025-04-14',
    'Integrity':        'Output_gpt-5-mini-2025-08-07' ,
     'Transparency':     'Output_gpt-4.1-nano-2025-04-14_few',
    'Benevolence':     'Output_gpt-4.1-nano-2025-04-14_few'
}
plot_dimension_frequencies_over_time(data_task2, my_mapping)

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# --- Example Usage ---
my_mapping = {
    'Incompetence': 'Output_gpt-5-mini-2025-08-07',
    'Unreliability': 'Output_gpt-4o-2024-11-20',
    'Unfamiliarity': 'Output_gpt-4.1-nano-2025-04-14',
    'Dishonesty':      'Output_gpt-5-mini-2025-08-07' ,
     'Deception':     'Output_gpt-4o-2024-11-20',
    'Opaqueness':     'Output_gpt-5-mini-2025-08-07_few',
    'Malevolence':     'Output_gpt-4.1-2025-04-14_few'
}
plot_dimension_frequencies_over_time(data_task2, my_mapping)

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
batch_9_file = "batch_outputs/09_output_15114efb-c64a-4ef9-ac8e-0efd0aa0147b.jsonl"
expected_size = 10000
batch_9_aligned = []
batch_data = {}
if os.path.exists(batch_9_file):
    with open(batch_9_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                clean_cid = obj['custom_id'].replace(" ", "")
                try:
                    response_text = obj['response']['body']['choices'][0]['message']['content']
                    batch_data[clean_cid] = response_text
                except (KeyError, TypeError):
                    batch_data[clean_cid] = "error"

for i in range(1, expected_size + 1):
    expected_id = f"row-{i:06d}"
    batch_9_aligned.append(batch_data.get(expected_id, "error"))

print(f"Batch 9 Aligned List Size: {len(batch_9_aligned)}")
print(f"Errors found in Batch 9: {batch_9_aligned.count('error')}")

try:
    error_idx = batch_9_aligned.index("error")
    print(f"Error found at list index: {error_idx}")
    print(f"This corresponds to Custom ID: row-{(error_idx + 40001):06d}")
except ValueError:
    print("No errors found in this batch.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
batch_10_file = "batch_outputs/10_output_3f42f9f3-aad7-41c5-9a86-ea39a9192384.jsonl"
expected_size = 10000
batch_10_aligned = []
batch_data = {}
if os.path.exists(batch_10_file):
    with open(batch_10_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                clean_cid = obj['custom_id'].replace(" ", "")
                try:
                    response_text = obj['response']['body']['choices'][0]['message']['content']
                    batch_data[clean_cid] = response_text
                except (KeyError, TypeError):
                    batch_data[clean_cid] = "error"

for i in range(1, expected_size + 1):
    expected_id = f"row-{i:06d}"
    batch_10_aligned.append(batch_data.get(expected_id, "error"))

print(f"Batch 10 Aligned List Size: {len(batch_10_aligned)}")
print(f"Errors found in Batch 10: {batch_10_aligned.count('error')}")

try:
    error_idx = batch_10_aligned.index("error")
    print(f"Error found at list index: {error_idx}")
    print(f"This corresponds to Custom ID: row-{(error_idx + 40001):06d}")
except ValueError:
    print("No errors found in this batch.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
batch_11_file = "batch_outputs/11_output_7d01c687-d513-4d13-90cc-e99d7d630c8f.jsonl"
expected_size = 10000
batch_11_aligned = []
batch_data = {}
if os.path.exists(batch_11_file):
    with open(batch_11_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                clean_cid = obj['custom_id'].replace(" ", "")
                try:
                    response_text = obj['response']['body']['choices'][0]['message']['content']
                    batch_data[clean_cid] = response_text
                except (KeyError, TypeError):
                    batch_data[clean_cid] = "error"

for i in range(1, expected_size + 1):
    expected_id = f"row-{i:06d}"
    batch_11_aligned.append(batch_data.get(expected_id, "error"))

print(f"Batch 11 Aligned List Size: {len(batch_11_aligned)}")
print(f"Errors found in Batch 11: {batch_11_aligned.count('error')}")

try:
    error_idx = batch_11_aligned.index("error")
    print(f"Error found at list index: {error_idx}")
    print(f"This corresponds to Custom ID: row-{(error_idx + 40001):06d}")
except ValueError:
    print("No errors found in this batch.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
batch_12_file = "batch_outputs/12_output_ab2196cc-cf58-45a9-81ed-ad179ce11bcd.jsonl"
expected_size = 10000
batch_12_aligned = []
batch_data = {}
if os.path.exists(batch_12_file):
    with open(batch_12_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                clean_cid = obj['custom_id'].replace(" ", "")
                try:
                    response_text = obj['response']['body']['choices'][0]['message']['content']
                    batch_data[clean_cid] = response_text
                except (KeyError, TypeError):
                    batch_data[clean_cid] = "error"

for i in range(1, expected_size + 1):
    expected_id = f"row-{i:06d}"
    batch_12_aligned.append(batch_data.get(expected_id, "error"))

print(f"Batch 12 Aligned List Size: {len(batch_12_aligned)}")
print(f"Errors found in Batch 12: {batch_12_aligned.count('error')}")

try:
    error_idx = batch_12_aligned.index("error")
    print(f"Error found at list index: {error_idx}")
    print(f"This corresponds to Custom ID: row-{(error_idx + 40001):06d}")
except ValueError:
    print("No errors found in this batch.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
batch_13_file = "batch_outputs/13_output_0a585bc1-bce8-4191-9a13-84d7b2dbf436.jsonl"
expected_size = 5061
batch_13_aligned = []
batch_data = {}
if os.path.exists(batch_13_file):
    with open(batch_13_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                clean_cid = obj['custom_id'].replace(" ", "")
                try:
                    response_text = obj['response']['body']['choices'][0]['message']['content']
                    batch_data[clean_cid] = response_text
                except (KeyError, TypeError):
                    batch_data[clean_cid] = "error"

for i in range(1, expected_size + 1):
    expected_id = f"row-{i:06d}"
    batch_13_aligned.append(batch_data.get(expected_id, "error"))

print(f"Batch 13 Aligned List Size: {len(batch_13_aligned)}")
print(f"Errors found in Batch 13: {batch_13_aligned.count('error')}")


try:
    error_idx = batch_13_aligned.index("error")
    print(f"Error found at list index: {error_idx}")
    print(f"This corresponds to Custom ID: row-{(error_idx + 40001):06d}")
except ValueError:
    print("No errors found in this batch.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
# 1. Concatenate all batches in order
final_output_list = (
    batch_0_aligned + batch_1_aligned + batch_2_aligned + batch_3_aligned +
    batch_4_aligned + batch_5_aligned + batch_6_aligned + batch_7_aligned +
    batch_8_aligned + batch_9_aligned + batch_10_aligned + batch_11_aligned +
    batch_12_aligned + batch_13_aligned
)

# 2. Add to your DataFrame
data_task2['final_mixtral_output'] = final_output_list

# 3. Final Verification
print(f"Total rows in DataFrame: {len(data_task2)}")
print(f"Total rows in aligned list: {len(final_output_list)}")

# Find all rows where the output is exactly 'error'
error_rows = data_task2[data_task2['final_mixtral_output'] == 'error']

if not error_rows.empty:
    print(f"Found {len(error_rows)} errors in the final column:")
    for idx, row in error_rows.iterrows():
        # idx is the global DataFrame index
        # We calculate the row-ID to confirm it matches your batch IDs
        custom_id = f"row-{idx + 1:06d}"
        print(f"Index: {idx} | ID: {custom_id}")
else:
    print("No 'error' values found in the final column.")

# Analysis Task 2 Again

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
def plot_dimension_frequencies_over_time(df, dimension_model_map, title="Dimension Frequency Over Time"):
    # 1. Convert UTC to datetime
    temp_df = df.copy()
    temp_df['date'] = pd.to_datetime(temp_df['created_utc'], unit='s')
    temp_df['month_year'] = temp_df['date'].dt.to_period('M')

    # --- NORMALIZATION STEP ---
    monthly_totals = temp_df.groupby('month_year').size()

    fig, ax = plt.subplots(figsize=(16, 10))

    # 2. Iterate through each dimension
    for dim, model_col in dimension_model_map.items():
        if model_col not in temp_df.columns:
            print(f"Warning: {model_col} not found in DataFrame. Skipping {dim}.")
            continue

        temp_df[f'has_{dim}'] = temp_df[model_col].fillna('').apply(
            lambda x: 1 if dim.lower() in str(x).lower() else 0
        )

        time_series_sum = temp_df.groupby('month_year')[f'has_{dim}'].sum()
        time_series = time_series_sum / monthly_totals

        time_series.index = time_series.index.to_timestamp()
        ax.plot(time_series.index, time_series.values, label=dim, marker='o', linewidth=5)

    # 3. Format X-Axis (Every 3 months)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%Y'))

    # 4. Set Y-Axis Range to (0, 1)
    ax.set_ylim(0, 1)

    # --- BOLD FONT AND LABEL UPDATES ---
    ax.set_xlabel("")
    plt.xticks(rotation=45, fontsize=24, fontweight='bold')
    plt.yticks(fontsize=28, fontweight='bold')

    ax.legend(bbox_to_anchor=(1.0, 1), loc='upper left', fontsize=20, prop={'size': 25, 'weight': 'bold'})

    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
# --- Example Usage ---
my_mapping = {
    'Competence': 'Output_gpt-5-mini-2025-08-07',
    'Reliability': 'Output_gpt-4.1-2025-04-14_few',
    'Familiarity': 'Output_gpt-4.1-2025-04-14',
    'Integrity':        'final_mixtral_output' ,
     'Transparency':     'Output_gpt-4.1-nano-2025-04-14_few',
    'Benevolence':     'final_mixtral_output'

}
plot_dimension_frequencies_over_time(data_task2, my_mapping)

plt.savefig('Dimensions_Trust_Frequency_Plot.pdf', format='pdf', bbox_inches='tight')

# 3. Show it if it wasn't already shown by the function
plt.show()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
# --- Example Usage ---
my_mapping = {
    'Incompetence': 'Output_gpt-5-mini-2025-08-07',
    'Unreliability': 'final_mixtral_output',
    'Unfamiliarity': 'Output_gpt-4.1-nano-2025-04-14',
    'Dishonesty':      'Output_gpt-5-mini-2025-08-07' ,
     'Deception':     'Output_gpt-4o-2024-11-20',
    'Opaqueness':     'Output_gpt-5-mini-2025-08-07_few',
    'Malevolence':     'Output_gpt-4.1-2025-04-14_few'
}
plot_dimension_frequencies_over_time(data_task2, my_mapping)

plt.savefig('Dimensions_Distrust_Frequency_Plot.pdf', format='pdf', bbox_inches='tight')

# 3. Show it if it wasn't already shown by the function
plt.show()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_dimension_frequencies_over_time(df, dimension_model_map, title="Dimension Frequency Over Time"):
    # 1. Convert UTC to datetime
    temp_df = df.copy()
    temp_df['date'] = pd.to_datetime(temp_df['created_utc'], unit='s')
    temp_df['month_year'] = temp_df['date'].dt.to_period('M')

    # --- NORMALIZATION STEP ---
    monthly_totals = temp_df.groupby('month_year').size()

    fig, ax = plt.subplots(figsize=(16, 10))

    # 2. Iterate through each dimension
    for dim, model_col in dimension_model_map.items():
        if model_col not in temp_df.columns:
            print(f"Warning: {model_col} not found in DataFrame. Skipping {dim}.")
            continue

        # FIXED LOGIC: Word boundaries prevent 'Familiarity' catching 'Unfamiliarity'
        pattern = rf'\b{dim}\b'

        temp_df[f'has_{dim}'] = temp_df[model_col].fillna('').str.contains(
            pattern, case=False, regex=True
        ).astype(int)

        time_series_sum = temp_df.groupby('month_year')[f'has_{dim}'].sum()
        time_series = time_series_sum / monthly_totals

        time_series.index = time_series.index.to_timestamp()
        ax.plot(time_series.index, time_series.values, label=dim, marker='o', linewidth=5)

     # 3. Format X-Axis (Every 3 months)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%Y'))

    # 4. Set Y-Axis Range to (0, 1)
    ax.set_ylim(0, 0.8)

    # --- BOLD FONT AND LABEL UPDATES ---
    ax.set_xlabel("")
    plt.xticks(rotation=45, fontsize=30, fontweight='bold')
    plt.yticks(fontsize=36, fontweight='bold')

    ax.legend(bbox_to_anchor=(1.0, 1), loc='upper left', fontsize=20, prop={'size': 25, 'weight': 'bold'})

    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
# --- Example Usage ---
my_mapping = {
    'Competence': 'Output_gpt-5-mini-2025-08-07',
    'Reliability': 'Output_gpt-4.1-2025-04-14_few',
    'Familiarity': 'Output_gpt-4.1-2025-04-14',
    'Integrity':        'final_mixtral_output' ,
     'Transparency':     'Output_gpt-4.1-nano-2025-04-14_few',
    'Benevolence':     'final_mixtral_output'

}
plot_dimension_frequencies_over_time(data_task2, my_mapping)

plt.savefig('Dimensions_Trust_Frequency_Plot.pdf', format='pdf', bbox_inches='tight')

# 3. Show it if it wasn't already shown by the function
plt.show()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
# --- Example Usage ---
my_mapping = {
    'Incompetence': 'Output_gpt-5-mini-2025-08-07',
    'Unreliability': 'final_mixtral_output',
    'Unfamiliarity': 'Output_gpt-4.1-nano-2025-04-14',
    'Dishonesty':      'Output_gpt-5-mini-2025-08-07' ,
     'Deception':     'Output_gpt-4o-2024-11-20',
    'Opaqueness':     'Output_gpt-5-mini-2025-08-07_few',
    'Malevolence':     'Output_gpt-4.1-2025-04-14_few'
}
plot_dimension_frequencies_over_time(data_task2, my_mapping)

plt.savefig('Dimensions_Distrust_Frequency_Plot.pdf', format='pdf', bbox_inches='tight')

# 3. Show it if it wasn't already shown by the function
plt.show()

# Dimension Co-occurrence

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
dimensions_refrences= {
'Competence': 'Output_gpt-5-mini-2025-08-07',
    'Reliability': 'Output_gpt-4.1-2025-04-14_few',
    'Familiarity': 'Output_gpt-4.1-2025-04-14',
    'Integrity':        'final_mixtral_output' ,
     'Transparency':     'Output_gpt-4.1-nano-2025-04-14_few',
    'Benevolence':     'final_mixtral_output'  ,

        'Incompetence': 'Output_gpt-5-mini-2025-08-07',
    'Unreliability': 'final_mixtral_output',
    'Unfamiliarity': 'Output_gpt-4.1-nano-2025-04-14',
    'Dishonesty':      'Output_gpt-5-mini-2025-08-07' ,
     'Deception':     'Output_gpt-4o-2024-11-20',
    'Opaqueness':     'Output_gpt-5-mini-2025-08-07_few',
    'Malevolence':     'Output_gpt-4.1-2025-04-14_few'
}

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
import pandas as pd
from itertools import combinations

# 1. Initialize a binary dataframe based on your dictionary
# Each column will be a dimension, and rows will be 1 if present in its assigned column
binary_matrix = pd.DataFrame(index=data_task2.index)

for dim, col in dimensions_refrences.items():
    # Check if 'dim' exists within the string/list of its assigned column
    # We use .fillna('') to handle any potential null values in the LLM output columns
    binary_matrix[dim] = data_task2[col].fillna('').apply(lambda x: 1 if dim in str(x) else 0)

# 2. Compute co-occurrence and Jaccard pairs
results = []
dims = list(dimensions_refrences.keys())

for dim1, dim2 in combinations(dims, 2):
    # Intersection (Both present)
    intersection = ((binary_matrix[dim1] == 1) & (binary_matrix[dim2] == 1)).sum()

    # Union (Either present)
    union = ((binary_matrix[dim1] == 1) | (binary_matrix[dim2] == 1)).sum()

    # Calculate Jaccard (handle division by zero if union is 0)
    jaccard = intersection / union if union > 0 else 0

    results.append({
        'Pair': f"{dim1} & {dim2}",
        'Co-occurrence': intersection,
        'Jaccard': round(jaccard, 4)
    })

# 3. Create and show the final table
co_occurrence_df = pd.DataFrame(results).sort_values(by='Co-occurrence', ascending=False)
co_occurrence_df.head(20) # Showing top 10 most frequent pairs

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# 1. Define the groupings based on your dictionary keys
trust_dims = ['Competence', 'Reliability', 'Familiarity', 'Integrity', 'Transparency', 'Benevolence']
distrust_dims = ['Incompetence', 'Unreliability', 'Unfamiliarity', 'Dishonesty', 'Deception', 'Opaqueness', 'Malevolence']

def get_cooccurrence_df(dim_list, binary_df):
    results = []
    for dim1, dim2 in combinations(dim_list, 2):
        # Intersection & Union
        intersection = ((binary_df[dim1] == 1) & (binary_df[dim2] == 1)).sum()
        union = ((binary_df[dim1] == 1) | (binary_df[dim2] == 1)).sum()

        jaccard = intersection / union if union > 0 else 0
        results.append({
            'Pair': f"{dim1} & {dim2}",
            'Co-occurrence': intersection,
            'Jaccard': round(jaccard, 2)
        })
    return pd.DataFrame(results).sort_values(by='Co-occurrence', ascending=False).reset_index(drop=True)

# 2. Generate the two separate DataFrames
# (Assuming binary_matrix was already created in your previous step)
df_trust = get_cooccurrence_df(trust_dims, binary_matrix)
df_distrust = get_cooccurrence_df(distrust_dims, binary_matrix)

# 3. Display results
print("--- Top Trust Co-occurrences ---")
print(df_trust.head(10))

print("\n--- Top Distrust Co-occurrences ---")
print(df_distrust.head(10))

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# Initialize a dictionary to store the counts
dimension_counts = {}

for dim, col in dimensions_refrences.items():
    # Count how many times the dimension name appears in its assigned column
    # We use .str.contains to find mentions within the text (case-sensitive by default)
    count = data_task2[col].fillna('').str.contains(dim).sum()
    dimension_counts[dim] = count

# Convert to a Series or DataFrame for a cleaner look
counts_df = pd.Series(dimension_counts).sort_values(ascending=False).to_frame(name='Mention Count')
print(counts_df)

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# Initialize a dictionary to store the row-level counts
dimension_counts = {}

for dim, col in dimensions_refrences.items():
    # 1. Use word boundaries (\b) to ensure 'unfamiliarity' doesn't count as 'familiarity'
    # 2. Use case=False to catch both 'Familiarity' and 'familiarity'
    # 3. .sum() on a boolean series counts each row only once (Binary Count)
    pattern = rf'\b{dim}\b'

    # We use na=False to treat NaNs as False
    count = data_task2[col].str.contains(pattern, case=False, regex=True, na=False).sum()

    dimension_counts[dim] = count

# Convert to a DataFrame for a cleaner look
counts_df = pd.Series(dimension_counts).sort_values(ascending=False).to_frame(name='Row Count (Exact Match)')
print(counts_df)

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# Initialize a dictionary to store the row-level counts
dimension_counts = {}

for dim, col in dimensions_refrences.items():
    # 1. Use word boundaries (\b) to ensure 'unfamiliarity' doesn't count as 'familiarity'
    # 2. Use case=False to catch 'Familiarity' and 'familiarity'
    # 3. .sum() on a boolean series counts 'True' as 1 and 'False' as 0
    pattern = rf'\b{dim}\b'
    count = data_task2[col].fillna('').str.contains(pattern, case=False, regex=True).sum()

    dimension_counts[dim] = count

# Convert to a DataFrame for a cleaner look
counts_df = pd.Series(dimension_counts).sort_values(ascending=False).to_frame(name='Row Count (Binary)')
print(counts_df)

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
col = dimensions_refrences['Integrity']
pattern = r'\bIntegrity\b'

# Find rows the NEW code catches but the OLD code missed
new_hits = data_task2[col].str.contains(pattern, case=False, regex=True, na=False)
old_hits = data_task2[col].fillna('').str.contains('Integrity', case=False)

discrepancy = data_task2[new_hits & ~old_hits]

print(f"Found {len(discrepancy)} rows that were hidden before.")
print("Sample of the text in those rows:")
print(discrepancy[col].head())

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import matplotlib.pyplot as plt

# 1. Prepare the date column (assuming 'created_utc' or 'date')
if 'date' not in data_task2.columns:
    data_task2['date'] = pd.to_datetime(data_task2['created_utc'], unit='s')

# Create a month-year period column for grouping
data_task2['month'] = data_task2['date'].dt.to_period('M')

# 2. Get the total number of rows per month (our denominator)
monthly_totals = data_task2.groupby('month').size()

plt.figure(figsize=(14, 7))

# 3. Iterate through dimensions and plot normalized frequencies
for dim, col in dimensions_refrences.items():
    # We use the regex word boundary \b to ensure 'Familiarity'
    # doesn't pick up 'Unfamiliarity'
    pattern = rf'\b{dim}\b'

    # Check for presence (True/False) per row
    has_dim = data_task2[col].fillna('').str.contains(pattern, case=False, regex=True)

    # Sum the 'True' values per month
    monthly_hits = data_task2[has_dim].groupby('month').size()

    # Normalize: Hits / Total Rows in that month
    normalized_freq = (monthly_hits / monthly_totals).fillna(0)

    # Convert index to string for plotting
    plt.plot(normalized_freq.index.astype(str), normalized_freq.values, label=dim, linewidth=2)

plt.title('Normalized Frequency of Dimensions (LLM Detection Only)', fontsize=14)
plt.ylabel('Proportion of Monthly Posts', fontsize=12)
plt.xlabel('Month', fontsize=12)
plt.xticks(rotation=45)
plt.legend(title="Dimensions", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.show()

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
# Check for multiple mentions in a single cell
fam_col = dimensions_refrences['Familiarity']
rel_col = dimensions_refrences['Reliability']

print(f"Total rows in data_task2: {len(data_task2)}")
print("-" * 30)

# Rows where 'Familiarity' appears more than once
multi_fam = data_task2[fam_col].fillna('').str.lower().str.count('familiarity').value_counts().sort_index()
print("Familiarity counts per row:")
print(multi_fam)

print("-" * 30)

# Rows where 'Reliability' appears more than once
multi_rel = data_task2[rel_col].fillna('').str.lower().str.count('reliability').value_counts().sort_index()
print("Reliability counts per row:")
print(multi_rel)

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# Identify the column
fam_col = dimensions_refrences['Familiarity']

# Filter for rows where 'familiarity' (case-insensitive) appears exactly twice
double_mentions = data_task2[data_task2[fam_col].fillna('').str.lower().str.count('familiarity') == 2]

# Display the first example's text
if not double_mentions.empty:
    print(double_mentions[fam_col].iloc[0])
else:
    print("No rows found with exactly two mentions.")

## 8. Reason and trustor extraction

Extracts reasons for trust/distrust and trustor categories, aligns batch outputs, cleans labels, and visualizes relationships between reasons, trustors, and Task 1 labels.

# Reason & Trustor

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
test_df_exp= test_df[test_df['majority_Label'] != "Neither"]
len(test_df_exp)

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
prompt_reason=  """

Instruction:
You are an assitant in detecting the reasons for trust or distrust towards Generative AI or Large Language Models mentioned in the text.
We define trust as a belief in the reliability, competence, or integrity of GenAI technologies / responsible companies, leading to positive expectations about their performance and ethical behavior. On the other hand, we define distrust as skepticism or concern about the reliability, competence, or ethical implications of GenAI / responsible companies, leading to negative expectations or cautious behavior toward these technologies or companies.

You will identify the reason for trust or distrust in each text from the following options:


Personal experience using AI models

Reports from media or experts

Discussions on social media or online forums

General perception of AI technology

General perception toward responsible companies

Experiences from friends, families, colleagues, etc. who used it

Other

Follow this format for the output:
[Label]

**Do not add any extra text

Here is the text:
"""

In [ ]:
# Purpose: Configure API clients and run LLM inference. Requires the relevant API key in environment variables.
import time
from tqdm import tqdm
from openai import OpenAI

# 1. Define your list of models
models_to_run = ['gpt-5.2-2025-12-11', 'gpt-5.1-2025-11-13', 'gpt-5-2025-08-07', 'gpt-5-mini-2025-08-07', 'gpt-5-nano-2025-08-07', 'gpt-4o-2024-11-20', 'gpt-4o-mini-2024-07-18', 'gpt-4.1-2025-04-14', 'gpt-4.1-mini-2025-04-14', 'gpt-4.1-nano-2025-04-14']

def classify_reason(text, model_name):
    retries = 2
    while retries > 0:
        try:
            # Try with temperature=0.0 first
            try:
                completion = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt_reason + text}],
                    temperature=0.0
                )
            except Exception:
                # Silently retry without temperature if the first attempt fails
                completion = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt_reason + text}]
                )

            return completion.choices[0].message.content

        except Exception as e:
            print(f"Error with {model_name}: {e}. Retrying...")
            retries -= 1
            time.sleep(5)

    return "API_ERROR"

# 2. Loop through each model and create the new columns
tqdm.pandas()

for model in models_to_run:
    print(f"Starting classification for model: {model}")
    column_name = f"LLM_{model}"

    # Apply to test_df['text']
    test_df_exp[column_name] = test_df_exp['text'].progress_apply(
        lambda x: classify_reason(x, model)
    )

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
prompt_trustor=  """

Instruction:
You are an assitant in detecting the trustor of trust or distrust towards Generative AI or Large Language Models mentioned in the text.
We define trust as a belief in the reliability, competence, or integrity of GenAI technologies / responsible companies, leading to positive expectations about their performance and ethical behavior. On the other hand, we define distrust as skepticism or concern about the reliability, competence, or ethical implications of GenAI / responsible companies, leading to negative expectations or cautious behavior toward these technologies or companies.

Trustor: The entity expressing trust or distrust. Trustors could be users of GenAI, software developers, or others.
You will identify the trustor of trust or distrust in each text from the following options:


Generative AI users: People who actively use AI models like ChatGPT

Software developers: Developers or engineers working with AI technology

Researchers or academics: Experts studying AI models, ethics, or machine learning

Tech industry professionals: People working in AI-adjacent roles, but not directly developing AI

General public: Individuals with no technical background but expressing opinions on AI

Media and journalists: Writers or influencers discussing AI in news or blogs

Business leaders or executives: CEOs, managers, or decision-makers in companies that use or invest in AI

AI ethicists or advocacy groups: People focused on the ethical and societal impact of AI, including nonprofits and watchdog organizations

Artists or creatives: Writers, musicians, designers, or other creatives affected by or engaging with AI-generated content

Educators and knowledge workers: People who deal with students and work in schools or higher education

Other

Follow this format for the output:
[Label]

**Do not add any extra text

Here is the text:
"""

In [ ]:
# Purpose: Configure API clients and run LLM inference. Requires the relevant API key in environment variables.
import time
from tqdm import tqdm
from openai import OpenAI

# 1. Define your list of models
models_to_run = ['gpt-5.2-2025-12-11', 'gpt-5.1-2025-11-13', 'gpt-5-2025-08-07', 'gpt-5-mini-2025-08-07', 'gpt-5-nano-2025-08-07', 'gpt-4o-2024-11-20', 'gpt-4o-mini-2024-07-18', 'gpt-4.1-2025-04-14', 'gpt-4.1-mini-2025-04-14', 'gpt-4.1-nano-2025-04-14']

def classify_trustor(text, model_name):
    retries = 2
    while retries > 0:
        try:
            # Try with temperature=0.0 first
            try:
                completion = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt_trustor + text}],
                    temperature=0.0
                )
            except Exception:
                # Silently retry without temperature if the first attempt fails
                completion = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt_trustor + text}]
                )

            return completion.choices[0].message.content

        except Exception as e:
            print(f"Error with {model_name}: {e}. Retrying...")
            retries -= 1
            time.sleep(5)

    return "API_ERROR"

# 2. Loop through each model and create the new columns
tqdm.pandas()

for model in models_to_run:
    print(f"Starting classification for model: {model}")
    column_name = f"LLM_{model}_trustor"

    # Apply to test_df['text']
    test_df_exp[column_name] = test_df_exp['text'].progress_apply(
        lambda x: classify_trustor(x, model)
    )

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
trustor_reason_data =test_df_exp.copy()

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
# 1. Group and get the mode
reason_data = df_5.groupby('text')['Reason'].agg(lambda x: x.mode().tolist()).reset_index()

# 2. Filter for rows where the list has exactly one item
reason_data = reason_data[reason_data['Reason'].map(len) == 1]

# 3. Clean up: Convert the 1-item list back to a simple value
reason_data['Reason'] = reason_data['Reason'].str[0]

# 4. Remove the '0' values
reason_data = reason_data[reason_data['Reason'] != '0']

reason_data.head()

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
def get_agreement_rate(group):
    # Count occurrences of each reason for this text
    counts = group.value_counts()
    if counts.empty:
        return 0
    # Agreement = (Frequency of the most common reason) / (Total number of reasons)
    return counts.iloc[0] / len(group)

# 1. Filter out '0' or irrelevant values if necessary (consistent with your cell [226])
df_filtered = df_5[df_5['Reason'] != '0']

# 2. Group by text and calculate the agreement rate per text
agreement_rates = df_filtered.groupby('text')['Reason'].apply(get_agreement_rate)

# 3. Compute the overall average agreement rate
average_agreement = agreement_rates.mean()

print(f"Average Agreement Rate for Reasons: {average_agreement:.2%}")

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
def get_agreement_rate(group):
    counts = group.value_counts()
    if counts.empty:
        return 0
    return counts.iloc[0] / len(group)

# 1. Filter df_5 to only include texts present in trustor_reason_data
# and remove '0' values
valid_texts = trustor_reason_data['text'].unique()
df_filtered = df_5[
    (df_5['text'].isin(valid_texts)) &
    (df_5['Reason'] != '0')
]

# 2. Group by text and calculate the agreement rate per text
agreement_rates = df_filtered.groupby('text')['Reason'].apply(get_agreement_rate)

# 3. Compute the overall average agreement rate
average_agreement = agreement_rates.mean()

print(f"Average Agreement Rate for Reasons (filtered): {average_agreement:.2%}")

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
# 1. Group and get the mode
trustor_data = df_5.groupby('text')['Trustor'].agg(lambda x: x.mode().tolist()).reset_index()

# 2. Filter for rows where the list has exactly one item
trustor_data = trustor_data[trustor_data['Trustor'].map(len) == 1]

# 3. Clean up: Convert the 1-item list back to a simple value
trustor_data['Trustor'] = trustor_data['Trustor'].str[0]

# 4. Remove the '0' values
trustor_data = trustor_data[trustor_data['Trustor'] != '0']

trustor_data.head()

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
def get_agreement_rate(group):
    # Count occurrences of each reason for this text
    counts = group.value_counts()
    if counts.empty:
        return 0
    # Agreement = (Frequency of the most common reason) / (Total number of reasons)
    return counts.iloc[0] / len(group)

# 1. Filter out '0' or irrelevant values if necessary (consistent with your cell [226])
df_filtered = df_5[df_5['Trustor'] != '0']

# 2. Group by text and calculate the agreement rate per text
agreement_rates = df_filtered.groupby('text')['Trustor'].apply(get_agreement_rate)

# 3. Compute the overall average agreement rate
average_agreement = agreement_rates.mean()

print(f"Average Agreement Rate for Trustor: {average_agreement:.2%}")

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
def get_agreement_rate(group):
    counts = group.value_counts()
    if counts.empty:
        return 0
    return counts.iloc[0] / len(group)

# 1. Filter df_5 to only include texts present in trustor_reason_data
# and remove '0' values
valid_texts = trustor_reason_data['text'].unique()
df_filtered = df_5[
    (df_5['text'].isin(valid_texts)) &
    (df_5['Trustor'] != '0')
]

# 2. Group by text and calculate the agreement rate per text
agreement_rates = df_filtered.groupby('text')['Trustor'].apply(get_agreement_rate)

# 3. Compute the overall average agreement rate
average_agreement = agreement_rates.mean()

print(f"Average Agreement Rate for Trustor (filtered): {average_agreement:.2%}")

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# 1. Merge the Reason data
trustor_reason_data = trustor_reason_data.merge(
    reason_data[['text', 'Reason']],
    on='text',
    how='left'
)

# 2. Merge the Trustor data
trustor_reason_data = trustor_reason_data.merge(
    trustor_data[['text', 'Trustor']],
    on='text',
    how='left'
)

# 3. Fill missing matches with 'not present'
trustor_reason_data['Reason'] = trustor_reason_data['Reason'].fillna('not present')
trustor_reason_data['Trustor'] = trustor_reason_data['Trustor'].fillna('not present')

# Check the result
trustor_reason_data.head()

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# 1. Define the list of columns from cell [231]
llm_models = [
    'LLM_gpt-5.2-2025-12-11', 'LLM_gpt-5.1-2025-11-13', 'LLM_gpt-5-2025-08-07',
    'LLM_gpt-5-mini-2025-08-07', 'LLM_gpt-5-nano-2025-08-07', 'LLM_gpt-4o-2024-11-20',
    'LLM_gpt-4o-mini-2024-07-18', 'LLM_gpt-4.1-2025-04-14', 'LLM_gpt-4.1-mini-2025-04-14',
    'LLM_gpt-4.1-nano-2025-04-14'
]

# 2. Create a mapping dictionary for the rename function
rename_dict = {col: f"{col}_reason" for col in llm_models}

# 3. Rename the columns in your main dataframe
trustor_reason_data = trustor_reason_data.rename(columns=rename_dict)

# Check the new column names
trustor_reason_data.columns.tolist()

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# 1. Identify all columns that start with 'LLM_'
llm_cols = [col for col in trustor_reason_data.columns if col.startswith('LLM_')]

# 2. Loop through and remove brackets using string replacement
for col in llm_cols:
    trustor_reason_data[col] = (
        trustor_reason_data[col]
        .astype(str)
        .str.replace('[', '', regex=False)
        .str.replace(']', '', regex=False)
    )

# Check the cleaned result
trustor_reason_data.head()

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
trustor_model= 'gpt-5.1-2025-11-13'
reason_model= 'gpt-5-mini-2025-08-07'

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
len(batches)

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json
import math
import re


# ---- 1) Constants you define once ----
MODEL = trustor_model                # or any chat model you want
SYSTEM_PROMPT = prompt_trustor  # your constant prompt
TEXT_COL = "all_text"

# (optional) basic cleaner so inputs won't break batching
RESERVED = re.compile(r"<\|[^>]+?\|>")  # catches tokens like <|endoftext|>
def sanitize(s):
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = str(s).replace("\r\n", "\n").strip()
    s = RESERVED.sub("[RESERVED_TOKEN]", s)
    return s[:8000]  # keep inputs reasonable

# ---- 2) Writer: one JSON object per line ----
def write_ndjson_from_df_trustor(df, custom_id_col=None, max_tokens=2000):
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        for i, row in df.reset_index(drop=True).iterrows():
            custom_id = (
                str(row[custom_id_col])
                if custom_id_col and custom_id_col in df.columns
                else f"row-{i+1:06d}"
            )
            user_text = sanitize(row[TEXT_COL])

            obj = {
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": MODEL,
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": user_text}
                    ],
                    # "max_completion_tokens": max_tokens,
                    # "temperature": temperature
                }
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import os

# 1. Create a directory for the trustor batch files
output_dir = "batch_files_trustor"
os.makedirs(output_dir, exist_ok=True)

generated_trustor_files = []

# 2. Iterate through the batches and generate .jsonl files
for i, batch_df in enumerate(batches):
    # Clean model name for the filename
    model_safe_name = MODEL.replace("/", "_").replace(":", "_")

    # Define the output path for this specific batch
    OUT_PATH = f"{output_dir}/batch_{model_safe_name}_part_{i + 1}.jsonl"

    # Call your function (assuming OUT_PATH is handled globally or added as an arg)
    # If you haven't modified the function signature, it uses the global OUT_PATH
    write_ndjson_from_df_trustor(batch_df, custom_id_col=None)

    generated_trustor_files.append(OUT_PATH)
    print(f"Generated Trustor Batch: {OUT_PATH}")

print(f"\nSuccessfully generated {len(generated_trustor_files)} files for processing.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
batch_jobs = []

print(f"🚀 Starting upload and submission for {len(generated_trustor_files)} files...\n")

for file_path in generated_trustor_files:
    # 1. Upload the file
    print(f"Uploading: {file_path}")
    with open(file_path, "rb") as f:
        uploaded_file = client.files.create(
            file=f,
            purpose="batch"
        )

    # 2. Create the Batch Job
    # Using the endpoint '/v1/chat/completions' as defined in your writer function
    batch_job = client.batches.create(
        input_file_id=uploaded_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h", # Standard window for 50% discount
        metadata={
            "description": f"Trustor processing: {file_path}",
            "model": MODEL
        }
    )

    batch_jobs.append(batch_job)
    print(f"   ✅ Submitted! Batch ID: {batch_job.id}")

print(f"\nSuccessfully submitted {len(batch_jobs)} batch jobs.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json
import math
import re


# ---- 1) Constants you define once ----
MODEL = reason_model                # or any chat model you want
SYSTEM_PROMPT = prompt_reason  # your constant prompt
TEXT_COL = "all_text"

# (optional) basic cleaner so inputs won't break batching
RESERVED = re.compile(r"<\|[^>]+?\|>")  # catches tokens like <|endoftext|>
def sanitize(s):
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = str(s).replace("\r\n", "\n").strip()
    s = RESERVED.sub("[RESERVED_TOKEN]", s)
    return s[:8000]  # keep inputs reasonable

# ---- 2) Writer: one JSON object per line ----
def write_ndjson_from_df_reason(df, custom_id_col=None, max_tokens=2000):
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        for i, row in df.reset_index(drop=True).iterrows():
            custom_id = (
                str(row[custom_id_col])
                if custom_id_col and custom_id_col in df.columns
                else f"row-{i+1:06d}"
            )
            user_text = sanitize(row[TEXT_COL])

            obj = {
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": MODEL,
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": user_text}
                    ],
                    # "max_completion_tokens": max_tokens,
                    # "temperature": temperature
                }
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import os

# 1. Create a directory for the trustor batch files
output_dir = "batch_files_reason"
os.makedirs(output_dir, exist_ok=True)

generated_reason_files = []

# 2. Iterate through the batches and generate .jsonl files
for i, batch_df in enumerate(batches):
    # Clean model name for the filename
    model_safe_name = MODEL.replace("/", "_").replace(":", "_")

    # Define the output path for this specific batch
    OUT_PATH = f"{output_dir}/batch_{model_safe_name}_part_{i + 1}.jsonl"

    # Call your function (assuming OUT_PATH is handled globally or added as an arg)
    # If you haven't modified the function signature, it uses the global OUT_PATH
    write_ndjson_from_df_reason(batch_df, custom_id_col=None)

    generated_reason_files.append(OUT_PATH)
    print(f"Generated reason Batch: {OUT_PATH}")

print(f"\nSuccessfully generated {len(generated_reason_files)} files for processing.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
batch_jobs_reason = []

print(f"🚀 Starting upload and submission for {len(generated_reason_files)} files...\n")

for file_path in generated_reason_files:
    # 1. Upload the file
    print(f"Uploading: {file_path}")
    with open(file_path, "rb") as f:
        uploaded_file = client.files.create(
            file=f,
            purpose="batch"
        )

    # 2. Create the Batch Job
    # Using the endpoint '/v1/chat/completions' as defined in your writer function
    batch_job = client.batches.create(
        input_file_id=uploaded_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h", # Standard window for 50% discount
        metadata={
            "description": f"Trustor processing: {file_path}",
            "model": MODEL
        }
    )

    batch_jobs_reason.append(batch_job)
    print(f"   ✅ Submitted! Batch ID: {batch_job.id}")

print(f"\nSuccessfully submitted {len(batch_jobs_reason)} batch jobs.")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
def print_batch_progress(jobs_list, label):
    print(f"--- {label} Progress ---")
    for job in jobs_list:
        # Retrieve the latest status from OpenAI
        status = client.batches.retrieve(job.id)

        total = status.request_counts.total
        completed = status.request_counts.completed
        failed = status.request_counts.failed

        # Calculate percentage
        if total > 0:
            percent = (completed / total) * 100
        else:
            percent = 0

        status_msg = f"ID: {job.id} | {status.status.upper():<12} | {percent:>6.1f}% complete"
        status_msg += f" ({completed}/{total} requests, {failed} failed)"

        print(status_msg)
    print("\n")

# Run for both sets of batches
print_batch_progress(batch_jobs, "Trustor Batches")
print_batch_progress(batch_jobs_reason, "Reason Batches")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json

# This will store all outputs across all 14 batches
all_trustor_outputs = []

print(f"--- Downloading Results for {len(batch_jobs)} Trustor Batches ---")

for job in batch_jobs:
    # 1. Refresh job status to get the output_file_id
    current_job = client.batches.retrieve(job.id)
    file_id = current_job.output_file_id

    if not file_id:
        print(f"Skipping {job.id}: No output file found (Status: {current_job.status})")
        continue

    # 2. Download the file content
    file_response = client.files.content(file_id)
    # The response content is typically bytes; decode to string and split by lines
    file_contents = file_response.text.strip().split('\n')

    # 3. Parse and store results for THIS specific batch
    batch_results = []
    for line in file_contents:
        if line:
            batch_results.append(json.loads(line))

    # 4. Verification and Append
    # Sort batch_results by custom_id if your writer function included them
    # batch_results.sort(key=lambda x: int(x['custom_id'].split('_')[-1]))

    print(f"Batch {job.id}: Downloaded {len(batch_results)} records.")

    # Append this batch's list to our final master list
    all_trustor_outputs.extend(batch_results)

print(f"\n✅ Done! Total outputs collected: {len(all_trustor_outputs)}")

In [ ]:
# Purpose: Prepare, submit, retrieve, or align batch-inference files for large-scale LLM labeling.
import json

# This will store all outputs across all 14 Reason batches
all_reason_outputs = []

print(f"--- Downloading Results for {len(batch_jobs_reason)} Reason Batches ---")

for job in batch_jobs_reason:
    # 1. Refresh job status to get the output_file_id
    current_job = client.batches.retrieve(job.id)
    file_id = current_job.output_file_id

    if not file_id:
        print(f"Skipping {job.id}: No output file found (Status: {current_job.status})")
        continue

    # 2. Download the file content
    file_response = client.files.content(file_id)
    file_contents = file_response.text.strip().split('\n')

    # 3. Parse into a temporary list for this specific batch
    batch_results = []
    for line in file_contents:
        if line:
            batch_results.append(json.loads(line))

    # 4. Sort internally by custom_id to fix potential shuffling
    # This assumes custom_id follows a sortable pattern like 'reason_0001'
    try:
        batch_results.sort(key=lambda x: x['custom_id'])
    except KeyError:
        print(f"Warning: No custom_id found in batch {job.id}. Order may be randomized.")

    print(f"Batch {job.id}: Stored {len(batch_results)} records in list.")

    # 5. Append the verified batch list to the final master list
    all_reason_outputs.extend(batch_results)

print(f"\n✅ Done! Total Reason outputs collected: {len(all_reason_outputs)}")

In [ ]:
# Purpose: Configure API clients and run LLM inference. Requires the relevant API key in environment variables.
# 1. Function to safely extract content or return a placeholder on error
def extract_batch_content(results_list):
    extracted = []
    for item in results_list:
        try:
            # Navigate the OpenAI Batch response structure
            content = item['response']['body']['choices'][0]['message']['content']
            extracted.append(content)
        except (KeyError, TypeError, IndexError):
            # Handle cases where the request might have failed or is malformed
            extracted.append("ERROR_OR_EMPTY")
    return extracted

# 2. Process both lists
trustor_text_list = extract_batch_content(all_trustor_outputs)
reason_text_list = extract_batch_content(all_reason_outputs)

# 3. Add to your dataframe
# This assumes the sorting in the previous step matched your df index
data_task2['Trustor_Final'] = trustor_text_list
data_task2['Reason_Final'] = reason_text_list

print(f"Successfully extracted {len(trustor_text_list)} Trustor and {len(reason_text_list)} Reason results.")

In [ ]:
# Purpose: Run an intermediate preprocessing, inspection, or analysis step used by nearby cells.
# 1. Strip the brackets from the string content in the new columns
data_task2['Trustor_Final'] = data_task2['Trustor_Final'].str.strip('[]')
data_task2['Reason_Final'] = data_task2['Reason_Final'].str.strip('[]')

# 2. Remove the raw prediction dictionary columns
# These were the columns containing the full batch request metadata
cols_to_remove = ['Reason_Prediction', 'Trustor_Prediction']
data_task2.drop(columns=cols_to_remove, inplace=True)

# 3. Preview the clean result
print("Columns removed and strings cleaned.")
data_task2.head()

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
# 1. Take only the part before the first newline
# 2. Strip any remaining brackets and whitespace
data_task2['Reason_Final'] = data_task2['Reason_Final'].apply(
    lambda x: str(x).split('\n')[0].strip('[] ').strip()
)
# Define the mapping for consolidation
rename_map = {
    'Experiences from friends, families, colleagues, etc. who used it': 'Experiences from friends, families, colleagues',
    'Experiences from friends, families, colleagues, etc.': 'Experiences from friends, families, colleagues'
}

# Apply the replacement
data_task2['Reason_Final'] = data_task2['Reason_Final'].replace(rename_map)

# 1. Identify valid categories (those with count > 1)
counts = data_task2['Reason_Final'].value_counts()
valid_categories = counts[counts > 1].index

# 2. Filter the dataframe to keep only rows with these valid categories
data_task2 = data_task2[data_task2['Reason_Final'].isin(valid_categories)]


# 3. Check the normalized counts
print(data_task2['Reason_Final'].value_counts())

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['font.family'] = 'Times New Roman'

# Crosstab: rows = label (bars), columns = reason (stack segments)
counts = pd.crosstab(
    data_task2['LLM_Predictions'],
    data_task2['Reason_Final']
)

# ---- Order reasons by their frequency within the "Trust" group ----
if "Trust" in counts.index:
    # Smallest -> largest so the largest is PLOTTED LAST (i.e., appears ON TOP)
    reason_order_small2large = counts.loc["Trust"].sort_values(ascending=True).index
else:
    # Fallback: global ordering if "Trust" missing
    reason_order_small2large = counts.sum(axis=0).sort_values(ascending=True).index

counts = counts[reason_order_small2large]

# ---- Order the bars (x-axis groups) by total frequency (descending) ----
bar_order = counts.sum(axis=1).sort_values(ascending=False).index
counts = counts.loc[bar_order]

# ---- Normalize by row (stacked to 1.0) ----
proportions = counts.div(counts.sum(axis=1), axis=0)

# ---- Plot ----
ax = proportions.plot(
    kind="bar",
    stacked=True,
    figsize=(16, 12),
    colormap="tab20"
)

ax.set_xlabel("")

# # Make ticks bold
# # for label in ax.get_xticklabels():
# #     label.set_fontweight('bold')
# for label in ax.get_yticklabels():
#     # label.set_fontweight('bold')
#     label.set_fontsize(36)

# # Make y-ticks larger
# for label in ax.get_yticklabels():
#     label.set_fontsize(36)

plt.xticks(rotation=45, ha="right")

# ---- Legend order: largest Trust reason FIRST (top of legend) ----
# Build a descending order (largest -> smallest) from the Trust row
if "Trust" in counts.index:
    legend_desc = counts.loc["Trust"].sort_values(ascending=False).index.tolist()
else:
    legend_desc = counts.sum(axis=0).sort_values(ascending=False).index.tolist()

# Get handles/labels from the plot (currently in column order = small->large)
handles, labels = ax.get_legend_handles_labels()
handle_map = {lab: h for h, lab in zip(handles, labels)}

# Rebuild legend in desired order (largest first)
ordered_handles = [handle_map[lab] for lab in legend_desc if lab in handle_map]
ordered_labels  = [lab for lab in legend_desc if lab in handle_map]

plt.xticks(rotation=45, ha="right")

# Make x-ticks larger
for label in ax.get_xticklabels():
    label.set_fontsize(45)
    # label.set_fontweight('bold')

# Make y-ticks larger
for label in ax.get_yticklabels():
    label.set_fontsize(45)
    # label.set_fontweight('bold')

ax.legend(
    ordered_handles, ordered_labels,
    #alltitle="Reason",
    bbox_to_anchor=(0.25, 0.8),
    fontsize=30,
    #title_fontsize=24,
    loc="upper left",
    ncol=1
)

plt.tight_layout()
plt.savefig("ACL_Fig4_Updated.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
# Define the mapping to one standard name
researcher_map = {
    'AI researchers or academics': 'Researchers or academics',
    'Researcher or academics': 'Researchers or academics',
    'Researchers or academics': 'Researchers or academics',
}

# Apply the replacement to the Trustor_Final column
data_task2['Trustor_Final'] = data_task2['Trustor_Final'].replace(researcher_map)

# Verify the merge
print("Updated Trustor_Final counts:")
print(data_task2['Trustor_Final'].value_counts())

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import rcParams

rcParams['font.family'] = 'Times New Roman'

# 1. Create the crosstab
counts = pd.crosstab(data_task2['Trustor_Final'], data_task2['LLM_Predictions'])

# 2. SORTING LOGIC: Sort the index based on the total row sums (overall count per category)
counts = counts.loc[counts.sum(axis=1).sort_values(ascending=True).index]

# 3. Normalize for the stacked percentage view
proportions = counts.div(counts.sum(axis=1), axis=0)

# 4. Plot
ax = proportions.plot(
    kind="barh",
    stacked=True,
    figsize=(20, 10),
    colormap="tab20",
    edgecolor='white'
)

# Formatting
ax.set_ylabel("")
ax.set_xlabel("")

for label in ax.get_xticklabels():
    label.set_fontsize(36)
for label in ax.get_yticklabels():
    label.set_fontweight('bold')
    label.set_fontsize(34) # Slightly larger for the new layout

# Legend
plt.legend( bbox_to_anchor=(0.98, 1), loc='upper left', fontsize=36)

plt.tight_layout()
plt.savefig("ACL_Fig5_Sorted.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import rcParams

rcParams['font.family'] = 'Times New Roman'

# 1. Create the crosstab
counts = pd.crosstab(data_task2['Trustor_Final'], data_task2['LLM_Predictions'])

# 2. Sort based on overall count (Descending for vertical bars)
counts = counts.loc[counts.sum(axis=1).sort_values(ascending=False).index]

# 3. Normalize
proportions = counts.div(counts.sum(axis=1), axis=0)

# 4. Plot as vertical bars (kind="bar")
ax = proportions.plot(
    kind="bar",
    stacked=True,
    figsize=(22, 10),
    colormap="tab20",
    edgecolor='white'
)

# Formatting
ax.set_ylabel("Proportion", fontsize=32, fontweight='bold')
ax.set_xlabel("")
ax.set_ylabel("")

# X-axis: Rotating labels for readability since they are now on the bottom
for label in ax.get_xticklabels():
    label.set_rotation(45)
    label.set_ha('right')
    label.set_fontweight('bold')
    label.set_fontsize(32)

# Y-axis
for label in ax.get_yticklabels():
    label.set_fontsize(36)

# Legend
plt.legend(bbox_to_anchor=(1, 1), loc='upper left', fontsize=36)

plt.tight_layout()
plt.savefig("ACL_Fig5_Vertical.pdf", dpi=300, bbox_inches="tight")
plt.show()

## 9. TACL editorial / revision experiments

Runs later additional model comparisons requested during revision, including Claude/GPT reruns, confusion matrices, and misclassification inspection.

# Try New Models for TACL Editorial

In [ ]:
# Purpose: Define the Task 1 prompt for classifying trust, distrust, both, or neither.
import time
from tqdm import tqdm

# 1. Variables for a single run
model_name = "claude-haiku-4-5"
prompt_to_use = task_1_prompt_few_shot  # Or task_1_prompt_few_shot
column_name = "LLM_claude-haiku_Few_Shot"

# 2. Function structure (Assuming you are using the Together AI client)
def classify_Claude(text, Model, Prompt):
    message = client.messages.create(
    model= Model,
    max_tokens=2048,
    # Update the thinking block here:
    messages=[
        {"role": "user", "content": Prompt + text}
    ]
)
    return message.content[0].text

tqdm.pandas()

# 3. Single execution with error handling
print(f"Processing: {column_name}")

def safe_classify(row):
    try:
        # Pass the specific text, model, and prompt directly
        result = classify_Claude(row['text'], model_name, prompt_to_use)

        return result
    except Exception as e:
        # Log the error so you know why it failed (e.g., model_not_available)
        return f"ERROR: {str(e)}"

# Apply to your dataframe
test_df[column_name] = test_df.progress_apply(safe_classify, axis=1)

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
# 1. Remove [ and ] and strip any extra whitespace
test_df['LLM_claude-haiku_Few_Shot'] = test_df['LLM_claude-haiku_Few_Shot'].astype(str).str.replace(r'[\[\]]', '', regex=True).str.strip()

# 2. Check the updated counts
print(test_df['LLM_claude-haiku_Few_Shot'].value_counts())

In [ ]:
# Purpose: Define the Task 1 prompt for classifying trust, distrust, both, or neither.
import time
from tqdm import tqdm

# 1. Variables for a single run
model_name = "claude-sonnet-4-6"
prompt_to_use = task_1_prompt_few_shot  # Or task_1_prompt_few_shot
column_name = "LLM_claude-sonnet_Few_Shot"

# 2. Function structure (Assuming you are using the Together AI client)
def classify_Claude(text, Model, Prompt):
    message = client.messages.create(
    model= Model,
    max_tokens=2048,
    # Update the thinking block here:
    messages=[
        {"role": "user", "content": Prompt + text}
    ]
)
    return message.content[0].text

tqdm.pandas()

# 3. Single execution with error handling
print(f"Processing: {column_name}")

def safe_classify(row):
    try:
        # Pass the specific text, model, and prompt directly
        result = classify_Claude(row['text'], model_name, prompt_to_use)

        return result
    except Exception as e:
        # Log the error so you know why it failed (e.g., model_not_available)
        return f"ERROR: {str(e)}"

# Apply to your dataframe
test_df[column_name] = test_df.progress_apply(safe_classify, axis=1)

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
# 1. Remove [ and ] and strip any extra whitespace
test_df['LLM_claude-sonnet_Few_Shot'] = test_df['LLM_claude-sonnet_Few_Shot'].astype(str).str.replace(r'[\[\]]', '', regex=True).str.strip()

# 2. Check the updated counts
print(test_df['LLM_claude-sonnet_Few_Shot'].value_counts())

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print(f"{'Model Column':<20} | {'Weighted F1 Score':<18}")
print("-" * 42)

for col in ['LLM_claude-sonnet_Few_Shot', 'LLM_claude-haiku_Few_Shot']:
    # Ensure data is clean
    y_pred = test_df[col].astype(str).str.strip()
    y_true = test_df['majority_Label'].astype(str).str.strip()

    # Calculate weighted F1
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # Print the result
    print(f"{col:<20} | {f1:.4f}")

In [ ]:
# Purpose: Define the Task 1 prompt for classifying trust, distrust, both, or neither.
import time
from tqdm import tqdm

# 1. Configuration
models = ["claude-sonnet-4-6", "claude-haiku-4-5"]
prompts = {
    "Zero_Shot": task_1_prompt,
    "Few_Shot": task_1_prompt_few_shot
}

tqdm.pandas()

# 2. Updated classification function
def run_classification(text, model_id, prompt_text):
    try:
        message = client.messages.create(
            model=model_id,
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt_text + text}]
        )
        return message.content[0].text
    except Exception as e:
        return f"ERROR: {str(e)}"

# 3. The Nested Loop (Creates 4 separate columns)
for model_id in models:
    # Simplify name for column (e.g., 'claude-sonnet')
    short_name = "claude-sonnet" if "sonnet" in model_id else "claude-haiku"

    for prompt_type, prompt_content in prompts.items():
        # Construct unique column name
        col_suffix = "" if prompt_type == "Zero_Shot" else "_Few_Shot"
        column_name = f"LLM_{short_name}{col_suffix}"

        print(f"--- Processing: {column_name} ---")

        # Apply only to this specific new column
        test_df[column_name] = test_df['text'].progress_apply(
            lambda x: run_classification(x, model_id, prompt_content)
        )

        # Immediate clean-up of brackets for this column
        test_df[column_name] = test_df[column_name].astype(str).str.replace(r'[\[\]]', '', regex=True).str.strip()

print("\nAll 4 columns created successfully!")
print(test_df.columns[-4:].tolist())

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
from sklearn.metrics import precision_score, recall_score, f1_score

# 1. Define the columns to evaluate
claude_columns = [
    'LLM_claude-sonnet',
    'LLM_claude-sonnet_Few_Shot',
    'LLM_claude-haiku',
    'LLM_claude-haiku_Few_Shot'
]

y_true = test_df['majority_Label'].astype(str).str.strip()

# 2. Print Header
print(f"{'Claude Model Variant':<30} | {'Recall':<10} | {'Precision':<10} | {'F1 Score':<10}")
print("-" * 70)

# 3. Calculate and Report Metrics
for col in claude_columns:
    y_pred = test_df[col].astype(str).str.strip()

    # Calculate weighted metrics
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f"{col:<30} | {recall:.4f}   | {precision:.4f}    | {f1:.4f}")

In [ ]:
# Purpose: Save the current processed dataframe or analysis output for reuse/reproducibility.
misclassified_data[['text', 'majority_Label', 'LLM_gpt-5.1']].to_csv('misclassified_data.csv',  encoding="utf-8-sig", index= False)

# Confusion Matrix for Zero-shot vs Few-shot

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Filter for Disagreement Instances
# Using your gt_data dataframe
disagree_df = gt_data[gt_data['LLM_gpt-5.1'] != gt_data['LLM_gpt-5.1_Few_Shot']].copy()

# 2. Check which model is correct against Ground Truth
disagree_df['Zero_Shot_Correct'] = (disagree_df['LLM_gpt-5.1'] == disagree_df['majority_Label'])
disagree_df['Few_Shot_Correct'] = (disagree_df['LLM_gpt-5.1_Few_Shot'] == disagree_df['majority_Label'])

# 3. Aggregate Accuracy by Label
# We want to see: For the instances they disagreed, how many did each get right?
comparison = disagree_df.groupby('majority_Label').agg({
    'Zero_Shot_Correct': 'sum',
    'Few_Shot_Correct': 'sum'
}).reset_index()

# 4. Melt for plotting
plot_data = comparison.melt(id_vars='majority_Label',
                            value_vars=['Zero_Shot_Correct', 'Few_Shot_Correct'],
                            var_name='Model', value_name='Correct_Count')

# 5. Visualize
plt.figure(figsize=(10, 6))
sns.barplot(data=plot_data, x='majority_Label', y='Correct_Count', hue='Model', palette='viridis')

plt.title('Performance on Disagreement Set: Which Model is Better at Breaking Ties?')
plt.ylabel('Number of Correct Predictions')
plt.xlabel('Ground Truth Label')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print(f"Total Disagreements Analyzed: {len(disagree_df)}")

In [ ]:
# Purpose: Define the Task 1 prompt for classifying trust, distrust, both, or neither.
import time
from tqdm import tqdm
from openai import OpenAI

# 1. Configuration
# Based on your prompt variables task_1_prompt and task_1_prompt_few_shot
models_to_run = ['gpt-5.1-2025-11-13']
prompts = {
    "Zero_Shot": task_1_prompt,
    "Few_Shot": task_1_prompt_few_shot
}

def classify_barriers_new_prompt(text, model_name, prompt_content):
    retries = 2
    while retries > 0:
        try:
            # Using the specific prompt passed to the function
            completion = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt_content + text}],
                temperature=0.0
            )
            return completion.choices[0].message.content
        except Exception as e:
            print(f"Error with {model_name}: {e}. Retrying...")
            retries -= 1
            time.sleep(5)
    return "API_ERROR"

# 2. Nested Loop for Models and Prompts
tqdm.pandas()
for model in models_to_run:
    for prompt_type, prompt_content in prompts.items():
        # Construct specific column names: e.g., 'LLM_gpt-5.1_Zero_Shot'
        column_name = f"LLM_{model}_{prompt_type}_second_try"

        print(f"Starting classification: {model} ({prompt_type})")

        # Apply to gt_data based on your current workspace variables
        gt_data[column_name] = gt_data['text'].progress_apply(
            lambda x: classify_barriers_new_prompt(x, model, prompt_content)
        )

print("\nSuccess! New columns added to gt_data.")

In [ ]:
# Purpose: Summarize label distributions and cross-tabulated relationships in the data.
# 1. Define the new columns for cleaning
new_cols = ['LLM_gpt-5.1-2025-11-13_Zero_Shot_second_try', 'LLM_gpt-5.1-2025-11-13_Few_Shot_second_try']

for col in new_cols:
    # 2. Remove [ and ] and strip whitespace
    gt_data[col] = gt_data[col].astype(str).str.replace(r'[\[\]]', '', regex=True).str.strip()

    # 3. Print distribution
    print(f"\n--- Value Counts for: {col} ---")
    print(gt_data[col].value_counts())
    print("-" * 40)

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# 1. Identify the specific GPT-5.1 columns you just created
comparison_cols = [
    'LLM_gpt-5.1-2025-11-13_Zero_Shot_second_try',
    'LLM_gpt-5.1-2025-11-13_Few_Shot_second_try',
]

# 2. Extract Ground Truth
ground_truth = gt_data['majority_Label'].astype(str).str.strip()

metrics_summary = []

for col in comparison_cols:
    # Ensure data is clean and matches ground truth casing
    y_pred = gt_data[col].astype(str).str.strip()
    y_true = ground_truth

    # Calculate performance metrics (weighted to account for label imbalance)
    # This matches the methodology in your cell [538]
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # Identify prompt type for the table
    prompt_type = 'Few-Shot' if 'Few_Shot' in col else 'Zero-Shot'

    metrics_summary.append({
        'Model Name': 'gpt-5.1-2025-11-13',
        'Prompt Type': prompt_type,
        'Accuracy': accuracy,
        'Precision (W)': precision,
        'Recall (W)': recall,
        'F1 Score (W)': f1
    })

# 3. Create and display the comparison DataFrame
gpt_comparison_df = pd.DataFrame(metrics_summary).round(2)
display(gpt_comparison_df)

In [ ]:
# Purpose: Evaluate model predictions against the human-labeled ground truth.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# 1. Update columns to include the GPT-5-mini Few-Shot results
comparison_cols = [
    'LLM_gpt-5.1-2025-11-13_Zero_Shot_second_try',
    'LLM_gpt-5.1-2025-11-13_Few_Shot_second_try',
    'LLM_gpt-5-mini_Few_Shot'
]

# 2. Extract Ground Truth
ground_truth = gt_data['majority_Label'].astype(str).str.strip()

metrics_summary = []

for col in comparison_cols:
    # Ensure data is clean (removing any leftover brackets/extra spaces)
    gt_data[col] = gt_data[col].astype(str).str.replace(r'[\[\]]', '', regex=True).str.strip()

    y_pred = gt_data[col]
    y_true = ground_truth

    # Calculate performance metrics using weighted averages
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # Determine Prompt Type
    prompt_type = 'Few-Shot' if 'Few_Shot' in col else 'Zero-Shot'

    # Standardize model name for display
    clean_name = col.replace('LLM_', '').replace('_Few_Shot', '').replace('_Zero_Shot', '').replace('_second_try', '')

    metrics_summary.append({
        'Model Name': clean_name,
        'Prompt Type': prompt_type,
        'Accuracy': accuracy,
        'Precision (W)': precision,
        'Recall (W)': recall,
        'F1 Score (W)': f1
    })

# 3. Create, Sort by F1 Score, and Display the final table
gpt_comparison_df = pd.DataFrame(metrics_summary)
gpt_comparison_df = gpt_comparison_df.sort_values(by='F1 Score (W)', ascending=False).round(2)
display(gpt_comparison_df)

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 1. Prepare labels
# Using the same cleaning logic from your [cell 172]
y_zero = gt_data['LLM_gpt-5.1-2025-11-13_Zero_Shot_second_try'].astype(str).str.strip().str.capitalize()
y_few = gt_data['LLM_gpt-5.1-2025-11-13_Few_Shot_second_try'].astype(str).str.strip().str.capitalize()

labels = sorted(y_zero.unique())

# 2. Compute Confusion Matrix
cm = confusion_matrix(y_zero, y_few, labels=labels)

# 3. Plot
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)

plt.title('Agreement: Zero-Shot vs Few-Shot (GPT-5.1)')
plt.ylabel('Zero-Shot Prediction')
plt.xlabel('Few-Shot Prediction')
plt.show()

# 4. Summary Stats
agreement = np.sum(y_zero == y_few) / len(y_zero)
print(f"Overall Agreement between models: {agreement:.2%}")

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Filter for Disagreement Instances
# Using your gt_data dataframe
disagree_df = gt_data[gt_data['LLM_gpt-5.1-2025-11-13_Zero_Shot_second_try'] != gt_data['LLM_gpt-5.1-2025-11-13_Few_Shot_second_try']].copy()

# 2. Check which model is correct against Ground Truth
disagree_df['Zero_Shot_Correct'] = (disagree_df['LLM_gpt-5.1-2025-11-13_Zero_Shot_second_try'] == disagree_df['majority_Label'])
disagree_df['Few_Shot_Correct'] = (disagree_df['LLM_gpt-5.1-2025-11-13_Few_Shot_second_try'] == disagree_df['majority_Label'])

# 3. Aggregate Accuracy by Label
# We want to see: For the instances they disagreed, how many did each get right?
comparison = disagree_df.groupby('majority_Label').agg({
    'Zero_Shot_Correct': 'sum',
    'Few_Shot_Correct': 'sum'
}).reset_index()

# 4. Melt for plotting
plot_data = comparison.melt(id_vars='majority_Label',
                            value_vars=['Zero_Shot_Correct', 'Few_Shot_Correct'],
                            var_name='Model', value_name='Correct_Count')

# 5. Visualize
plt.figure(figsize=(10, 6))
sns.barplot(data=plot_data, x='majority_Label', y='Correct_Count', hue='Model', palette='viridis')

plt.title('Performance on Disagreement Set: Which Model is Better at Breaking Ties?')
plt.ylabel('Number of Correct Predictions')
plt.xlabel('Ground Truth Label')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print(f"Total Disagreements Analyzed: {len(disagree_df)}")

In [ ]:
# Purpose: Create visualizations for model performance, temporal patterns, or label distributions.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Filter for Disagreement Instances
# Using your gt_data dataframe
disagree_df = gt_data[gt_data['LLM_gpt-5.1-2025-11-13_Zero_Shot_second_try'] != gt_data['LLM_gpt-5-mini_Few_Shot']].copy()

# 2. Check which model is correct against Ground Truth
disagree_df['Zero_Shot_Correct'] = (disagree_df['LLM_gpt-5.1-2025-11-13_Zero_Shot_second_try'] == disagree_df['majority_Label'])
disagree_df['Few_Shot_Correct'] = (disagree_df['LLM_gpt-5-mini_Few_Shot'] == disagree_df['majority_Label'])

# 3. Aggregate Accuracy by Label
# We want to see: For the instances they disagreed, how many did each get right?
comparison = disagree_df.groupby('majority_Label').agg({
    'Zero_Shot_Correct': 'sum',
    'Few_Shot_Correct': 'sum'
}).reset_index()

# 4. Melt for plotting
plot_data = comparison.melt(id_vars='majority_Label',
                            value_vars=['Zero_Shot_Correct', 'Few_Shot_Correct'],
                            var_name='Model', value_name='Correct_Count')

# 5. Visualize
plt.figure(figsize=(10, 6))
sns.barplot(data=plot_data, x='majority_Label', y='Correct_Count', hue='Model', palette='viridis')

plt.title('Performance on Disagreement Set: Which Model is Better at Breaking Ties?')
plt.ylabel('Number of Correct Predictions')
plt.xlabel('Ground Truth Label')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print(f"Total Disagreements Analyzed: {len(disagree_df)}")

## 10. Final data-sharing export

Exports the camera-ready data-sharing CSV after removing or preparing columns as needed.

In [ ]:
# Purpose: Save the current processed dataframe or analysis output for reuse/reproducibility.
data_TACL.to_csv('data_TACL_Camera_Ready_Data_Sharing.csv', encoding="utf-8-sig", index= False)

## 11. Remaining unclassified/diagnostic cells

These cells were not clearly part of the main sequential pipeline, so they are preserved here for inspection rather than deleted.

# Classification Pipeline